# Submission 20: S64b + two-at-once + the evident fixes

**What it is:** submission 19 (`1030-sub19-qwen38-flashnext-s48-k384h-lev`) with today's measured improvements and two scheduler changes:
- **The S64b server** (Kaggle 4 x L4, run `1031_0445`, arm S64b: +12.7 % generated tokens over submission 19 on a
  20-min bench, 0 OOMs): the sub 20 build `c414512aa7` (section 5), the request split (inert), the session fork (a
  task's attempts restore a parked template of their shared prompt prefix), the three later exact memory levers;
  **64 slots**, mem-fraction 0.899, 2,048-token prefill chunks.
- **8 reducer threads (T8, burst 2):** the CPU all-reduce's threads; -2.5 ms an all-reduce at 64 requests.
- **Two-at-once (`PAR2`):** a task whose 1st attempt has used 64k tokens (`PAR2_AT`) with nothing accepted gets its
  2nd attempt beside it at once, ahead of new tasks, while the pacer has slack; the 3rd and 4th follow as before.
- **The evident fixes:** G3's vote bookkeeping (one accepted vote per conversation, first-pass chronology, the
  `acc[1:]` slice), the cancellation checkpoint (a program written but never submitted is graded once at the stop,
  <= 2 s), the nudge's truthful wording ("up to two failing examples"), `RETRY_FIX` (a sibling's program no longer
  cancels a failed task's 2nd attempt) and `OWN_ANSWERS` (each attempt file keeps its own answer).

Tonight's eval180 runs on 4 x L40S: the sequential arm (this schedule without two-at-once and the fixes) scored 215 of
249 outputs at the 2-answer pick; two-at-once and two-at-once + fixes were still running when this was built.

**Start-up:** the server boots once from the compile caches of this build's seed notebook (`1101-seed-flashnext-sub20-s64b-t8`), checked
against this machine first; without a matching seed it boots with a warm boot.

**This Save & Run All (`REHEARSAL = 'bench'`):** all 120 public evaluation tasks at full strength, solving stopped after
105 min: a load test of the seeded boot, 64 slots, two-at-once and the fixes, not a score. The scored rerun
ignores `REHEARSAL` and runs the hidden test set at full strength to the 11 h 45 min stop.

Built by `docs/reports/1101_0215_sub20_build/build_sub.py`. The cells below keep their own section notes; where a note
names an older value, the settings cell and this header are current.


## 1. Settings

The clock starts here. These are the only values to change between runs.
- **Full strength** applies to the scored rerun, and to any run with `REHEARSAL = 'off'`.
- **`REHEARSAL`** picks what a commit or interactive run does instead: `'smoke'`, `'load'` or `'bench'` (see the top). The scored rerun never uses it, whatever is set here.
- **`HARNESS_TOKENS`** is one harness attempt's budget over all its turns, and **`BUDGET_AT`** the fractions of it at which the model hears how much is left.
- **`GATE_HIGH`, `GATE_ROOM`** are the KV gate's marks (section 9): 90 % since submission 9 (measured in `1020-kv-gate-probe`; 82 % before).
- **`SHARE`**: a task's accepted program also answers the task's other test inputs (section 10). `False` gives submission 7's schedule, one conversation per test output.
- **`EXACT_TURNS`**: the server ends every turn exactly, at its stop token or its budget (section 5), so the next turn continues on a clean state and a cut turn continues in its own session. `False` gives submission 8's server and its rebuild from the history.
- **`SLOTS`**: requests in flight, 64 (the server's CUDA graphs reach 64). The server's own switches, the block tier
  among them, are set in section 6.
- **`VISION_LOOK`, `VISION_TOKENS`**: the picture-only first look of every attempt and its turn (section 9).
- **`TAIL_TIERS`, `TAIL_LATE_AT`, `TAIL_PLAUS_AT`**: which outputs get a second attempt first, by the tokens at the first
  acceptance (submission 13). **`TEMPERATURE`**: the sampling temperature of every request.
- **`MAX_ATTEMPTS`** (10-30): up to 4 attempts a task, in sequence; a task is settled once two accepted attempts
  agree on every output (section 10).
- **`PAR2`, `PAR2_AT`** (sub 20): two-at-once. A task whose 1st attempt has used `PAR2_AT` tokens (64k) with nothing
  accepted gets its 2nd attempt beside it, ahead of new tasks, while the pacer has slack. **`RETRY_FIX`**: a sibling's
  accepted program no longer cancels a failed task's 2nd attempt. **`OWN_ANSWERS`**: each attempt file also keeps the
  attempt's own answer. G3's vote bookkeeping and the cancellation checkpoint have no switch (bug fixes, section 10).
- **`TAIL`, `TAIL_LATE`, `TAIL_MIN_LEFT`, `WRAP_S`, `TOK_PER_S`, `FRUGAL_AUTO`, `FRUGAL_EARLY_ON`, `FRUGAL_EARLY_OFF`, `TAIL_VIEW`, `TAIL_EARLY`, `TAIL_RIVAL`, `PACE_*`, `DRAIN_S`**: the end of the run (section 10): `FRUGAL_EARLY_*` set the graded frugal's first stage, `TAIL_VIEW = 'plain'` shows attempts 2-4 the task as given ('transpose' would show it transposed), `TAIL_EARLY` gives early acceptances a second attempt last (tier E), `TAIL_RIVAL` gives one to acceptances whose review found a rival rule (tier R). `TAIL = False`, `WRAP_S = 0` and `FRUGAL_AUTO = False` give submission 10's solve.

In [1]:
import glob, json, os, re, time

T0 = time.time()               # the clock starts at the first cell
HOURS = 11 + 45 / 60           # stop at 11 h 45 min, 15 min inside Kaggle's 12-hour limit (sub 13: the cancel, the
                               #   server stop and the final file take ~7 min at most after it)
DEADLINE = T0 + HOURS * 3600

# Full strength: one harness attempt per task (its other test outputs reuse an accepted program), nothing else
SLOTS = 64                     # requests in flight: 64 (arm S64b of the x15 probe, 10-31: +12.7 % over sub 19)
HARNESS_TOKENS = 163840        # a harness attempt's budget across all its turns (sub 17: 196,608; 160k priced neutral)
BUDGET_AT = (0.5, 0.75, 0.9)   # the model hears how much of its budget is left at these fractions of it
GATE_HIGH, GATE_ROOM = 0.90, 0.05   # the KV gate: park above HIGH; bring back and admit under HIGH - ROOM
SHARE = True                   # a task's accepted program answers its other test outputs: one conversation per task
EXACT_TURNS = True             # the server ends every turn exactly (stop token, budget); a cut turn continues in its session
# A first look at the pictures (section 9): a separate pass that sees the task only as pictures and describes it
VISION_LOOK = True             # False: submission 10's attempt
VISION_TOKENS = 12288          # the look's turn; an empty look gets one 1,536-token answer turn, thinking off

# What a commit or interactive run does: 'smoke', 'load', 'bench', or 'off' (the full workload on the visible test file)
REHEARSAL = 'bench'

SMOKE_TASKS = 3                # smoke: the first 3 tasks of the visible test file (easy public training tasks),
SMOKE_HARNESS_TOKENS = 32768   #   harness attempts at 32k tokens,
SMOKE_MINUTES = 15             #   solving stopped after 15 min: about 20 min in all

LOAD_TASKS = 9                 # load: the first 9 public evaluation tasks (13 test outputs), full strength,
LOAD_MINUTES = 50              #   solving stopped after 50 min: about an hour in all, and the score on those tasks

BENCH_MINUTES = 105            # bench (sub 20's load test, 11-01): all 120 public evaluation tasks at full strength,
                               #   solving stopped after 105 min, so the Save & Run All ends before the 05:30 IST slot
BENCH_SET = 'eval'             # the competition's own public evaluation file (no dataset of ours, no eval children)
BENCH_TASKS = []               # the whole file: 120 tasks

# The end of the run (section 10): second attempts in the run's idle end, a wrap-up before the stop, and a pace line
TAIL = True                    # an independent second attempt (never shown the first answer) in the idle end, by tier
TAIL_LATE = 70_000             #   (TAIL_TIERS, TAIL_RIVAL, TAIL_EARLY; TAIL_LATE only without tiers); it starts
TAIL_MIN_LEFT = 20 * 60        #   only when no first attempt can start, and never in the last 20 min
TAIL_TIERS = True              # sub 13: second attempts by the tokens at the first acceptance: no accepted program (U),
TAIL_LATE_AT, TAIL_PLAUS_AT = 98_304, 32_768   #   then after 96k (L), a rival rule (R), after 32k (P), earlier (E)
TEMPERATURE = 1.0              # the sampling temperature of every harness request (the model card's 1.0, as in sub 12)
WRAP_S = 6 * 60                # in the last 6 min no attempt starts, and one without a program gives its answer
TOK_PER_S = 22                 # a safe decode speed per conversation: a turn is cut to end by then (lowered on a slow machine)
FRUGAL_AUTO = True             # if the pace line says the first pass would not finish, reviews go (graded: early ones first)
FRUGAL_EARLY_ON, FRUGAL_EARLY_OFF = 30, 45   # graded frugal (10-27): under 30 min of slack (2 readings) the reviews of
                               #   acceptances at <= TAIL_PLAUS_AT tokens go; back above 45 (under 10: every review)
TAIL_VIEW = 'plain'            # (10-30) attempts 2-4 see the task as given (subs 16-17: transposed; it did not
                               #   replicate on unseen tasks)
MAX_ATTEMPTS = 4               # (10-30) up to 4 attempts a task, in sequence; a task is settled once two accepted
                               #   attempts agree on every output
OWN_ANSWERS = True             # (fixes arm, 10-31) each attempt file also keeps the attempt's own answer(s)
PAR2 = True                    # (two-at-once A/B, 10-31) a hard task's second attempt starts beside its first: once
                               #   its first attempt has used PAR2_AT tokens with nothing accepted, ahead of new tasks
PAR2_AT = 65536                # (33 % of recorded attempts pass 64k unaccepted: about the hardest third)
RETRY_FIX = True               # (10-31) a task whose own first attempt found no program (tier U) always gets its
                               #   retry: a sibling's accepted program (wrong in 51-60 % of hard attempts) no longer
                               #   cancels it
TAIL_EARLY = True              # (10-27) acceptances at <= TAIL_PLAUS_AT tokens get a second attempt too, last (tier E)
TAIL_RIVAL = True              # (10-27) acceptances whose review found a rival rule get one too (tier R, after L): an
                               #   accepted answer differing from both replaces the rival, else the pair stays
PACE_START, DRAIN_S = 3600, 35 * 60    # the pace line from the first hour on; the last attempts end this long after its projection
SLEEP_BLOCKED, STATUS_EVERY, PACE_EVERY, CANCEL_WAIT = 30, 300, 600, 120

## 2. A valid `submission.json` first

`answers` holds the model's grids. Until a test output has one, its attempt 1 is the test input itself (some ARC outputs equal their input) and its attempt 2 is `[[0]]`; with one model grid, attempt 2 is the test input.

In [2]:
DATA = os.path.dirname(glob.glob('/kaggle/input/**/arc-agi_test_challenges.json', recursive=True)[0])
tasks = json.load(open(f'{DATA}/arc-agi_test_challenges.json'))
print(DATA)
print(len(tasks), 'tasks,', sum(len(t['test']) for t in tasks.values()), 'test outputs')

SUBMISSION = '/kaggle/working/submission.json'
answers = {}  # (task_id, test index) -> [attempt_1, attempt_2], None where the model has not answered

def write_submission():
    sub = {}
    for tid, task in tasks.items():
        sub[tid] = []
        for i, test in enumerate(task['test']):
            a1, a2 = answers.get((tid, i), [None, None])
            sub[tid].append({'attempt_1': a1 or test['input'],
                             'attempt_2': a2 or (test['input'] if a1 else [[0]])})
    with open(SUBMISSION + '.tmp', 'w') as f:
        json.dump(sub, f)
    os.replace(SUBMISSION + '.tmp', SUBMISSION)

write_submission()
print(os.path.getsize(SUBMISSION), 'bytes written')

/kaggle/input/competitions/arc-prize-2026-arc-agi-2
240 tasks, 259 test outputs
200863 bytes written


## 3. Which run this is, and the machine

**Recognising the scored rerun.** Either test is enough:
- Kaggle's environment variable is set;
- or the test file holds tasks that are not public. The visible test file holds public training tasks; the hidden one does not.

**Checks.** A failed check stops a commit, so a broken version never reaches Submit. The scored rerun only notes the failure and carries on, because it must still end with its file.

Four L4s are required: the model is split across all four GPUs.

In [3]:
known = set()
for split in ('training', 'evaluation'):
    if os.path.exists(f'{DATA}/arc-agi_{split}_challenges.json'):
        known |= set(json.load(open(f'{DATA}/arc-agi_{split}_challenges.json')))
RERUN = (os.getenv('KAGGLE_IS_COMPETITION_RERUN', '').lower() not in ('', '0', 'false')
         or any(t not in known for t in tasks))
FULL = RERUN or REHEARSAL == 'off'
print('run type:', os.getenv('KAGGLE_KERNEL_RUN_TYPE'), '| scored rerun:', RERUN, '|',
      'full strength' if FULL else f'{REHEARSAL} rehearsal')

def check(ok, problem):
    if not ok:
        if RERUN:
            print('WARNING:', problem, flush=True)
        else:
            raise RuntimeError(problem)

check(REHEARSAL in ('smoke', 'load', 'bench', 'off'),
      f"REHEARSAL must be 'smoke', 'load', 'bench' or 'off', not {REHEARSAL!r}")
check(0.5 <= GATE_HIGH - GATE_ROOM < GATE_HIGH <= 0.92, 'the KV gate needs 0.5 <= HIGH - ROOM < HIGH <= 0.92')
check(all(0 < f < 1 for f in BUDGET_AT), 'BUDGET_AT must hold fractions between 0 and 1')
check(TAIL_VIEW in ('plain', 'transpose'), f"TAIL_VIEW must be 'plain' or 'transpose', not {TAIL_VIEW!r}")
check(MAX_ATTEMPTS in (2, 3, 4), f'MAX_ATTEMPTS must be 2, 3 or 4 (TAIL = False switches later attempts off), not {MAX_ATTEMPTS!r}')

!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
!echo "$(nproc) CPUs" && free -g | head -2 && df -h /tmp /kaggle/working | tail -2

n_l4 = get_ipython().getoutput('nvidia-smi -L | grep -c "NVIDIA L4"')
n_l4 = int(n_l4[0]) if n_l4 and n_l4[0].isdigit() else 0
check(n_l4 == 4, f'expected 4 NVIDIA L4, found {n_l4}')

run type: Batch | scored rerun: False | bench rehearsal
index, name, memory.total [MiB], driver_version
0, NVIDIA L4, 23034 MiB, 580.178.04
1, NVIDIA L4, 23034 MiB, 580.178.04
2, NVIDIA L4, 23034 MiB, 580.178.04
3, NVIDIA L4, 23034 MiB, 580.178.04
46 CPUs
               total        used        free      shared  buff/cache   available
Mem:             188           3         182           0           4         185
overlay         8.0T  7.0T  1.1T  88% /
/dev/loop2       20G  2.2M   20G   1% /kaggle/working


**The plan for this run:**
- **Full strength (the scored rerun):** every task of the test file, up to `MAX_ATTEMPTS` (4) harness attempts each at 163,840 tokens: first
  an attempt for every task's first test output; its accepted program answers the task's other test outputs, which get an attempt
  of their own only when none was accepted; then attempts 2-4 by tier until two accepted attempts agree. Stopped at
  11 h 45 min.
- **`smoke`:** the first 3 tasks of the visible test file (easy public training tasks), harness attempts at 32,768
  tokens, solving stopped after 15 min.
- **`load`:** the first 9 public evaluation tasks at full strength, solving stopped after 50 min.
- **`bench`:** `BENCH_TASKS` of `BENCH_SET` at full strength, solving stopped after `BENCH_MINUTES`. Here the load test:
  all 120 public evaluation tasks (172 test outputs) from the competition's own evaluation file, for 105 min (the
  user, 11-01: about 2 hours, so the Save & Run All ends ~30 min before the 05:30 IST slot; Kaggle's pace for 120 tasks
  would be ~345 min). It checks the seeded boot, 64 slots under load, two-at-once and the fixes; few later attempts
  finish, so it is not a score estimate.

Everything else is the same code.


In [4]:
if FULL:
    todo, harness_tokens, minutes = tasks, HARNESS_TOKENS, None
elif REHEARSAL == 'smoke':
    todo, harness_tokens, minutes = dict(list(tasks.items())[:SMOKE_TASKS]), SMOKE_HARNESS_TOKENS, SMOKE_MINUTES
else:   # 'load': public evaluation tasks; 'bench': BENCH_SET's tasks; both at full strength
    evaluation = json.load(open(f'{DATA}/arc-agi_evaluation_challenges.json'))
    if REHEARSAL == 'bench' and BENCH_SET != 'eval':   # our own set: its challenges and solutions files, attached
        BENCH_DIR = os.path.dirname(glob.glob(f'/kaggle/input/**/{BENCH_SET}_challenges.json', recursive=True)[0])
        evaluation = json.load(open(f'{BENCH_DIR}/{BENCH_SET}_challenges.json'))
        print('bench set', BENCH_SET, 'from', BENCH_DIR)
    if REHEARSAL == 'bench' and BENCH_TASKS:   # a slice of the set, in the listed order (the evaluation file too)
        check(set(BENCH_TASKS) <= set(evaluation), f'BENCH_TASKS not in {BENCH_SET}')
        evaluation = {t: evaluation[t] for t in BENCH_TASKS}
    n, minutes = (LOAD_TASKS, LOAD_MINUTES) if REHEARSAL == 'load' else (len(evaluation), BENCH_MINUTES)
    todo, harness_tokens = dict(list(evaluation.items())[:n]), HARNESS_TOKENS
outputs = [(tid, i) for tid, task in todo.items() for i in range(len(task['test']))]
print(f'{"full strength" if FULL else REHEARSAL + " rehearsal"}: {len(todo)} tasks, {len(outputs)} test outputs,',
      f'one harness attempt per task' + (", its accepted program answering the task's other test outputs" if SHARE else ' per test output') + f' (up to {harness_tokens:,} tokens);',
      f'stop {f"at {int(HOURS)} h {round(HOURS % 1 * 60)} min" if FULL else f"after {minutes} min of solving"}')

bench rehearsal: 120 tasks, 172 test outputs, one harness attempt per task, its accepted program answering the task's other test outputs (up to 163,840 tokens); stop after 105 min of solving


## 4. The offline runtime

`/kaggle/input` is a network mount, slow for one reader but fast for many. So all model files are read once, in parallel, in the background, while the runtime installs; the server then loads them from memory.

The runtime is the output of the notebook `1006-sglang-runtime-cp312`: a standalone Python 3.12 plus every wheel of the `lmsysorg/sglang:v0.5.21` image. It installs offline into a venv in `/tmp`, using `uv` from the same wheelhouse.

In [5]:
%%bash
du -shL /kaggle/input/models
# 16 readers over the model files, timed, except the PLE files: the loader reads their rows straight into its
# pinned table (SGLANG_AGI2_PLE_DIRECT_READ, section 6), so caching them too would only evict other weights; nor
# RadixArk's 192 expert shards, which the QG overlay replaces (reading both would be ~203 GB on a ~188 GB host)
nohup setsid bash -c 'T=$(date +%s); find -L /kaggle/input/models -type f ! -name "*plefp8*" ! -path "*hf-7b719225242a*/layer-*-experts-*" -print0 | xargs -0 -P 16 -n 4 cat > /dev/null; echo "precache done in $(( $(date +%s) - T )) s"' > /tmp/precache.log 2>&1 &
echo "precaching in the background (pid $!)"
# The runtime's wheelhouse too: on 10-24 its mount (gcsfuse) read ~20 MB/s for one reader, and the install took 13 min
# until 16 parallel readers had pulled the 7 GB into the page cache (then 2 min)
RT=$(dirname "$(find /kaggle/input -maxdepth 4 -name python312.tar.gz.bin | head -1)")
nohup setsid bash -c "find -L $RT/wheels -type f -print0 | xargs -0 -P 16 -n 2 cat > /dev/null" > /tmp/wheel_precache.log 2>&1 &
echo "precaching the wheelhouse in the background (pid $!)"

191G	/kaggle/input/models
precaching in the background (pid 142)
precaching the wheelhouse in the background (pid 159)


In [6]:
%%bash
set -e
RT=$(dirname "$(find /kaggle/input -maxdepth 4 -name python312.tar.gz.bin | head -1)")
echo "runtime: $RT"
SECONDS=0
tar -xzf $RT/python312.tar.gz.bin -C /tmp
/tmp/python/bin/python3.12 -m venv /tmp/sgl
/tmp/sgl/bin/pip install -q --no-index --find-links $RT/wheels uv
UV_LINK_MODE=copy /tmp/sgl/bin/uv pip install --python /tmp/sgl/bin/python --offline --no-index --no-deps \
    --find-links $RT/wheels -r $RT/requirements.lock
echo "installed in ${SECONDS} s"
/tmp/sgl/bin/python -c "import torch, sglang, flashinfer; print('torch', torch.__version__, '| sglang', sglang.__version__, '| flashinfer', flashinfer.__version__, '|', torch.cuda.device_count(), 'GPUs')"

runtime: /kaggle/input/notebooks/carriem7/1006-sglang-runtime-cp312
installed in 157 s
torch 2.13.0+cu130 | sglang 0.5.21 | flashinfer 0.6.18 | 4 GPUs


Using Python 3.12.15 environment at: /tmp/sgl
Resolved 363 packages in 5ms
Prepared 362 packages in 55.14s
Installed 362 packages in 1m 34s
 + absl-py==2.5.0
 + accelerate==1.15.0
 + addict==2.4.0
 + ai-dynamo==1.6.0.dev20260930
 + ai-dynamo-runtime==1.6.0.dev20260930
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + airportsdata==20260905
 + aisimulate==0.13.0.dev202609270000000058
 + annotated-doc==0.0.5
 + annotated-types==0.8.0
 + anthropic==1.11.0
 + antlr4-python3-runtime==4.9.3
 + anyio==4.15.1
 + apache-tvm-ffi==0.1.11
 + argcomplete==3.7.2
 + asttokens==3.0.2
 + attrs==26.1.0
 + av==16.1.0
 + azure-core==1.41.0
 + azure-identity==1.25.3
 + azure-storage-blob==12.31.0
 + better-profanity==0.7.0
 + black==26.5.1
 + blake3==1.0.10
 + blobfile==3.0.0
 + bokeh==4.0.0.dev5
 + boto3==1.43.106
 + botocore==1.43.106
 + build==1.6.1
 + cache-dit==1.5.1
 + certifi==2026.7.22
 + cffi==2.1.1
 + cfgv==3.5.0
 + charset-normalizer==3.5.2
 + chex==0.1.87
 + click==8.5.0
 + c

## 5. Our SGLang build: Flash-Next on 4 × L4

One diff over the runtime's stock SGLang 0.5.21: `git diff v0.5.21 c414512aa7 -- python/sglang` of our fork
(branch `agi2/feat/sub20-levers`, 1,051,502 bytes, 126 files, sha256 `c83f4c936e58`). It is submission 17's build
(`agi2/feat/sub17` 06e9a8204c: the Flash-Next L4 stack, the QSA block tier with promote-on-touch and per-layer
residency, the exact top-k) plus these, each behind its own switch (section 6):
- **The sparse verify accept** (exact up to float rounding at the sampling CDF): DFlash's accept runs on the
  distributions it already has, the target's sharded top-20 and the draft selector's candidates, instead of scattering
  both into two full-vocabulary fp32 matrices. At 48 slots the sampler's reserve falls from 0.78 to 0.31 GB a GPU.
- **The stage cache** of the QSA block tier (exact: the attention reads the same bytes): each host-resident 4-token
  group a step selects is copied once into a GPU scratch and kept across steps until its rows change, so host RAM can
  hold most of the pool. X is 6.0 here (S64b): ~3.47M index slots a GPU at 64 slots (`1031_0445`); at X 7.5 and mf 0.88 it measured 4,173,120
  (1030_2128 arm B; boot-dependent).
- **INT8 + a per-key scale index keys**, the one lossy switch. The compressed keys that rank which 2,048 tokens each query attends are
  stored at ~56 % of their BF16 bytes and cast to BF16 on the GPU before the indexer's
  matmul; the attention itself reads the same K/V. On identical tokens (16 prompts of 16k), FP8 keys were at the
  run-to-run floor on the model's own continuations and 1.27x it on prompt text, with no significant shift.
  INT8 keys have ~5x less rounding error: on gemma-normed keys the top-512 overlap with BF16 is 99.3-99.9 % (FP8: 97.6-99.5 %), and their decode kernel matches an fp32 reference on the stored integers.
- **The draft's graph-index cap** (exact): the draft's FlashInfer CUDA-graph index buffer is sized for its 2,048-token
  window instead of 48 x 4 rows x the 262,144-token context. On Kaggle the draft graphs' memory fell from 0.62 to 0.22 GB,
  and 0.86 GB stayed free at ready at 48 slots in sub 18's build (0.40 without it; 0.85 GB with the stage cache at mf 0.88, 1030_2128).
- **The decode-lever bundle** (`agi2/feat/qsa-idx-shared` 17b4f71a70, merged in `5d76d788f1`): M-RoPE positions
  uploaded asynchronously, Marlin's `cache13` left empty, the expert-parallel top-k sum over local slots only, the EP id
  map as one Triton gather, and the draft's candidate head on a 32,768-token subset (the cell below). Per-kernel
  timing on one L4: −2.5 ms a step; on 4 × L40S +2.4 % end to end at unchanged acceptance.
- **The memory levers** (`agi2/feat/mem-levers` eb3a105f01, all exact): FlashInfer's int workspaces at 1 MiB (27 x 8 MiB
  before, for plans of at most 2,878 B), one shared fp32 logits buffer at the vocab-shard width for every
  target-verify graph (the same bits as each graph's own copy), and the eager input registry capped at the
  prefill budget: 715 MiB a rank on 4 × L40S (`docs/reports/1030_1550_parallel4/memaudit`), spent on the KV pool.
- **The grouped INT8 decode indexer** (`agi2/feat/qsa-idx-shared-int8` 3b0a8643c7): in a target verify, each
  request's INT8 index keys are read once for its 4 rows instead of once a row; logits bitwise equal, top-k
  identical, 5.71 → 3.09 ms a step a GPU at 48 requests on an L4 (`1031_0108`). With it the indexer skips the
  `-inf` fill of logits past each row's length, which the top-k never reads (~1 ms a step a GPU at 48 requests).
- In the diff but off here: the page-granular host tier, split-KV sparse decode,
  the top-k diagnostics, the grouped QSA indexer for BF16 keys, the fused and int8 hyper-connection mix and the skinny GEMM.

Measured for submission 18's build (earlier configurations, census expert list) on Kaggle 4 × L4 at 48 slots: 48 x 24k at 140.6 ms a step with FP8 keys at X 6 (6.9 % host share, ~965 tok/s), against 179.7 ms for submission 17's server at 48 slots (`1030_0046`, `1029_2325`); 48 x 96k (FP8 keys, X 7.5, 4.74M index slots) at 173.3 ms a step with all 48 running, a median host share of 30 %, ~820 tok/s, 0 errors, no OOM or retraction (submission 17's 24-slot server: ~650-700 tok/s at long context).

Stored compressed (gzip + base64) to keep the notebook under Kaggle's 1 MiB source cap; the cell writes exactly the
diff's bytes. Then the 384-expert keep list (48 layers × 384 expert ids, re-chosen from all 512 on ARC harness
traffic; the list order is the expert-parallel placement) and the draft's candidate vocabulary, also stored compressed.


In [7]:
# sglang-v0.5.21-to-sub20-c414512.diff: stock v0.5.21 -> agi2/feat/sub20-levers c414512aa7 (python/sglang), stored compressed
import base64, gzip, hashlib
data = gzip.decompress(base64.b64decode('''
...
'''))
open('sglang-v0.5.21-to-sub20-c414512.diff', 'wb').write(data)
print('sglang-v0.5.21-to-sub20-c414512.diff:', len(data), 'bytes, sha256', hashlib.sha256(data).hexdigest()[:12])
check(hashlib.sha256(data).hexdigest() == 'c83f4c936e5848a2e4a12c3690d1b7c24657925e55bff8e94f9883c7456eaa75', 'the build diff does not match its sha256')


sglang-v0.5.21-to-sub20-c414512.diff: 1051502 bytes, sha256 c83f4c936e58


In [8]:
%%bash
set -e
SITE=$(/tmp/sgl/bin/python -c "import os, sglang; print(os.path.dirname(os.path.dirname(sglang.__file__)))")
patch -p2 --fuzz=0 --forward --no-backup-if-mismatch -d "$SITE" < sglang-v0.5.21-to-sub20-c414512.diff > /tmp/patch.log 2>&1 || { tail -20 /tmp/patch.log; exit 1; }
grep -c '^patching file' /tmp/patch.log
grep -q "def agi2_expert_keep_lists" "$SITE/sglang/srt/configs/qwen4_exp.py"
test -f "$SITE/sglang/srt/layers/attention/qsa/pair_layer_split.py"
test -f "$SITE/sglang/srt/speculative/dflash_length_cap.py"
test -f "$SITE/sglang/srt/models/qwen4_exp_ple_direct.py"
test -f "$SITE/sglang/srt/speculative/fn_conf.py"
test -f "$SITE/sglang/srt/mem_cache/agi2_qsa_block_tier.py"
test -f "$SITE/sglang/srt/mem_cache/agi2_qsa_host_tier.py"
test -f "$SITE/sglang/kernels/jit/csrc/elementwise/fast_topk_exact.cuh"
grep -q SGLANG_AGI2_FAST_TOPK_EXACT "$SITE/sglang/kernels/ops/attention/fast_topk.py"
grep -q SGLANG_AGI2_QSA_BLOCK_TIER_POT "$SITE/sglang/srt/mem_cache/agi2_qsa_block_tier.py"
grep -q SGLANG_AGI2_QSA_BLOCK_TIER_PERLAYER "$SITE/sglang/srt/mem_cache/agi2_qsa_block_tier.py"
grep -q "def stage_cache_enabled" "$SITE/sglang/srt/mem_cache/agi2_qsa_block_tier.py"
grep -q SGLANG_AGI2_QSA_INDEX_INT8 "$SITE/sglang/srt/mem_cache/qsa_kv_pool.py"
grep -q _tilelang_qsa_mqa_decode_int8k_kernel "$SITE/sglang/srt/layers/attention/qsa/mqa.py"
grep -q SGLANG_AGI2_SPARSE_ACCEPT "$SITE/sglang/srt/environ.py"
grep -q SGLANG_AGI2_SWA_GRAPH_IDX_CAP "$SITE/sglang/srt/layers/attention/flashinfer_backend.py"
grep -q SGLANG_AGI2_DRAFT_VOCAB "$SITE/sglang/srt/models/dflash.py"
grep -q SGLANG_AGI2_MOE_EP_SUM "$SITE/sglang/srt/layers/moe/fused_moe_triton/fused_marlin_moe.py"
grep -q SGLANG_AGI2_QSA_IDX_SHARED_INT8 "$SITE/sglang/srt/layers/attention/qsa/mqa.py"
grep -q SGLANG_AGI2_QSA_NO_LOGITS_FILL "$SITE/sglang/srt/layers/attention/qsa/qsa_indexer.py"
grep -q SGLANG_AGI2_FI_INT_WORKSPACE_BYTES "$SITE/sglang/srt/layers/attention/flashinfer_backend.py"
grep -q SGLANG_AGI2_SHARD_LOGITS_BUFFER "$SITE/sglang/srt/model_executor/runner/decode_cuda_graph_runner.py"
grep -q SGLANG_AGI2_EAGER_CAP "$SITE/sglang/srt/model_executor/runner/eager_runner.py"
echo "Flash-Next build in place: $SITE"


126
Flash-Next build in place: /tmp/sgl/lib/python3.12/site-packages


**The request split**, applied over this build (Python only; inert until `SGLANG_AGI2_QSA_REQ_SPLIT=1`):

In [9]:
%%bash
set -e
# the QSA request split over this build: fork agi2/feat/qsa-req-split's increment over 5d76d788f1 (10 files, Python
# only; inert unless SGLANG_AGI2_QSA_REQ_SPLIT=1). It applies over c414512aa7 unchanged (git apply --check).
SITE=$(/tmp/sgl/bin/python -c "import os, sglang; print(os.path.dirname(os.path.dirname(sglang.__file__)))")
echo 'H4sIAAAAAAACCu1dbXPbRpL+rl8xJ9degAiESVEUJbm4tU7ieFN2Eq/tzd2WSocaEkMSEQhQAKiX+PLfr7tnBpgZgKTkJJX9cE7FJIF57emXp3t6xnEyn7Neb5FUjD9fP1TLPHteLlKeLZ6XRfU85Q+iKJ/zqhJZlcC7mxKK8aSI6E1UrtOkCtcPbPobKh8kWSzumZiez8QsHo6nYdifif7x7Kw/mLNBv396cnLQ6/V+0wgPjo6Oftso//Y31js/DobsCP4ejxn8jsWc5XeZiFXZJC69+tsF+yBuNiKbicskq64CtpxHlbivolmezZOFz3p/ZW+TsqK3FwdM/0nmrGBJyX7IM8HyghUhdlEYJfBPvqlCvl6LLK579GWJQlSbIsMCB0f43zNY3z/hD3bMF8kxjAeoUFaMCMm8D6/fvvzhdfTy9XfH0T8+vIzev/pH9OHd2+8+BizPWJWvWT5n1VIwXANZ6QU8mrPpA9Kbb9LKD6FxbP+fwDiFU5h+ZnnWI6phY5zNN2naq1eYEcHYHU+qkiUZVZjnaZrfJdmCcShaiHgzE+xumaSCXlNb2GMheFzSI2rki5LBauKqMo+4WBQBDIMXpWB1fwGrEuhvmQMRqL5/wQbDcMxWJYytrMQayrKTM3bPBqfXARsOwnN8cnoCT47PrrFfb9Af9qPBeNgHEhWzJasKPhOlH7KPMBabxIvkVpRM3IrioX7DofuVYEuc76d+wAa/Mk/SrUiqB6QSEkOV7q3zPMVeyzSvAjaDkYsMR8TjVVKWSMJ5ka+IDKoKEAKIUPDoWjz4jGcx0GW13lQwkE7i/wwUL+rRFbj0iyLfrLHXlVhNoYj3MztiS5/9hR2H7O8w+l5fVwC6FQI65LMqfXDWP2Qvaa69gTE23WlS6nHFyE+6Js0VK0NRi3V+DoAJYPbszU8sJbIqfplu5nNQGyavQpO6m/9hA+aRYsC3hXghR2SMf1MK7LSixpHMJRHEaCBwZmEVCIjE2HN1l7NSVDQS2Q6SJk7Kn3PQK8AfH3LNCjwjViqBgYTk4ZIDT0wfcJU4jKlDmgI2zaulWpJSShMtlKajalyOC0cI9aEyTL4eeECqAMYrODBukd8BlYHvqoSnuCBI8F9EkSMP6CVF1Sf7BHHKaUyGXFZL4EUpsSUrNytjMrc8hT5RP6D+7BHTF1gMqXIHvA5sCBJEFRJY5EoR7et/fvOytyj4eglDi2OpCFBYH3CpSuYBA2BJBrKjubYP8yiwJHZAk0SaAG1LWHdZ2Ifhz4DcMEmtKOCJSMWsgkfVEmt7vYGUGBwiEgJ5U8yxlj1vrfVI4nmc3LMZEBQItYHBZLgOrFxiI5zBKObwns+KvCyJkWAKKOE1HwOx4wR5GMRSLy/oJewR2shgEjDOLC6xw1qy2SFq9OUgDMNDWmyHRSVbNsUTudBaSuSg4UX4J1qlg6P3H6K///j9q0H07v2rb7/7bzbR0zqUBhOtelFGIuPTVMQeGeopaMSLgyPDvjavcc4LUUVYBmrdRre88A632rlD3+oGSRjlc89Qn9AfyK7q7vDw8KWhyYijSfvicjak9hoCZyAIYINmwlnNigwPrZuxjgETKeigfggdWRMcIBJJyiQrKw4gphkgcHBV+PZSh1CoqMq7pFp6Dn191YE1a54mvIxWfO11oKI4mRmz/0Ep5LjbkiQxVtFGOm6eSsWtFfUyT2OmJKCDc5vZA3IDjsCuororA945w5V1pEacsIi+ABKonHJhtlkhmSLST9GSUICqDFSW9f9jwo7VtGkZeALj/wkrvCqKvPDm23kKxF/AuKkd0tLHpO4RX8B8sbuALUAhfaISvx6qrgEjwqg//Sp/IVnAsui5oqkTMG5R8Ep4CcKWZnAwaNBfoPlS4Uy1acDXwPXIAa6XusQV9A4NX5LBu7K4r4avmmGqCpSdh8ikE0kjWm545h0aP2nKoALzkFdx8adpPrsmOOYjnjFMt8k8SAhp1rKdvNcwDbEzTGYXZ8uSOJ4QpS/SZaEafcr3BBUnqE+gs0LNl8Q1osFHWACEFufbMBDVIltamZTQDYaS4Lo7lC+PfvhNuWcGTIB27lixyTSK7Jq8tBXYOmglCWLjW1QUpabsPClKrftXzoDoXS1WyAYgM48Vusv+lZ76ap2DfUnzxQJMKbILPlU/QyDiW/gqCi+KUClGkR8m2Tz3msEctl2UC/aXeM/Cl2rBJdZpcxBQROkdaKw8DJr+UpEpyhsP6YH6bRuHeVJ5K16CU5BdIIl8yyjAIuFLNgMxzgsCLjAC71sO+tZHlJMRNDF0myQXeRCaWCtFfGwpBPywFgZ55/h6Ap0baknKJxY3Cv11W5nLi8wWbOo8nPHKu5Qzo44zcRch9im9jPXYyocRxJ8RmoD/I+WHPSkqYddTAYn4fDSezY/HJzwMT8+HMzGfD2fHnxWQcNp/UizCqYthiJPTUXDGjvBj0MdAxCzlYOLBJHwnC3rfg5ucvEt5Bbps9ePaN6IHYKkrERHsvVCr8VFkJTDQ/8qYw4Q+glYN9BviC4JCUIa4zCiESDiSuHMi6zfsUE6MFg113avF8PDwQ6URLwY1oLGetIqEsLVDieBauhH4KxXox1Gxhsd/n+YYii7B75Z++KJsOvLueEaIXgZ6mORoLZi+cogS5YAGKKVEYkJrOfqGBES+qE13PSoQyVw7ETQ51pB6LVdgBTYi5hUPq/xaZFGVR9e3EbnuTSwJYOw9MFNsFtd9RBL2R+QfGcZDstjZMBgDi52NgvFTOUzqMWaFtOreYUizTcwjcrYMFiBGE+k8jOrxxWKWx8Io7ulWfBtTPLpegLwIjo49VqmZtg0YvUYRge0tdVTuQlLofBScAoXOx58hhGa3+g+xADgQMSximaD8l53LJsvVZcI0zxaeMaXGC5WjnnTNxYJyRbkFPEhg0GUjdRhAuXYWL9fiQN2VhrPdcpDtruTMVgkpoaK8HFyFpPki6TJ74Hy3yaF7Q6SgaOGwR5sgUu3dYWzGazoN3JKBKkiWKUqTa+E5JXzfYppa3pylfsZeSTGcLTcEy0E3Ub89sP0LwA0vpBpagynItBICT10HnBKK/tzxIg7chsucgooEb3WYpo7OgRd4LVSvrCItX4bEvaPRMBiADRmNzoPj06eybwZPYOS/CFQsJH31k8igEHg8JUFYEVu0LfK1aBjYtzVbCJ5qhEbPbCiSeMqjvjQw7FivZmCogEi5IJTapRuwyYavtlinb5N7EfcIHjGq1isELNZDPQKYyAtCjdN8k8UcgTSqVZIAxeuNjXrakAJtQF1H53ccG7lIgRECbMu8YdBbEk8RMIyXAw8DLlAsaQbKpPzX4bKWTTM1Lk0iautdNJ9OIUIdUoJtDW1rWnKl328AVK2UM30IjE5RP0kyk1Z1H9hakmlocOh/jgU2xjbZM/iGvsbUJ9vJ8ngVTvpA4swQcKYUoDKscWYI4ha6e17aW5BuiHYX9J9aSZOPArr6+Cowxu16Ep0zM7Ww1L/N6w7V27x0FbxFZLfVywvpLcP4mnIdzTcvHaUu0cWmFNF8g5pGswppNteoy8J2QQmdPbugZqOg/dgYpo2PiPDb2CGwaWJTO2ihs11IBCNVCpBOpE5eKJ2Mb6Q70KWQ/ctm6FfSzIxPEUaOzvufByMbBb/DgBx0zWuvKemAno1h2U5lYxghhvSTxSbflF4XIm2ae0z9NnQmM0FTRQGOVrwqkntlHOrJ2z4cmQjzgcSqY4lVz/rnweD0c9ZBhVWSGU8bunewrnJUo0c7Gy3GPNqFO3dsH9Pmhfa0mgCMDPzyTJor/RKBid6tkpbM7FcbtXpXQ25BJYW0myTN0kHbYeECx5ezd5zA4oLBNTt194mY15GE4JtOtREmVOAwmvJqttTxQtDNqP/sYOGfZC6MoIOjottay7YpRs3GqFiPO3S58dYRTEPOnrGv5Dbbush/Bvciurm+IBhN2huwc7VsNksbBK53/SqepGFn/ENj4oiWYcGRvyLjvedKiTGoG5wNgonrwIq76DabsXok2WcDcD8HINong2Bw/jmibdBqYlK1sxSNZWL+cMqZYSDju2OcyAsPOvU2MKfm5hXAYowXSIDs+Yj/4GfFCzRHt6JI5g/qWVzwedXtexm6n/ZLQXfyhYBWpqnj8YlsUS1LjEjeU98pPgLCe4/QczgiBeRXNxiwW2+q0rGOW1XdY4RyuyveicY79IlUYV9/fIn7hhhVrVgOfrPf7qdNlL1C2xbcdiOG/HY30LEY3QV3uuWyZlcfnbBHypWMMOglJBEsO9b9psPmtRjs6aHrOxB4mTaE+x8Z6HHw27P48THs7Q2oYHZ/PJ2PRqPZST8M+/HgdCRG09nw6cHsHR09Pqq9oxHUaYPh8RjhCn0OzgylBhU/UL2XUO0rWct7qRtWD6xgt7I4jqeGj6UGuDTDR4arUPK5uDpwWcWUgVueIrSzTBDtYEkLJB/3Br4PfXryNQCB4THCO+LMv6EaTWYwtGUeG4EBbbwBLtqG3bf9/qfDI8pjatx+lb7UhKdx+76lTgzIQXoFE0a68Iz3ePj15vlPCn3hFr3GWb4dxIfp40a8SYAw31RSxpA4vy8W0gEGVL9G1gP0E9gI29HLijOs8fzOAKqD+WxVC52b2MgYs6Ei4Sn6zY1roQklraah7MjDMCCJ7WIESkZPjtGzGwxHJ5Zr92QR7cx1oPCMm0hJnJIQHsvX17gplsxEeeh46/xWYBiG2KTLOXdjNeQkwvdOt5AYUQrHDk4MGKC127a9UTUNMNgp1qr6NlN1A8x9E8K0cYW93kBlWITVOrqhVJIo26z0Q/odJyujPuaewHABkEjvC1vT3CLXcnQ+oLU87Q//fy3/rdfShCyR7hnxbNwkTng3Qec8A4vUezfYVzyDdsFwl0CZeJNu31LvLKlxBxfHs/H8bBqH4Xg45SdDPj+P9+GO7ha3AYzu0rRH1x+e4O7cEX05Pml4+4Mu6tkxaL6a8oinsLC8yp2cGjsOZ5cM6ZuK5k7FIsk8zPWghcJMc3SFwMxthBlclGwarZO1SGU8VXJ1IYRkyNAp8VuiI5xl4q6uIBPGOJgqQRmyOgVcuRAyL/xpFk0bsJZh46UCNeu0NO2aFiPMnNPvQzPz0vTVX4vKSIXWG1yKtIxIi7BEzpKczjRlxPdNKyrnHPfU2gvjrK8O/MrBgKQXnPIGkB3oxYwDjlRcRK8jaNqD/wNMncqwXXxadoXSKPCXbUTLBaxJoruC9sLr2xBhVAmardMp1OuG3IoIFKMY8nhDkyduO4iajjAH8OyJUoKy+ytMgfBev/sIZW4ToOS3gw7/cOmumE5i1RiMqHBYZ4dq2NXRFGVtTzCLe4tTOeh+TEuJC4lJTwUtBlAdkE5ZdVfQGYsmNYk6uyfSOQ3MiVp2uZdHHb3SDI/YgP1V8g3QBi2WNF1NOiAKIvQcScnzlv7FNh9ZsQ5zd4JfanUv2V4m8CGqxU5QSDCmQcFMsJJ3CSFvHmMOqOYHt0VPDlJZDBw1bXkg25AYqGxz2R05D6Bbqx7Vwp7LcK+RESup6Z4TETBW36Q5brU3eyop03MS989Gg3PBT8PwdDw9PxmfDsf7Tc++xrdaoX0V6QBZf4CeLX4Yfu1XWO4jFPuAgTVDX4BD9F6USQwO6oMqK49gpPmUp02us9eX5MfVpmJqf9VVZl8j5w4681DMXAxlXPaZkzpHUo6D2KEkbdSZEinH79XJvIa4GImgWvLkNsehkbkK4vfpVx8jbZRhXYfVAmb9NDNhtEWVL2cg463aNuKc7dxBfiMedCo2EQiBbZNSzN69ev/25b9evb9QBPmkO/lV4whQtybFPoE7BQrXs8foY372o+VGw2OVHLspCI3slZxt1ZTsTEf9+ek8ng4HYXgyGo9Gw/P4bPx42dna/H7p2VqVXJXz8RmFhvBzcNyI0JufvsZKXxt1nCVc5yXC/DUwu4j4rEpuxUSG1M0X0Lt8R7bhZsOzKpIRGs0ZTgTwyy+lfeDFwo5omyEGVAbrIp/jeUIDUtqAog5j3+XFdStv/jEyuS9reVOKR+QsP30zaBvyc3eBgBKtTS0T73WYvK7y6iyA6/wp50xG7tW5CzcHvx3QcxM3jK3X6SZJ42i15Pqt93ixhG958SDt+15htAtr83V+dgoO0/kQPKf5Ge+f9qfz4ezxIug0ul/wnAoobsMBSRt+DAaNsL0XNx/zj0i6d3QuqiYZv+VJSni5TH4RpNdM+KtjZdoxIsyjdulYEwHtwkNq15t+yFR4+5jU1pjoRe3ikO1EqbN8HJmLW3uO8Lw55QoPZ3SIxvCc1BksI3t/wTFaSIHUgFnQ+QVUXQFGzW9VVhWe1qRuMbNvyWO2FBgerUeMgfCuKCzNwYmP6lAAIGd5qKus/ZqGrihyXsn+kw0ItMopW1FA8lsVeWEYJRgvPJQOS3xFJP5xjVLO08v6rLp5WP0Z+06CQ6QtVpeHM/XZySVHRYoviK3AFN6TzrtDl6MQeArWaEqqJnEPHSFcpbiyJ8JFKBMYcU9W+XUKBFMutzzjKBMrfZncODymrJPhcBAMRnvYtjkJIgcUy/RtPFM540XxwJSLDy/e/HRoZ2n0jGAU7h5JwVWYQwYEkO8oEoDEAYIy+X1TwthNf+gPcbBrh9J1rB+R4rsP/TlHEpOStc+TW+fHMW7tdrvtIKM6tIiuqn2k0XGx5MnWCbv8HCeudiGJb7v8w6tWUk9rlWXHzVpH04fahcPBGQPGY5AX+1t8FN84yL5pZAt8lbqCsvubk3jzOTiC0G1fCs1oSLr+9NQKlO0UGkmCPDMI4JnD6Yxc1m9Nnb+DjobqB7V50VyasV0/WSahblNqVuMUuiSZSmHHaxOy2ii0BcJvjm8A0Xn2oMsCycslyKWtnhU/1cPu3DW6NDhsnVCu9sR6WPFrSkGAhTJTzJB18egJcu+WHlDVTsBw3VeeVzb2ocATfaX2OQwDTJyEly2gnaC2UVFLrIo56DgQv7V9pvPeY4Pz2gk9BvdZL+WU9T0m0IjTMnUa8th95xo5oFm50wTa07hyWkLztBAZHnQFVrqUo7piRxMz+FQnglrcviNho0s2ZNN+y4jL59bmHI5fy9McXVwysy73O7L+jL0TRQ+qUZhnKvhK36ZBgy1A+HjtA/DNPQPjUiS0CZPTBjHeGMLNkOkzaf8iKIwwWaeD3CachEbmS8gIkAwsKkOvQkaz9eYJMSDKML19JGi2CyvQPO5PhzNxHA+nYchPTs/j6Xm/P308aHYa3Q+anQqoSI/HFOPBj8GJlZFFavRj/uYnVKXe3x+mRRK/TTLBC/nIxMrXBKEDdkuf2jgs5P4WPoro4g4zSC7dHxjapuQLrOLJRtiRasVnz9nrryy1C1YxK3GLwTl8m8SK8XQMwzp/+jlBos7T+/I+HYKE8qzQtuiR2a/OLfC2O4f+C2qJriHR13/oS1H07gHgVO35eg3U9p1zue2QlHV6uzOjgN446sAIcdFrikQ1GcJNklgb4K9F4fnhrqUyahvIniDR8kGrEBBjZyffWNzajKJqUUdAO5KJmetrNeGvC2VOpdKFmtg1JgxTugetgoeRzApt5t1SZPUBclSiAEb8kP0XGmiYbo8GZoIHHQSlay6wrVig0aA7o+CRJyOifh0SJWO8T/FgpKCkdKSTSNyvt+qcdjm9u3kmTkbH4/5YhOHp2Swengs+3HtEuKu9rZqmoywFwk6HI7lnD5+Dob1pf/Lqfl1v039DKrp4i4Rxkjww5IR+xQRTKCkrC355rVjJNkl/9/K79xFFPOsbvOx7ulRKNl4pQGmJGP3KmhuGVNiYJNcZGEJ5G2K7su65100EKlqnn8vEU7823GqudBkJQZGea9pVCdRGXjMKIzeofihvtniqNpSXKSiDTHH8Nz/1cJPfvlJp160LAcFXs1udxV7v6cl7T8gpwxzU5rIylJWkO688cKDuH+yXSjoWzQorOltx0S1bwq1GInnPxKRp1a699VaJ5jYJhLz6eRvxPZUtmkdF2Xk8SCUPw5BlhNE44e917hXDMCeKrTuOZaiBT2pBUApidC4VxOnITep5jH6QAti2sHjjS+umF8sfNaizww3tuY6SJl/7lpmGsjpevieK3Cb5DWK0Iv9Zs0lu/cIkVGXzDtwjLwtRyEtDpL1zFZfe6/lLbN0sp2EM3X+V0Y0tW9LAtQcAEkLLNh7K3NfxyfCz9LpyD420n4OdmevdexZ/4OmS7iR2HZ914p7OXWkYQ2Jelut0MoKPeOpX4gjcOPH/nKOPZmrVtsOPZpndBxXNku3cYvstphgb1nOZb9JYb2cZi+gIhcy+VonXsKZ2vWhVrXWGtk7NCxz58BW/yqNlg7P+WStX+7H8KiNj7XOQvKIZwEedMkc/NH1p8nSBFj6FcdJtNUNp4/FZu03KN5fZkIjEjV9fKjqXyWKVA6bG+n+mRGw51kECIeLm+kDlzNc4YcUfMJun/7yPKo3/AK6QrOR2SrnB+qpBpRDLHIUJask4AF7mxF6/+v57hSpoa5SiJBJsJb/IHzNkHhCMexrknyOBxlpuE0B78XcfaME+QtzHoSzNL73LwRWwiGf2AvyGtwMyEMCu0yZG0Y63Rn625/kBi6uHtZhYzeMTeCFuQf7sN/TI7dPQAqpXFtn2zhx9OzQrnx8wKddngxOCD2fD8+D86WLt3v+CrG9n3bqnvJIYkIXv3AhTR926EGK9cx51gQerelF2xCWXSRzjYRBMunG8DRlIUPm0zYFXu4qTX+swm0ifPni25cRXG3WozZaK8OZdVhrnOF6496WCz4XH8KC4iGpHMlqleGVbXHZ1KuGLvKKVznpA2+qOVhG2K7iENHlbcdNQWYnT42B48nR26l4qzPizJuTtXp/uNGrM/wDpmOEp5aY1IqXbHsUhNjxtt9xxfqdhIOfI9h42+m2nfPS9vPLIalfMrfYYaUuk9haNnQh0G5NKxlaU/85oOwuVrnFHtNyildsgL6xbsSVcx2i72h/sBzVTZRrcDfAib93px9rcGZeT5Fn6YN7Tq6/Tpa0JCxrKTNdUyA1n4lXFwB03CBumeato0L3CKKz1/cRaIDijAPuWG5PtXaBHGb9FuhHa4OmLFlyY/ocdLsKLyh4FbbBcJ7ZxHHG8pIXa7N4QekaHTjHzXF1ohOdQ6dQ11e7kWC/JMP4nt+nkdS0K3yIvc/kMeF9uB0svwu3V2KJQV4hj9FDde94CLYgkt64FbixbBxfRfjs3PuyyDAXeFZusQne3hHbSuh4bpoOgrW1I3fLtLXIU27V1DPH0xMekEGdc8nQQ1mjduwXjwlnX+VLUNeZzeL7bsDNbzWCflhdaiUAfgeptgle804W8KIvY08D/1QGQ1mwtfsSTnfhpXJ2DO5cT1reQg4oeEO0GZgTpRt37BWPR145Re5fYzJW139o1BFmjuQTQac0gRFU8tO/B1XYM/+qCG5NtFmOyy35MtrHePMlABbrj2Dk1e8vWgd1qlgH9WxQdiLvj2KLj5uaIUbuBcN4CwHkDfGmb44C9fPs2+ubV1z9+80qHoT/+692rD8hrByqnRxP18GI31Nh7ZruUd0M9LytwjFZ4uEM92bZpsKuC2j0YjmazPh+I6TwMxfl8MI/H0/P+vt2DnQ1v20bYWYkQWn/UJ7yPn+YdMR90jQ+ygvcVL8U7iq1Txq3fPjezSuIdB6e6Dk/hbjcwx5bYmB0vVFpKnpxottYH/87JVBcqRK/u8sAMs2sh1nXO1LV8qJKp8CY1EGydNIjJh0gH89/YYCV/KFsdy38G4RYaw8x03HbDfc+Uw3qgupHAD8ytCijo6wKt8z7HoIH7vWHf0eJ4jNu4lhqMDgAxDByZuX2HnfkhygJR2j0048tsRCAntkk6+Xc4S2RyxY5cokts/Wp3UtZ2BmsdJoLnDnc7PI3Xpu+Xj/8DkfaXd4ZpAAA=' | base64 -d | gunzip > req_split.diff
echo "2e63bc1ed4bbcdc83bcdcf3a51304dc8151a59da1facc1019e4287b8b094756b  req_split.diff" | sha256sum -c -
patch -p2 --fuzz=0 --forward --no-backup-if-mismatch -d "$SITE" < req_split.diff
grep -q SGLANG_AGI2_QSA_REQ_SPLIT "$SITE/sglang/srt/layers/attention/qsa/pair_layer_split.py"
grep -q agi2_free_of_parity "$SITE/sglang/srt/mem_cache/memory_pool.py"
echo "request split in place"

req_split.diff: OK
patching file sglang/srt/layers/attention/qsa/pair_layer_split.py
patching file sglang/srt/layers/attention/qsa/qsa_indexer.py
patching file sglang/srt/layers/attention/qwen_sparse_attn_backend.py
patching file sglang/srt/managers/scheduler.py
patching file sglang/srt/mem_cache/agi2_qsa_block_tier.py
patching file sglang/srt/mem_cache/kv_cache_configurator.py
patching file sglang/srt/mem_cache/memory_pool.py
patching file sglang/srt/mem_cache/qsa_kv_pool.py
patching file sglang/srt/models/qwen4_exp.py
patching file sglang/srt/session/streaming_session.py
request split in place


**The session fork**, applied after it (Python only; inert until `SGLANG_AGI2_SESSION_FORK=1` and the harness's `FORK`):

In [10]:
%%bash
set -e
# the session fork (fork agi2/feat/session-fork cc16a7a200 over c414512aa7: streaming_session.py + the scheduler merge; inert
# unless SGLANG_AGI2_SESSION_FORK=1): a task's attempts restore a parked template of their shared prompt prefix
SITE=$(/tmp/sgl/bin/python -c "import os, sglang; print(os.path.dirname(os.path.dirname(sglang.__file__)))")
echo 'H4sIAAAAAAACCrVa72/bRhL97r9ioaIoeaYYyU6cVImCpr02LdqmQZwecAgC3ppcWYQokiVXln0f7m+/N7NLcklRjhPcBYVd88fu7OybN29mmaSrlZhOr1Mt5KPyTq+L/FF9ncn8+lFd6UdbmctrVdWP6nitkl2mqrC8E1cPffIkzRN1K2ZSncVPV8+ukjBcza/OZk/OZ8lczGezi8ePT6bT6cPnPjk9Pf2M+b/7TkzPn55dBBfilH+fzQWuxZmsa3HZPOmdCPdfuhK1ylZhkmXbKC7yVXot0lrkhRZvilwJmScCA+joSup4HVbqr3rRH4H+xetdvlFJhNuRLiJ1G2e7RIW7MpFaeYP3/RNxcuq+/ZWQ1+kZ7KjrtMjFqqg2wrt8/durN6+jV69/OYsuf7y8/OWPN9FPf7z71V8IKbTalhmG/qYWMHKqi43Khd5VuVB5UgupRaprUVZqlWZZKP7EzlRCr9Vw3uJGVZksRVYUJb9SqXqX6cYDZVXEMEol4k7pQNSF2KrqOs3hIi3WqlJiX+wyctAuj9ewK1FxkShRa1ViOpgRF9stxh3OC1NELbEGDG2MT3NdsAW1xrqei40qtSh2mNVZSWPeKs3Teq1qMgOTqFsZ6+zOPllsSx32J1zhojD4CYGfsPF0ui2LSmPKSsktlhU1N2QtItqTqIbp/bEAmOZWeLg7i/7DPHlRiajCAg9xdPAs/YtqsRTXCm7QlRdVgZhYqyYBQ9Iffw12eeN37KAOqo8/R3Dvlvf+97e/Re9fvSbj4SSvtarurIrSBIZNJr7/iVGrkHec3FzKSm7rcCtvo1ztI0YAVr0U8/Ex/MXxsY+EnkwSOA829YP1K/FTmmkEA29D/57ZHvK+s1H8M6rTfyvPH3ucH1rxmOYPb9ygZZbW+sg9uC65n5qtrx8dIPUYQ9/3giXq8ydxPJNzdbUKwyfJE3V1fjW7evYpor534GN8fe9LTNuPmbQfB/MLomxg+f0/3/4Y/fDzjz/8+sub12DcE3DUNTifdgf/gzFCoPE3vuZFUS63KoqYWS2bHiXQQBTw9dUdyGolwSaGUGW9sdzRkA2oqchjRXFTqR2RIN5RIMw7cA4RsA7Fq5aJad6GPTwiojQB+eVapkB2G0i+qHb4m+lvLSuM6c55ayakaMGkCBJA5TmMa8bdr4ta0cBpTdNNXjST49rLiTgVvD4O2FPclTk2Ao7CLfCmLvADg8UFIFOsDD2bOQznCiJEGL5Kq1qbZLJfK3PNRCivUUsw5j7Va16Fk4jWKc1xB8LOwf8yoUmsK03GCMV75w0wvCprHh3L0sYuj9eOS1d3WtU+cSdNSjPl6lZzajTD8OPw0r5ZxiO7RiEzgCy5E1sktwW/ai2Djb/+I6CUBLM4q5lA2Kg78Bk9uJXbK7g7KzTN6mH/buDJy8vfjYd83hp6sFTVtCr2osLKauHR/FhFsal9zpLS5HDKVls4vclPe06Zkq6nicobJ1MKRcI6RCqwHkUmR0WRNynqiR+q/CatEDgAvzc5BnEi5Bkezoo9osMn+vYmc7qqq52agBU7al+KSaixJeEEV1v80FV29oQY9MSKqOgfaaV3MnuDJG9V0GQyucRS0lxlSC/I/ZTu2jhvoQvC22F/6pDDfT4/p3ifn50Hc0ejvWU0XpKr7egV9ESVLMQ79devN7/kq6IVD9gj3k/7CGVscOcevzj66lhmsgKWVQY9dF2A7+MNIZxetkhJ7LuW13PG3IKUiLlggiOSeiFWWQGYLcUsnJlMlCZ48o9SY20y+/AbAPYB7338iGdYNo5puiEYrfCBfV4bFPZ5knXZnc2oo0MxwMgLtYnkdZERNFMroERG+CK4Us6HemsXHfBVQBHGVCa4m8krlSmJuGimhUO3Sub7dZop41aOVPYTrYVEgbxDQO1yDW/iZ5rxGik1iquiqgh/HfkAGUlVcNDbpTXPGLeTg831oirXMlfY+auiyHD9J5nVioDYQPHvlVzpd4AYgeZnhN7CIOvZU0LW2ewsmM8c9W98dwl/WWR9Jf6oSBQzE1qLraxMTFzKbQEAS6jOdlM8dZPGGoonU00GWwjyGbk3hscR5O0gzKM+5Qh2UNDMSxsCGs6KeMNkYVLK+7eikjlyUBpv6kYft3gIzcu1vAEga/XXwF2jCOlVCUaH605uE+pYY1O8snCGH/0+OP5lIuBfxLPuaM2kDtIg3hl8pnahJRosmjAt9rkDHbv3rDI/EUfOk2ZRQzjQP+Rx9kxEMp+klUcVXUCcw8yBjFRHvBvmfd8p4Ii+8CoRiVkNlwrS7iIxmGUuU6C0iwrxolUuTGXnFwPANQxoked9j8B6y0n+B4lStLGBbNfVXbRlmWlUgDXfiOSF+J1umVff8iVfTF92PuPb77gu+thf2DtFOKatc54RJDQNCZp1rWnzJeAe6/SmBRwv8nk3XEGA2KeQH0xvHugVtbRY4VdtuNWWb5XcC16Mzy46dcumT1VLDJ6lqcejdtMjMsUz3vAPajF+xyltRiqFiv1gh7WOZmDbQQMexFH3VhU1ltg84JhB5ZvzPMywr/QM+UwjzJ8+o+rxt88IVU/O5sH8/PNRZcITvJKu7ki1/mceIKhTlOimXq/XkBTTGBR7De2YY2tTLSE9wVY/ZCjSaDOxs0WMiEjcIXGdZJtDJy+W3ZPMJcAwVqJo6yLTDgi7ETh0Njf4L2qHiEAg8DWW5UFbXnOjoXusHZwe8z+BJ1No9qrWtkhdHPRcxvnR5mcSFC1VMkI4aggp9TeNXIfLgXwkIs6knWjvTxUTko4sfIDnhKtPqhSBsLCoUhQ6UZqXO0o69YdF/PEwAjCKh5u+eCHiEfTTmKfuoDvdDgfTpu37w6HZ5IakYRb9dKdDuR4LlVlKaMjY+PZPbnwRx0QkCus2P9S7mDpKq13mpkrLRQ5QyNIm95nHmNg57ihAnsznwVMEyON58O2XxQeWQqEaITe0cpXHHw3cPoQbamD9xVLctbfDnH8ydfuM9Oawt0gvbm5Ek1Y2N4sexkff6XaG2a+9NEJCLdJRiTK/mzRaFZA1QCwx92DXO/IlrcYzHFmdiVMHNSr7jFUOnPxXmCiNPYo2N26jxZ1ZXlEhRDAYf6DRDISPi8cXhI+Lp8+C+dkXA6RZy1E+P3h6fA++6jqbdMeWIaSIFqyISxIauqlqV5Xi1mYHnnZjbCoyCny6dKcMTf1yQA99sCwMHuh/wXos3YxZpOTrXjnwTX1IJg46drkR755jw4BBWjw0JrZyX7wUMzaF+RSTU3OXS5gULmkbAPfWF5nSVNmNGOnM2BQSwMZ7lL5D++oxvfBQT/e7gdyJClPUqN7h+cDkrW20tOWwqbQW4mvo8K+tKkee+Tqcr8Tv3zM2/ElwOJITfgz0b58y0OczlNTPvgzoZlMJdaxxIu6CeCa2h5kbfmxrcW94u9VNTiHv9VGcL/NAUBIsd0vzKxDqVldyyT8DW5Ivza/AFuVL8yvoSvKlTrcqpB/eAHT/jxmCgzS97GXHD4v8Yz/WOFkOj5GQOXtP+OLlUuRdDnW22z+5B45I50MIbilDe2xyqapVZGKGej9ToWe++JuYq3MDGEjL+ZwQc34RXHwhNzqK1iT/ocylFpo95aISZ6jmP6vCcYr3j064orw40mJxOpgo77v+KTWJk0Hb1HS5XuQvJ7a92t1s1Bx3O52srN3WjamX6UUqcu1JRr9JuqXTHCrDOJ1wtybjY7pUD3uobj/2eTdlr2AwnZet3JgmTCdYTYm9TrU9bzFtl+YtpJbUnJhxJSsF9S/hHloOTA/FK9sudvM5loVivrUwMKeCPGBgasKWuaSgJyp2u29xYDOnWSVth+0H16SurxRcr/pVIrI5cNwVWT0tdEyunR5DZXujTomVxlVMX25Rcz13xB31rLquKFlAtw+qifFZq/hQKOJd6G78DLn57LWD9/Q3mYK3rfQgI/Cnoy0Or4aWcUZq4FHjjKbvv9/dJS3VCP2DPRjW5wYBUbFpS1l6bWCEmx8nl+5JNyCx66XCJgDbnMh70mAIWZHcBygGbKIFy0bdDU8gxzfFyHeHUzzjgGXni6DrOC1JNfi9PT0UM6e9g8tuwz/Azo+2+OvuH/UEe0BzXLeKwPSjmv6c6rzTesF3o6cj39Mh+XZ6zXDvws3STLgD6CAwX3Vt5kZuNZThkJajIr22tW57vSVEYT/ES8dxqPrlrTcLehenYt7HXNmpOCK23gBL1pH2UKBFTnME1MisgarEKvJiP1L3DEVfOSqqS1dVmk7kgbe72sm42+lD0OmIKZ0WLgyPbME7qtS4wTas4sy2eJ33mZ1tz9Lsgm9WbSfHVvEhGoOEWxfg4WuZ5geduiFndQVtU8J9bh16tMh8QBmRqKwXVK01Hx9MMQUf55IbKcgm7n74zeZ9R05M463S6yIZlS5DohtRLsHRkyHeX2o/L8Zplvw+VI2cgGcHAZ5T+dRKSocAjZ7kG9Tu6e6EbHojVbl3w3zfNaH7PclP6LNj8H1AU/pVq3EOQWnQ2ugZPnRoJI059JUJHdlc7RIWVJAyWbqhA9Xm8yBzmNPN10qjnORKTi0n8JLsevrAcULfJwUi5k52mqc6ygoJDpB0PGOapp10oqscLeJBimUkQB6WQN1qcySZUglvjl3ac8Out9CukrHCcpP1aG0EKbmYD6SbDOORPzR5ifuYUsA3u2EPEzNS9wV8fQfzsQ4WfzDMtJIM+/bYiVz1fFz5SWz53gEAN6X/d22oo8UJdoSG0EVkWooHXRoYOeV1Ap8enkEdyKcikT3HqnaZ8s2iel9O7CV/OGF7xC2eP9WLube/8tBlH/TR7m2XHWmZHS3s+Ex1dsGf58znT764x8DRWBSbkILLA98HIpFaOpZxb4Gu8eCKggnQjtWgm1KGtnAf+4RyYT6ucA/3haUKcwjJ57e07q0sCfiOimkjqJMw9DFOSadcfDzU7+kX2xL+yXXEQ5pTWYRkvUzwp9dYGR59zg8ONiUqkR2ws7dL8k//dkwOjcqq0Cq2RwfL2eCZ4dnCMg/s/j19wj2i+cXFFx8njbaCNjfHOkAMnulniqtj3cqDXTo4cl709d/Yp0W9L4uOnXTYE0gr6j4lWcph77s86HD27DcfBJFAM59UjDQ/7QoXnVJtP7UynyjglXvUaL/HOdLfvH8n+h5pGgjgy/bsxWWzvcQuvBD5otHf7XdNpdRrS5cLURZ1qvnziZJWYL6z0vS5cKW41w2uNd1HLdNswFVOb5LuepubgOf1T/4LrXnRvM0uAAA=' | base64 -d | gunzip > session_fork.diff
echo "00453989f7d67acfe0ea23d03515870eceaed3bc0b69fc0dd42e056cdc8f943c  session_fork.diff" | sha256sum -c -
patch -p2 --fuzz=0 --forward --no-backup-if-mismatch -d "$SITE" < session_fork.diff
grep -q SGLANG_AGI2_SESSION_FORK "$SITE/sglang/srt/session/streaming_session.py"
grep -q "_agi2_ss._TMPL_TAG" "$SITE/sglang/srt/managers/scheduler.py"
grep -q "def _agi2_drop_fork" "$SITE/sglang/srt/session/streaming_session.py"
echo "session fork in place"

session_fork.diff: OK
patching file sglang/srt/managers/scheduler.py
patching file sglang/srt/session/streaming_session.py
Hunk #10 succeeded at 1162 (offset 9 lines).
Hunk #11 succeeded at 1175 (offset 9 lines).
session fork in place


**Three later memory levers**, applied after it (Python only; inert until switched on, which only the slot-ladder arms do):

In [11]:
%%bash
set -e
# the three later exact memory levers (fork agi2/feat/mem-levers a65140400c merged over c414512aa7 as c756b83ee0:
# expert_location.py, flashinfer_backend.py, dflash.py; disjoint from the split and the fork; each inert unless
# SGLANG_AGI2_LAZY_GRAPH_MASK / SGLANG_AGI2_EPLB_MAP_HOST / SGLANG_AGI2_DRAFT_SUCC_COMPACT = 1)
SITE=$(/tmp/sgl/bin/python -c "import os, sglang; print(os.path.dirname(os.path.dirname(sglang.__file__)))")
echo 'H4sIAAAAAAACCq1abXPbNhL+rl+BUecmZEzRkuw4jnrqxHXStHNJ7Enc6915PDyIhCScKZJHgJZVT/777QIgCVKUbLfnZGyJBLCLfX12gYjP52QwWHBJ6GG2kcs0ORSLmCaLQ5HLQ5bFs0N2n7FcBnEaUsnTxM82ZPb0sT2eROyeHJ+cHI1nR/P5K99nr4ZDSo/Y7JiMhsOT4+PeYDB4Dv3ewcHBs3h4+5YMRq+8E3IAv1+Tt297ZJ6nKxIE80IWOQsCwldZmoMUkiSVap7okV759D8iTXoH5ksqqudxuljwZFF9z2kSpSuzeEQlDWMqBBPl++qR4uj1yZE3OiIHr0+OveMT4Iqod+RTGrH4PE3mfPFTmr9X2/lodjPpEfsnZ8B+Qj6nCUN2ewcRm5OALvg4WNEsSJNgmQrp9A5w8IpJihxMyEWGa9H4urn4JzPgxusduGTww6PjJnrhfr+PJInz9cPHs88fgrMPv4yD95cffww+nV0GP198vfJICoY22xDgjxaxdCfklrEMBchDGgcyDWgcB9lyI9R3YJ6kCZFLRnAD8Dne+L0DTS7gCZdBTtckoxHIVhKZEiemG5YLr1zSI0mxqhfURiFcwhN5ckycV8fkE/+RUPLh8ldCJTk+JXoBck+OTo9JOd7TJGm8QjaARzAkUoBJ50Tm/I7TmGQxDdmKJdInv0ihFswZjXAtmjO1h4iLjMpwCdTnwCrLgvIJbHuR5lwuV0Qw6Xpq+EcQnaYr0vgOF9ox5UWcvXDBZiOC0gayMwq+EIJJEqfIQEXMgSVZQmcxC9A5XJ/8BjNJAgYDKkFqKxAuCCuVS5yGdo6EkY1yJyji79WTOWdxhJ8SEAWnaNn4+BwGhmm28YBfI1ItAUKjiEUgWQmf55THAtRTRPGmVG7E7njI/MqM9Ac+r2yVcKGsm6Q5OJ7Pkjueg08vmHT6O62t78Farh+na5Y7Lu4NBE+c/ghfyLxg+HfDRN81Bmy5UknYuEzKyJQANTAgFjquDw9qHlPmd+sFmEaiJeNqYK0EiygGEJb7a5onIH2nfqHkoZxKadb4A/rChIhbnmUg1m6jmP5FNFQO35U20E+M1EFBbt9rEtu5ma5x9erWW/cRWS6yQvn1tHru73F/PWeeM9jotJzrg0+zGFT6snrCYuV7geC/g7GjWso3XARhEVHCYsHIsBkC91F+IoNBmBV6UaNE9G1Lg7u0t48yqOqWZZWe0K8w8hHnL/5oXgeshvJkkcXMKTctljRjrvVaS/CQjF++HA/Nc6OpLS3BPwzQ4MurrJBMxVleBc8qowblDKdnTALyVRCqhDWxs5dXvkdbgYCd3E4w/sLjJ2YXNV2FgbYT+ioHtBjT48EE9JQpOruK0f1Jb9CyzW6aelkzy7E35tnb2Db1XTm3/Pl/kXMxzePPd+Tq4t0FpCI+35A1xnS0mEJijJ0XSbglD/DaSOBAp+9nEkNftB/86Xx4SKUE/4LVDueATiBJzFkezGh4C+vtwoJPm2qg4dHpaTQ8jkajse9H7Dg6YfR0NnoMGj6RxC6k+MTpCNPGIwSO8Hs0RIym883X386Ci88f/xn89uXs8vL9lyFY6L78tDW+ylE9UOUckoSIeQQ5YLAGqaRrkqdrSPsreh/MEJM45vEBSdga8A4wKODL2EVEIyTGd0jnOBxhj3l/D56cSHYPHsKS78m7n3CP4xeCRDmdSySskY+IUykmZOgfD8mHEhShFPyetd0PwPnPwS/v/hGcn10+YbuN8a2UvCMd9w6ApS0k+fHsX/80q306+/q3DiwJgRQ9S4Ot81/fnQ0WOc2WJCyETFeDFRW3ZFbMQblapLaMkKYlJlJAhDp1ddxdK6TTWCbL2ZwDDFwDAXBpTPVzngM2DCHJShb5bTETAHDoo3kB1ND/0gL8TVFVyxJcViEnZN5wuaaAIBjMVKl7gkqq4SpqxtYccUZvxsYmAO69OdFjJXwCZNUpw0fU1xr9ROU9oezRQxjIv3z3Hj57hBYyNS9V8ErTuJqd0RyzkC7iRq9PvdEI6rjR6dgbD+uiSQn9F/TfMymTH7ULO2elf5sHbqt+UhibxXMfkUKgjCa4vYPEFwEsFdf85np4A7IallHXxFMEd2oaQrHAmISVY7qW1doOlBFNQVt5uPR/Z3kqnOY8/Gkb6Uu9Gj61TBXS/NbMSG4yNtWrK0vuGqNA97SPzPVb791m2lJkdW7Tu0D2A4XCECY9hcvmgiC8ThObNIfpDId0a+eO0OcD7WalzAPjhqKETMYXLYctHdURTMcHiPQLQmfpHXO3ie7Xmqqzt+Z04z/7Z67BYEx/3xAdmXDBiWLnYZ+MDW6bAPz7pry6I6L1d9F0HpoK+gYJ4aFLRd9cCDgcAgnmIdolvQ4aLeEhyp48V54NL+jcxZM9oXN2hzd0j2t6RMde927qv7cYMTKZw5au7/0wBjOBOInSvMdoqcbrsAKDbh5ZLH3mYjowjscnqr81PnrjjV79mbj4HfkJaIWquo/5LYOE9vHs688mmznvP59fvHv/RaEZsM8kTQYhLQSNwYigoE/J54srUkDJRe0l27kOhq4K7KjcpRxKU11o8N+xBfFvy0YCsPF/Q8IHR8nXXLAGl9XOIK1CRpYYGZRLOWhRbukfkEopptFYUVfdkhlEFEyhRgh+rXGIT8B7w0xxwj4zNh2KlvE/Aci4k0dCVhn5hA0NAK3PQPeiSW5vI2FfCDI0UPB/Lhr1u2gannfGFeIAun2YiW9ua3pXHvqDYcTZsyUP8NJ2kGjGgzZj1ccq94CjWk6Nm+XoqVC6LZimjhGsHN7GIBDDcixwnccKMlUfisNIlSq7Sq/2IFNkhexkOJ69nrPXvv+KnrDZ7PXszfyxImtrsV3l1NZADdaOXquYhH+PRhiTVHtahGkOtW20YMJ0EFzdvH6rlYDtBx4zJ9okdMXD6RWgTI+YqmyKfQAzoirVsMsZcYFdqWnARZBkhWv3wi2KARZWxkpemlAPYCJigPdEmgcSF5kYk7qClJPmZpQowmeMQTKdQ0KIJxxbswGPukcUCc03AfaJZPeAJY8ilnS+okm4RA4xW+4kINMsKFsxutFvD6qb+rbc6h5DtUddo8Yp0IpIkaHJU9U7oiGiNSWlZgB89+Xsp6vg66/n58H5xafLs/OrsgPbFBx4Q85WNLtuCOsGwwALpQ6Hgq6Y5kAF0QIzgKK4ZHFU1VNqB7oRzyOM9iGN/arRfMs2SKul2esmMzfbRgLMVGEHgqcVdK7bCrieeCo96N83PrvPYE/OYOQR+K80Acbb2CdOmQxGNzcWEon4ajpqtu8shpCbLSO+bnF80+j62UamCBoGyYHZF+OJKFZ2Q3MWZ7k3i8N88AN8DKHwa7Dw0tilvZqnRFyxjQ3G/6+T64gyTyGLrbH9Jhz85ZWIoqzkPKKMWNl8GX91YJy2h5rQ9UaHrvEYysxxDafOS0V9VaaY5k6S+J/SqIiZHdcxB0jlTapabqjXa8jeM1JrZwUtpUCJJwW8m98GAg8mQ0evC+bjPnfGuNG31K1ukDTgoFylKajkTbwEBwjUiL5WpNtASHqudcjxBOzT5fq12WsPFcVMMEmUjNbLVFj+bcJK6eJp3AZATwogzSm7BdZcTIn6D84ct2Y22sU70pL9s+XVU4Un7Mch5N4ZBOGOuqUV2KYWElIqNlLdOxMZm7a2tT2+Iedp0963R9sOMG14w/ZY7R5T/afjfTvgTtsPOuaosKuloT56OzFeqa9tzPKHNPS4duqH9Sxd4A1NRDo+9Y7ePDsidW/b3rKWAcZTZSDGNoKKIRMitJNuYwuFIyC8WrEA8uz+EGBuAaheawe00AfUuvB8IWobMzzUhDoizd8vzs9+dH2omngcBTHEOAju5jh7W8aaBSgfaxoqCmHtZwGfJlVYk4ZLQDrYqN3YU7HirIJZhUWaMSyLCxx2B9XXbLAGcIJ7VkeFC37HLIyDGdSqUK/guWVcZjUIPwBlIGgiMMWuMBmCxOxxXC+putEcmNZugoGWh0tknsygZk82upFsETzHWw9pEkJ6Bj3gxQdYZs34YikR/dFIl9QMch6r9WWKTI5qy/COTeSTL8qZTDW7waivTidrPGbyyxNz0p4MZNx2aB2TKjFNyQ4ns+hHGhKWVg6u4ph6UK3h6y/NujFOk4UV6ZVSESGqCepM9np409gjrlyeZk+nZEgUVJAOPl/xxHFd8tfWU3qPT3+Y6uXbO6Yc8uXfaVyw93kO3moKfq2M2rGMEeJVIivfkrSQAm3weuiRB7U+lOTuTo5/ICOldRQ/iC92kMPr0eQG3igEC/DV9cFwgOPHGH0Wn6ppBIYKQyTTllck/L8Fs5n9juDFC11XY1cAPWsCvoa9qmot4uD5ghAc7QKcIGaWkysXNu7tmqih+gueOZahNrUEnWGWFjm4HJCqX5Ue32xROLYkD1QZYNkJCK6yLW1v+LnqSdg26G5Rup7YarqpTNAcNgPgDXQN5YCeYWQHINScYj3lOMoQsEdiLbpt+I/xppGYKuDK5anujfypZcu7Ifp1fTNEf2/eC2nPGkxLiVkTyyetKyUHegPWQP19F4UdIItUJHcPLXWg/rZil2/uQZjqbJXJjYNa/PO28127ggbjX6iSFa9zJZgbeVKGfKLvlVWOmLN5IXT9P8/TRBKHCmKn45/Pg99OSZQysSWjCHLRHYsCvXIQorsFDIOCaoc1L0PtRxJ238GCDHY8MZmeSpW0vu/clmi3MftYl6jjviLLVMzBNAbZNpfVYS6Xfn/fNShlclj6lvgK0noWsyCjcmmBSZXt6q8vDe47Mrjv6HTkHR3VuM8cM7/Dnao7N45+Uj+w4d/aI4sy+2mDU74NZjfrdR5kNdKHzmc6NE7Iw6L0hW86d+DRG9gMXlUhzsPa8pR125nsFnK/R7oPrrC37OrT4AcU0rdWJtp3dL1tGk87vZ7sPPjcBsLIE8jTLS+k1o3GXaAZZ0Aek4C3UEo70HMLyDwKn4mzBf79bjbMUcMiTmc0tnKcujOUQJIFrG3Fm6WCvwbXXl1qzcKCBWT0TY17HUVMI+zKnGBlxOoQCviiSAHj4vm9wvKYlE3ejrmQ9qHdguIJD7zQOBPijqatCJvdCLzm2+z8aQhiIW9qZkDUYiuutrjikOOTha+A85yr8FYTRtdWtyPpgqJlfLi8gu93nK3Jh1Oi+0qDIvNgZoLXc6Cg98mZkZlcUlmCUNAtDyVIhyd4RI0kVXFTtyfbWBQWsDZJ8aitiYRXGJ6mlcFA4Ea5O65CpLvBpwbQEF1y0Bb6oS8z5X6I2lpmnqi7c3GVVnSnyLlG0hWK2JOa1bithKIcCAiqJqlaMdB320qe1M1HrXPHsOCpFucQN4fG4bSWy/RN4iY8QUSs6LiIUbCl+gcY1SsDcGrsGW838PZhv7bkJ+xJr1ltCc3JiVlSMgustphoN5SvFalrCBcTLcnr/OZGdRXz+mSpXtEFsB1mhS20jvN4TURtszU4wTZojcUatpRU5YmTPAv3t0Nq64jycdiPjoPGL17UIceELq2JKFWMgFhWdnDZcSGijKpoXt/XkKemXYEfvIPb72zcbaNPfTJZ9bvE/jjsNDH3zoTbLZCH5BvG0YddNNtIsk7TGC133hNR+4Wyr3FVuD7kVY/71cVTleygskoEyj2wu3cm2XWdXHWcMtmGwZVf6Q/+HPAZBoD/AQEpg/idNAAA' | base64 -d | gunzip > memlev3.diff
echo "11e43b03e5039b22abe64dc1462dc7dcef73a57521644c0da817563ad6ca2817  memlev3.diff" | sha256sum -c -
patch -p2 --fuzz=0 --forward --no-backup-if-mismatch -d "$SITE" < memlev3.diff
grep -q SGLANG_AGI2_LAZY_GRAPH_MASK "$SITE/sglang/srt/layers/attention/flashinfer_backend.py"
grep -q SGLANG_AGI2_EPLB_MAP_HOST "$SITE/sglang/srt/eplb/expert_location.py"
grep -q SGLANG_AGI2_DRAFT_SUCC_COMPACT "$SITE/sglang/srt/models/dflash.py"
echo "three memory levers in place"

memlev3.diff: OK
patching file sglang/srt/eplb/expert_location.py
patching file sglang/srt/layers/attention/flashinfer_backend.py
patching file sglang/srt/models/dflash.py
three memory levers in place


In [12]:
# keep_384_c512.json: 384 routed experts a layer, re-chosen from all 512 on ARC harness traffic
# (fn_expert_choose512.py, 10-30): KL to BF16 0.2475 against the census list's 0.340; written to the
# path the switches cell reads, so the switches and the seed's fingerprint stay the same; stored compressed
import base64, gzip, hashlib, json
data = gzip.decompress(base64.b64decode('''
...
'''))
open('/tmp/keep_384_code_bal4.json', 'wb').write(data)
check(hashlib.sha256(data).hexdigest() == 'ccaac87ab41f3038aff0a983856f3ef44afb3efda1a164d1e7612224539cbaab', 'the keep list does not match its sha256')
k = json.loads(data)
print('keep list:', len(k['layers']), 'layers x', sorted({len(v) for v in k['layers']}), 'experts, sha256', hashlib.sha256(data).hexdigest()[:12])


keep list: 48 layers x [384] experts, sha256 ccaac87ab41f


In [13]:
# draft_vocab_32k.json: the DFlash2 draft's candidate head scores these token ids only (SGLANG_AGI2_DRAFT_VOCAB, the
# lever bundle; docs/reports/1029_0021_upstream_sweep_flashnext), stored compressed
import base64, gzip, hashlib
data = gzip.decompress(base64.b64decode('''
...
+5fe/qW3f+ntX3r7l95e//s/LNBpfIJVAwA=
'''))
open('/tmp/draft_vocab_32k.json', 'wb').write(data)
print('draft vocabulary:', len(json.loads(data)), 'token ids, sha256', hashlib.sha256(data).hexdigest()[:12])
check(hashlib.sha256(data).hexdigest() == 'f72ceca4e3822fa7fdfdab6c5989cc0142e4af7fdde197cc22fc083e492de098', 'the draft vocabulary does not match its sha256')


draft vocabulary: 32768 token ids, sha256 f72ceca4e382


## 6. The SGLang server

The S64b server (the x15 probe's arm S64b, Kaggle run `1031_0445`): submission 19's with the sub 20 build, the request
split (inert), the session fork and the three later memory levers, at 64 slots and 2,048-token prefill chunks, in
Kaggle's paths:
- **Model:** RadixArk's NVFP4 Qwen3.8-Flash-Next with its experts **re-rounded by GPTQ (QG)**: the same files, tensor
  names and layout, only the experts' `weight` and `weight_scale` re-solved against calibration activations (block error
  −63.5 %, KL to BF16 −25 %, acceptance x0.995; `docs/reports/1029_1845_nvfp4_rerounding_scope`). The launch builds an
  overlay folder, RadixArk's files with QG's 192 expert shards, and stops unless all 192 are in place.
- **Model split:** tensor parallel 4 × expert parallel 4. NVFP4 experts run as Marlin W4A16 (the L4 has no FP4 tensor
  cores); dense layers BF16; FP8 KV cache; bf16 GDN state; the embedding and the PLE n-gram table in pinned host RAM.
- **Experts:** 384 a layer, re-chosen from all 512 on ARC harness traffic to reduce each MoE layer's output
  error against the full model (`fn_expert_choose512.py`, greedy; KL to BF16 0.2475 against the census
  list's 0.340, all 512: 0.195), placed over the 4 GPUs by harness routing counts.
- **KV pool:** the QSA pair layer split stores each QSA layer's KV on one GPU of its pair; the GDN ReplaySSM fold drops
  the speculative state snapshots. The QSA block tier (the build section above) at 6.0x with a 32,768-group stage cache a layer and
  INT8 + a per-key scale index keys holds ~3.47M index slots a GPU (S64b's boot, `1031_0445`), cold 4-token groups waiting in host RAM.
- **Requests:** 64 decode at a time (`SLOTS`), decode CUDA graphs for 1, 2, 4, 8, 12, 16, 24, 32, 40, 48, 56, 64 (they must
  reach `SLOTS`; a batch between two sizes runs on the next one up); 66 request rows, because a conversation resting
  between tool calls keeps its row; 68 GDN state slots.
- **Session fork:** a task's attempts restore a parked template of their shared prompt prefix instead of prefilling it
  again (`SGLANG_AGI2_SESSION_FORK=1` here, the harness's `FORK`); only the prompt is shared, never an earlier answer.
- **Streaming sessions** with retract, evict and park (an idle conversation waits in host RAM, up to 8 GB a GPU: the
  tier's host rows took ~13 GB a GPU at submission 19's 7.5x, less at 6.0x).
- **Speculative decoding:** DFlash2, 4 draft tokens a step, with exact turn boundaries and the sparse verify accept.
- **All-reduce:** our int8 CPU reducer, 8 threads on the machine's last 8 cores (T8, burst 2), a 4,032 KB envelope; everything else stays off them.
- **Memory:** 89.9 % of each GPU (the stage cache's scratch sits outside it). DFlash needs at least 1.0 GB free after the target's graphs, or it
  drops its own draft graph without an error and runs ~14 ms a step slower: the check after the launch reads it from
  the log. The vision encoder runs in groups of at most 2,048 visual tokens.

The launch is a file, `launch_server.sh`, so a crash mid-run restarts the server with the same switches
(`restart_server`, used by the harness's `ServerGuard`). Its compile cache lives in `/tmp/cache`: SGLang's
JIT kernels, Triton, inductor, TileLang and the CPU all-reduce's extension.
- **With a seed attached** (this build's seed notebook): it unpacks the seed and checks everything the caches' keys
  depend on (`jit_seed_check.py`: the GPUs, both compilers, Python, the venv, the patched SGLang tree, the switches,
  the command line, the model path included, and every seeded file). If all match, it boots once.
- **With no seed, or one the check refuses:** it empties the cache and boots as submission 12 does. A warm boot with
  the plain NCCL all-reduce compiles every kernel, then the real server starts with our CPU all-reduce.
- **The weight loader** prefetches the checkpoint files except the PLE files and RadixArk's own expert shards, and loads
  with 2 threads. The PLE rows are read straight into their pinned tables.
- **The lm_head stays BF16** (submission 12 ran it in FP8): the one lossy piece with measured drift, for ~+4 % speed.



**Which features this server runs**, as environment variables the server inherits:


In [14]:
SESSIONS = True   # streaming sessions hold the harness's conversations (section 9)
# The CPUs this container may use (cpu_count() also counts ones it may not: 48 against 46 on L4x4), kept from the
# first run, since the restriction below shrinks this kernel's own set.
CPUS = [int(c) for c in os.environ.setdefault('ARC_CPUS', ','.join(map(str, sorted(os.sched_getaffinity(0))))).split(',')]
SMR_THREADS = 8   # (sub 20: burst 2's T8, -2.5 ms all-reduce at 64) the reducer's threads get the last cores
KEEP = CPUS[:-SMR_THREADS]
os.sched_setaffinity(0, KEEP)
KEEP_LIST = '/tmp/keep_384_code_bal4.json'   # written in section 5
# The final Modal round's server environment (config l4x4_flashnext_nvfp4_prune384code_16slots_kvpool_tuned_bf16head_
# loadopt_dflash2_mf90, round 1025_2021), with the block tier and the vision encode budget, and sub 17's switches
# (docs/reports/1026_1946_structural_decode/SUB17_SERVER.md)
FLAGS = dict(
    # exact pinned host buffers above 1 GB (the PLE tables and the session park; no 2x rounding)
    PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True,pinned_max_round_threshold_mb:1024',
    # the KV pool: GDN fold, QSA pair layer split, vision blocks in host RAM, the reserve cuts
    SGLANG_ENABLE_GDN_REPLAYSSM_FOLD='1', SGLANG_AGI2_QSA_PAIR_LAYER_SPLIT='1', SGLANG_AGI2_FN_VISION_HOST_BLOCKS='1',
    SGLANG_FLASHINFER_WORKSPACE_SIZE='100663296', SGLANG_VLM_CACHE_SIZE_MB='0',
    # the experts and the weights' layout
    SGLANG_AGI2_FN_EXPERT_KEEP=KEEP_LIST, SGLANG_AGI2_FN_EXPERT_KEEP_ANY_ORDER='1', SGLANG_AGI2_HOST_EMBED='1',
    SGLANG_AGI2_MARLIN_NO_SWIZZLED='1', SGLANG_AGI2_MARLIN_INPLACE_REPACK='1',
    # the PLE rows read straight into their pinned tables, and the loader's prefetch kept off the PLE files
    SGLANG_AGI2_PLE_DIRECT_READ='1', SGLANG_AGI2_PREFETCH_EXCLUDE='plefp8',
    # QSA decode on sm_89 and its speed levers
    SGLANG_AGI2_QSA_TRITON_DECODE='1', SGLANG_AGI2_QSA_IDX_GPC='16', SGLANG_AGI2_QSA_IDX_STAGES='1',
    SGLANG_AGI2_QSA_DIRECT_SPARSE='1',
    # the QSA block tier (sub 14), here at 7.5x for 48 slots (sub 18; 3.0x at 24 slots in subs 15-17): ~4.7M index
    # slots a GPU, 48 conversations of ~96k; 4-token groups unselected for a while go to host RAM
    # in rounds every 64 steps, keeping 4-9 % of the GPU rows free, at most 32,768 groups a round
    SGLANG_AGI2_QSA_BLOCK_TIER_X='6.0', SGLANG_AGI2_QSA_BLOCK_TIER_EVERY='64',
    SGLANG_AGI2_QSA_BLOCK_TIER_MAX_GROUPS='32768', SGLANG_AGI2_QSA_BLOCK_TIER_RESERVE='0.04',
    SGLANG_AGI2_QSA_BLOCK_TIER_LOW_WATER='0.06', SGLANG_AGI2_QSA_BLOCK_TIER_HYSTERESIS='0.03',
    # sub 17: promote-on-touch (host-resident groups a request selects move back to GPU rows in rounds; the audit
    # checks every round's books) with one residency class per QSA layer this GPU holds
    SGLANG_AGI2_QSA_BLOCK_TIER_POT='1', SGLANG_AGI2_QSA_BLOCK_TIER_POT_AUDIT='1',
    SGLANG_AGI2_QSA_BLOCK_TIER_PERLAYER='1',
    # sub 17: two host syncs dropped (QSA metadata, extend); fast_topk's overflow bucket refined exactly (upstream
    # clips a bucket of more than 4,096 candidates and drops true winners)
    SGLANG_AGI2_QSA_NOSYNC='1', SGLANG_AGI2_EXTEND_NOSYNC='1', SGLANG_AGI2_FAST_TOPK_EXACT='1',
    # sub 18: the stage cache (each host-resident group a step selects is copied once into a GPU scratch and kept
    # across steps: the same bytes, read on the GPU) behind a 2,048-token never-demote window
    SGLANG_AGI2_QSA_BLOCK_TIER_STAGE='1', SGLANG_AGI2_QSA_BLOCK_TIER_STAGE_MAX='32768',
    SGLANG_AGI2_QSA_BLOCK_TIER_STAGE_CACHE='1', SGLANG_AGI2_QSA_BLOCK_TIER_WINDOW='2048',
    # sub 18: the QSA index keys in INT8 + a per-key scale (the one lossy switch: they only rank which 2,048 tokens
    # each query attends; the attention reads the same K/V)
    SGLANG_AGI2_QSA_INDEX_INT8='1',
    # a prefill batch's summed context: QSA's cached-prefix extend copies every request's whole history, so
    # four long continuations in one pass could need ~1 GB transient; the first request always passes (GPT review 10-30)
    SGLANG_PREFILL_MAX_SUM_SEQ_LEN='262144',
    # prefill all-reduces through the CPU reducer in 2 MB pieces, int8 on the wire (no KL cost: 1030_1941)
    SGLANG_AGI2_SMR_SPLIT='1',
    # sub 20: three exact memory levers, 715 MiB a rank (docs/reports/1030_1550_parallel4/memaudit): FlashInfer's int
    # workspaces at 1 MiB (plans need < 3 KB), one shared fp32 logits buffer at the vocab-shard width for every
    # target-verify graph, and the eager input registry capped at the prefill budget
    SGLANG_AGI2_FI_INT_WORKSPACE_BYTES='1048576', SGLANG_AGI2_SHARD_LOGITS_BUFFER='1', SGLANG_AGI2_EAGER_CAP='1',
    # sub 20: the decode indexer reads each request's INT8 keys once for its 4 verify rows (bitwise equal, 1.85x at 48
    # requests: 1031_0108) and skips the -inf fill of logits past each row's length, which top-k never reads
    SGLANG_AGI2_QSA_IDX_SHARED_INT8='1', SGLANG_AGI2_QSA_NO_LOGITS_FILL='1',
    # S64b (x15 probe 10-31): the three later exact memory levers (lazy graph mask, EPLB map on host, draft
    # successor compaction), the CPU reducer's 4,032 KB envelope, and the session fork (attempts 2-4 restore
    # their task's parked first-turn prefix)
    SGLANG_AGI2_LAZY_GRAPH_MASK='1', SGLANG_AGI2_EPLB_MAP_HOST='1', SGLANG_AGI2_DRAFT_SUCC_COMPACT='1',
    SGLANG_SM_REDUCE_MAX_KB='4032', SGLANG_AGI2_SESSION_FORK='1',
    # the vision encoder in groups of at most 4,096 visual tokens (its activations grow with the call; a 16-slot
    # Modal round ran out of GPU memory there, 1025_1906)
    SGLANG_MM_ENCODE_MAX_TOKENS='2048',   # S64b: 2,048
    # log every Triton compile after serving starts (the block tier's move kernel compiles its first few tail
    # sizes mid-run, ~1-2 s each; seed review 10-26). Not part of the seed's fingerprint (its ENV_SKIP)
    SGLANG_TRITON_SLOW_COMPILE_THRESHOLD_SECS='0',
    # DFlash2 and the verify path
    SGLANG_LOGPROB_CHUNK_SIZE='128', SGLANG_DFLASH_DRAFT_WINDOW_KV='1', SGLANG_AGI2_DRAFT_WINDOW_KV_PAGED='1',
    SGLANG_ENABLE_DFLASH_SHARDED_VERIFY='1', SGLANG_AGI2_VERIFY_TAIL='1', SGLANG_AGI2_EARLY_PUBLISH='1',
    SGLANG_AGI2_DFLASH_PINNED_NTNP='1',   # (the lm_head stays BF16: no SGLANG_ENABLE_FP8_LM_HEAD)
    SGLANG_AGI2_SWA_ONLY_WRAPPER0='1',
    # sub 18: the verify accept on the sparse distributions it already has (the sampler's reserve 0.78 -> 0.31 GB at
    # 48 slots) and the draft's graph-index buffer sized for its 2,048-token window (0.40 GB), both exact
    SGLANG_AGI2_SPARSE_ACCEPT='1', SGLANG_AGI2_SPARSE_ACCEPT_CHUNK='8', SGLANG_AGI2_SWA_GRAPH_IDX_CAP='1',
    # sub 18: the decode-lever bundle (agi2/feat/qsa-idx-shared 17b4f71a70): M-RoPE async upload, Marlin's
    # cache13 left empty, the EP top-k sum over local slots, the EP id map as one gather, and the draft's
    # candidate head on a 32,768-token subset (written in section 5)
    SGLANG_AGI2_MROPE_ASYNC='1', SGLANG_AGI2_MOE_EMPTY_CACHE13='1', SGLANG_AGI2_MOE_EP_SUM='1',
    SGLANG_AGI2_EP_MAP_TRITON='1', SGLANG_AGI2_DRAFT_VOCAB='/tmp/draft_vocab_32k.json',
    # the int8 CPU all-reduce, on the last SMR_THREADS cores
    SGLANG_ENABLE_SM_REDUCE_ALLREDUCE='1', SGLANG_SM_REDUCE_FMT='int8', SGLANG_SM_REDUCE_DOWNQ='1',
    SGLANG_SM_REDUCE_THREADS=str(SMR_THREADS), SGLANG_SM_REDUCE_CHUNK_KB='64', SGLANG_SM_REDUCE_PUBWARP='1',
    SGLANG_SM_REDUCE_CPUS=','.join(map(str, CPUS[-SMR_THREADS:])),
)
if SESSIONS:   # the session fixes: retract, evict, park; QSA's session prefix and two-group pending ring
    FLAGS.update(SGLANG_ENABLE_STREAMING_SESSION_RETRACT='1', SGLANG_ENABLE_STREAMING_SESSION_EVICT='1',
                 SGLANG_ENABLE_STREAMING_SESSION_PARK='1', SGLANG_STREAMING_SESSION_PARK_HOST_GB='8',
                 SGLANG_AGI2_QSA_SESSION_PREFIX='1', SGLANG_AGI2_QSA_RING2='1')
# Exact turn boundaries, only where their patch is in place: without it a cut turn is rebuilt, as in submission 8
EXACT_TURNS = EXACT_TURNS and bool(glob.glob('/tmp/sgl/lib/python3*/site-packages/sglang/srt/speculative/dflash_length_cap.py'))
if EXACT_TURNS:
    FLAGS.update(SGLANG_ENABLE_DFLASH_LENGTH_CAP='1', SGLANG_ENABLE_DFLASH_STOP_CAP='1',
                 SGLANG_ENABLE_STREAMING_SESSION_FOLDED_CLAIM='1')
FLAGS['ARC_SLOTS'] = str(SLOTS)                                 # read by the launch cell: CUDA graphs up to this batch
FLAGS['ARC_MAX_REQS'] = str(SLOTS + 2 if SESSIONS else SLOTS)   # resting sessions hold their rows: slots + 2
FLAGS['ARC_SESSIONS'] = '1' if SESSIONS else ''                 # read by the launch cell: --enable-streaming-session
# sub 14's server shape, read by the launch cell (unset there, they are sub 13's 0.92 / graphs to 12 / 18 slots)
FLAGS['ARC_MEM_FRACTION'] = '0.899'                              # --mem-fraction-static (sub 17: 0.896 at 24 slots)
FLAGS['ARC_GRAPH_BS'] = '1 2 4 8 12 16 24 32 40 48 56 64'             # --cuda-graph-bs-decode: 10 sizes up to SLOTS (sub 17:
                                                                #   1-24)
FLAGS['ARC_MAMBA_SLOTS'] = str(SLOTS + 4)                       # --max-mamba-cache-size: 52 GDN state slots
FLAGS['ARC_SSM_DTYPE'] = 'bfloat16'                             # --mamba-ssm-dtype: the GDN state in bf16 (sub 16: fp32)
for k in [k for k in os.environ if k.startswith(('SGLANG_', 'PYTORCH_CUDA_ALLOC', 'ARC_SLOTS', 'ARC_MAX_REQS', 'ARC_SESSIONS', 'ARC_MEM_FRACTION', 'ARC_GRAPH_BS', 'ARC_MAMBA_SLOTS', 'ARC_SSM_DTYPE'))]:
    os.environ.pop(k)  # a second server in the same session must not inherit the previous one's switches
os.environ.update(FLAGS)
check(os.path.exists(KEEP_LIST), f'the keep list {KEEP_LIST} is missing (section 5)')
print('exact turns' if EXACT_TURNS else 'no exact turns', FLAGS)
print(f'{len(CPUS)} CPUs; this kernel and the server on {len(KEEP)} of them, the reducer on {CPUS[-SMR_THREADS:]}')


exact turns {'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True,pinned_max_round_threshold_mb:1024', 'SGLANG_ENABLE_GDN_REPLAYSSM_FOLD': '1', 'SGLANG_AGI2_QSA_PAIR_LAYER_SPLIT': '1', 'SGLANG_AGI2_FN_VISION_HOST_BLOCKS': '1', 'SGLANG_FLASHINFER_WORKSPACE_SIZE': '100663296', 'SGLANG_VLM_CACHE_SIZE_MB': '0', 'SGLANG_AGI2_FN_EXPERT_KEEP': '/tmp/keep_384_code_bal4.json', 'SGLANG_AGI2_FN_EXPERT_KEEP_ANY_ORDER': '1', 'SGLANG_AGI2_HOST_EMBED': '1', 'SGLANG_AGI2_MARLIN_NO_SWIZZLED': '1', 'SGLANG_AGI2_MARLIN_INPLACE_REPACK': '1', 'SGLANG_AGI2_PLE_DIRECT_READ': '1', 'SGLANG_AGI2_PREFETCH_EXCLUDE': 'plefp8', 'SGLANG_AGI2_QSA_TRITON_DECODE': '1', 'SGLANG_AGI2_QSA_IDX_GPC': '16', 'SGLANG_AGI2_QSA_IDX_STAGES': '1', 'SGLANG_AGI2_QSA_DIRECT_SPARSE': '1', 'SGLANG_AGI2_QSA_BLOCK_TIER_X': '6.0', 'SGLANG_AGI2_QSA_BLOCK_TIER_EVERY': '64', 'SGLANG_AGI2_QSA_BLOCK_TIER_MAX_GROUPS': '32768', 'SGLANG_AGI2_QSA_BLOCK_TIER_RESERVE': '0.04', 'SGLANG_AGI2_QSA_BLOCK_TIER_LOW_WATER': '0.06', 'SGLANG_AGI2_QSA_BLOCK_TI

In [15]:
%%writefile launch_server.sh
# The server's launch, as a file: the cell below runs it, and so does restart_server() after a crash.
#   bash launch_server.sh            start the server (the first launch in a container also prepares the cache)
#   bash launch_server.sh pack DIR   write the compile cache of this container as a seed into DIR, and stop
# (Submission 12's launch_server.sh plus the seeded compile cache, docs/reports/1024_1137_jit_cache_seeding/, and the
#  weight loader's prefetch and threads, docs/reports/1024_1318_weight_loading_phase1.md)
[ -f sglang.log ] && [ -z "${1:-}" ] && mv sglang.log "sglang_$(date +%m%d_%H%M%S).log"   # keep the previous server's log (a crash's)
MODEL=$(dirname "$(find -L /kaggle/input/models -path '*qwen3-8-flash-next-nvfp4*' -name config.json | head -1)")
DRAFT=$(dirname "$(find -L /kaggle/input/models -path '*qwen3-8-flash-next-dflash2*' -name config.json | head -1)")
# sub 18: the QG experts (GPTQ-re-rounded NVFP4, docs/reports/1029_1845_nvfp4_rerounding_scope) as an overlay folder:
# RadixArk's files, with the 192 expert shards taken from the QG Model; fail closed unless all 192 are in place
QG=$(dirname "$(find -L /kaggle/input/models -path '*agi2-qg1-8352943a271b*' -name MANIFEST.sha256 | head -1)")
if [ "$QG" != "." ]; then
    rm -rf /tmp/model_qg && mkdir -p /tmp/model_qg
    for f in "$MODEL"/*; do
        b=$(basename "$f")
        case "$b" in
            layer-*-experts-*.safetensors) ln -s "$QG/$b" "/tmp/model_qg/$b" ;;
            *) ln -s "$f" "/tmp/model_qg/$b" ;;
        esac
    done
    [ "$(ls -L /tmp/model_qg/layer-*-experts-*.safetensors | wc -l)" = 192 ] || { echo "QG overlay incomplete"; exit 1; }
    MODEL=/tmp/model_qg
else
    echo "QG experts not attached"; exit 1
fi
echo "model: $MODEL"
echo "draft: $DRAFT"
mkdir -p /tmp/cache/{triton,inductor,flashinfer,xdg,torch_extensions,sglang_jit,sglang,cute_dsl}
# Kernels compiled at start-up must use the wheelhouse's CUDA 13 toolkit, not the image's CUDA 12.8. That toolkit
# pairs nvcc 13.4 with CUDA 13.0 runtime headers (as frozen in the lmsysorg image), so CCCL's exact-version check is off.
export CUDA_HOME=/tmp/sgl/lib/python3.12/site-packages/nvidia/cu13
# /tmp/sgl/bin comes last: the venv's ninja builds the JIT kernels if the image has no ninja of its own.
export PATH=$CUDA_HOME/bin:$PATH:/tmp/sgl/bin NVCC_APPEND_FLAGS="-DCCCL_DISABLE_CTK_COMPATIBILITY_CHECK"
export LD_LIBRARY_PATH=$LD_LIBRARY_PATH:$CUDA_HOME/lib
ln -sfn lib $CUDA_HOME/lib64 && ln -sf libcudart.so.13 $CUDA_HOME/lib/libcudart.so  # the names the JIT linker asks for
env | grep -E '^(SGLANG_|PYTORCH_CUDA_ALLOC|ARC_(SLOTS|MAX_REQS|SESSIONS|MEM_FRACTION|GRAPH_BS|MAMBA_SLOTS|SSM_DTYPE))' | sort  # the switches set above
# ARC_MEM_FRACTION, ARC_GRAPH_BS, ARC_MAMBA_SLOTS (sub 14) and ARC_SSM_DTYPE (sub 17) come from the switches cell;
# unset, they are sub 13's values (0.92; decode graphs 1-8, 10, 12; 18 GDN state slots; fp32 GDN state), so earlier
# builds launch exactly as before. The GDN state dtype must be this flag: with --enable-linear-replayssm-spec the
# server overwrites the env var SGLANG_MAMBA_SSM_DTYPE with the flag's value (serving_hook.py:398-399).
export HF_HUB_OFFLINE=1 TRANSFORMERS_OFFLINE=1 TOKENIZERS_PARALLELISM=false PYTHONUNBUFFERED=1
# Every compile cache in one real directory under /tmp (a cache path under a symlink never hits: seen on Modal).
# SGLANG_CACHE_DIR too: at import SGLang points TileLang's cache and CUDA's own PTX cache under it (environ.py
# third_party_cache_defaults); left unset they land in ~/.cache/sglang, outside the seeded tree.
export TRITON_CACHE_DIR=/tmp/cache/triton TORCHINDUCTOR_CACHE_DIR=/tmp/cache/inductor SGLANG_JIT_CACHE_DIR=/tmp/cache/sglang_jit
export FLASHINFER_WORKSPACE_BASE=/tmp/cache/flashinfer XDG_CACHE_HOME=/tmp/cache/xdg TORCH_EXTENSIONS_DIR=/tmp/cache/torch_extensions
export SGLANG_CACHE_DIR=/tmp/cache/sglang
# CuTe DSL (FlashInfer's rmsnorm kernels on sm_89) keeps its own file cache, by default /tmp/<user>/cutlass_python_cache;
# its key holds the working directory, so the server always starts from this one.
export CUTE_DSL_CACHE_DIR=/tmp/cache/cute_dsl
# The last line's two switches are the weight loader's (checkpoint prefetch, 2 loader threads): they change how the
# weights are read, not what is compiled, so the seed check leaves them out of the command line it compares.
ARGS=(--model-path "$MODEL" --speculative-draft-model-path "$DRAFT" --served-model-name qwen3.8-flash-next
      --host 127.0.0.1 --port 30000 --tp 4 --reasoning-parser qwen3 --context-length 262144 --mem-fraction-static ${ARC_MEM_FRACTION:-0.92}
      --max-running-requests $ARC_MAX_REQS --chunked-prefill-size 2048 --mamba-ssm-dtype ${ARC_SSM_DTYPE:-float32} --kv-cache-dtype fp8_e4m3
      --enable-metrics --decode-log-interval 200 --trust-remote-code --disable-radix-cache --ep-size 4
      --quantization modelopt_fp4 --fp4-gemm-backend marlin --moe-runner-backend marlin --page-size 64 --ple-offload-embedding
      --cuda-graph-max-bs-decode $ARC_SLOTS --cuda-graph-bs-decode ${ARC_GRAPH_BS:-1 2 3 4 5 6 7 8 10 12} --max-mamba-cache-size ${ARC_MAMBA_SLOTS:-18}
      --min-free-slots-delay 1 --enable-linear-replayssm-spec --disable-prefill-cuda-graph --linear-attn-verify-backend triton
      --tool-call-parser qwen3_coder --image-processor-backend pil ${ARC_SESSIONS:+--enable-streaming-session}
      --speculative-algorithm DFLASH --speculative-num-draft-tokens 4 --speculative-draft-model-quantization unquant --enable-prefill-delayer --prefill-max-requests 4 --prefill-delayer-queue-min-ratio 0.07 --prefill-delayer-max-delay-passes 30 --prefill-delayer-max-delay-ms 3600000
      --weight-loader-prefetch-checkpoints --model-loader-extra-config '{"num_threads": 2}')
printf '%s\n' "${ARGS[@]}" > server_args.txt   # the seed check compares the seed's command line with this one
CHECK=${JIT_SEED_CHECK:-$PWD/jit_seed_check.py}   # written by its own cell, beside this file
if [ "${1:-}" = pack ]; then   # the seed builder: this container's caches, after a warm boot and a real boot
    exec /tmp/sgl/bin/python "$CHECK" pack --out "$2" --args-file server_args.txt --fingerprint /tmp/jit_seed_fingerprint.json
fi
if [ "${1:-}" = check ]; then  # the seed check alone, on a seed already unpacked into /tmp/cache
    exec /tmp/sgl/bin/python "$CHECK" verify --seed "$2" --args-file server_args.txt
fi
stop_all() {   # the launcher, its TP workers and detokenizer, and any Python of ours still holding a GPU; bounded waits
    pkill -f "[s]glang.launch_server"; sleep 5; pkill -9 -f "[s]glang.launch_server"; pkill -9 -f "[s]glang::"
    # Only PIDs that are ours: in a gVisor sandbox nvidia-smi reports PID 1, the container's init (killing it ended a
    # Modal box twice, 1024_1311 / 1024_1333); on Kaggle it reports the real PIDs, so nothing changes there.
    for pid in $(timeout 20 nvidia-smi --query-compute-apps=pid --format=csv,noheader 2>/dev/null); do
        [ "$pid" -gt 1 ] 2>/dev/null && grep -qa python "/proc/$pid/cmdline" 2>/dev/null && kill -9 "$pid" 2>/dev/null
    done
    for i in $(seq 60); do [ -z "$(timeout 20 nvidia-smi --query-compute-apps=pid --format=csv,noheader 2>/dev/null)" ] && break; sleep 2; done
}
# The warm boot: compiles every JIT kernel with the plain NCCL all-reduce (the CPU reducer off), sends the warm-up
# requests, and stops; the real server then starts on a warm cache. It runs only when no seed covers this build.
warm_boot() {
    T0=$(date +%s)
    echo "warm boot: NCCL all-reduce, compiling the JIT kernels into /tmp/cache ($(date +%H:%M:%S))"
    env -u SGLANG_ENABLE_SM_REDUCE_ALLREDUCE -u SGLANG_CRASH_ON_JIT_COMPILE nohup setsid /tmp/sgl/bin/python -m sglang.launch_server "${ARGS[@]}" \
        > sglang_warm.log 2>&1 < /dev/null &
    WPID=$!
    for i in $(seq 480); do
        curl -sf -m 5 http://127.0.0.1:30000/health > /dev/null && break
        kill -0 $WPID 2> /dev/null || { echo "warm boot exited: $(tail -3 sglang_warm.log)"; break; }
        [ $((i % 12)) -eq 0 ] && echo "  warm boot +$(( $(date +%s) - T0 )) s: $(grep -c . sglang_warm.log) log lines"
        sleep 5
    done
    /tmp/sgl/bin/python warm_requests.py || echo "warm requests failed; the real server starts anyway"
    stop_all
    touch /tmp/cache/warm_done
    echo "warm boot done in $(( $(date +%s) - T0 )) s; cache $(du -sh /tmp/cache | cut -f1)"
}
start_server() {
    nohup setsid /tmp/sgl/bin/python -m sglang.launch_server "${ARGS[@]}" > sglang.log 2>&1 < /dev/null &
    SPID=$!
    echo "server starting (pid $SPID)"
}
# The seed: the compile cache of an earlier run of this exact build (an attached input). Used only when the check
# finds every seeded file intact and everything the cache keys depend on unchanged; otherwise the cache is emptied
# and the warm boot runs as before. A seeded boot that still dies before /health also falls back to the warm boot.
SEEDED=0
if [ ! -f /tmp/cache/warm_done ]; then
    SEED=${JIT_SEED_DIR:-$(dirname "$(find -L /kaggle/input -maxdepth 5 -name jit_seed_manifest.json 2>/dev/null | head -1)")}
    if [ -f "$SEED/jit_seed.tar" ] && [ -f "$CHECK" ]; then
        TS=$(date +%s)
        tar -C / -xf "$SEED/jit_seed.tar"   # the builder's mtimes kept: ninja reads them (the next line)
        # sm_reduce's extension is rebuilt by ninja whenever an input looks newer than its objects, and this install's
        # sources and headers all do: date them back (ninja never reads contents; the check below compares those).
        /tmp/sgl/bin/python "$CHECK" torchext | tee seed_torchext.json
        if /tmp/sgl/bin/python "$CHECK" verify --seed "$SEED" --args-file server_args.txt | tee seed_check.json | grep -q '"ok": true'; then
            SEEDED=1
            touch /tmp/cache/warm_done
            echo "seeded compile cache from $SEED: $(du -sh /tmp/cache | cut -f1) in $(( $(date +%s) - TS )) s; one boot"
        else
            echo "seed does not cover this build ($(cut -c1-400 seed_check.json)); emptying the cache for the warm boot"
            rm -rf /tmp/cache/* && mkdir -p /tmp/cache/{triton,inductor,flashinfer,xdg,torch_extensions,sglang_jit,sglang,cute_dsl}
        fi
    else
        echo "no compile-cache seed attached"
    fi
    if [ $SEEDED = 0 ]; then
        # What the caches' keys depend on, taken before any boot: lets this container's caches be packed as a seed
        # later (`launch_server.sh pack DIR`), checked in the same state a reusing container is checked in.
        [ -f "$CHECK" ] && /tmp/sgl/bin/python "$CHECK" fingerprint --out /tmp/jit_seed_fingerprint.json --args-file server_args.txt
        warm_boot
    fi
fi
touch /tmp/jit_seed_boot_marker   # a cache hit writes no file: anything newer than this under /tmp/cache was compiled
start_server
if [ $SEEDED = 1 ]; then   # wait here, so a boot that dies can still fall back
    TB=$(date +%s)
    for i in $(seq 720); do
        curl -sf -m 5 http://127.0.0.1:30000/health > /dev/null && break
        kill -0 $SPID 2> /dev/null || break
        [ $((i % 12)) -eq 0 ] && echo "  seeded boot +$(( $(date +%s) - TB )) s: $(grep -c . sglang.log) log lines"
        sleep 5
    done
    if ! curl -sf -m 5 http://127.0.0.1:30000/health > /dev/null; then
        echo "seeded boot did not come up ($(tail -2 sglang.log | cut -c1-300)); falling back to the warm boot"
        stop_all
        mv sglang.log "sglang_seeded_failed_$(date +%m%d_%H%M%S).log"
        rm -rf /tmp/cache/* && mkdir -p /tmp/cache/{triton,inductor,flashinfer,xdg,torch_extensions,sglang_jit,sglang,cute_dsl}
        /tmp/sgl/bin/python "$CHECK" fingerprint --out /tmp/jit_seed_fingerprint.json --args-file server_args.txt
        unset SGLANG_CRASH_ON_JIT_COMPILE   # (a test instrument only) the fallback compiles what it needs
        warm_boot
        start_server
    else
        echo "seeded boot up in $(( $(date +%s) - TB )) s; new files in the compile cache (misses):" \
             "$(find /tmp/cache -type f -newer /tmp/jit_seed_boot_marker -not -path '*/tilelang/tmp/*' -not -path '*/.staging*' | wc -l);" \
             "target-verify capture $(grep -o 'Capture target verify CUDA graph end. elapsed=[0-9.]*' sglang.log | head -1 | cut -d= -f2) s"
    fi
fi


Writing launch_server.sh


In [16]:
%%writefile warm_requests.py
# The warm boot's requests: a two-chunk prefill with a decode, and a picture, so the prefill, decode and vision
# kernels are compiled before the real server starts. Standard library and Pillow only.
import base64, io, json, time, urllib.request
from PIL import Image

def post(path, body, timeout=900):
    req = urllib.request.Request('http://127.0.0.1:30000' + path, data=json.dumps(body).encode(),
                                 headers={'Content-Type': 'application/json'})
    return json.loads(urllib.request.urlopen(req, timeout=timeout).read())

grid = ' '.join(str((i * 7) % 10) for i in range(900))   # ~1,800 tokens: two 1,024-token prefill chunks
t = time.time()
r = post('/v1/chat/completions', {'model': 'qwen3.8-flash-next', 'max_tokens': 64, 'temperature': 1.0,
                                  'messages': [{'role': 'user', 'content': 'Describe this grid briefly: ' + grid}]})
print(f'warm text request: {r["usage"]["completion_tokens"]} tokens in {time.time() - t:.0f} s', flush=True)
img = Image.new('RGB', (240, 240), (0, 0, 0))
for x in range(0, 240, 24):
    for y in range(0, 240, 24):
        img.paste(((x * 3) % 255, (y * 5) % 255, 128), (x, y, x + 20, y + 20))
buf = io.BytesIO()
img.save(buf, format='PNG')
url = 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()
t = time.time()
r = post('/v1/chat/completions', {'model': 'qwen3.8-flash-next', 'max_tokens': 32, 'temperature': 1.0,
                                  'messages': [{'role': 'user', 'content': [
                                      {'type': 'image_url', 'image_url': {'url': url}},
                                      {'type': 'text', 'text': 'How many squares are there?'}]}]})
print(f'warm picture request: {r["usage"]["completion_tokens"]} tokens in {time.time() - t:.0f} s', flush=True)


Writing warm_requests.py


**The seed check**, as a file the launch runs: `fingerprint` before a cold container's first boot, `pack` to write a
seed, `verify` to check one before it is used, and `torchext`, which dates the CPU all-reduce extension's sources back
so its build tool sees the seeded build as current (it compares file times, and this container's install is newer).
Standard library plus the fork's own cache code.


In [17]:
%%writefile jit_seed_check.py
"""The seeded compile cache: pack it after a warm boot, and check it before a boot that relies on it.

JIT-cache seeding (docs/reports/1024_1137_jit_cache_seeding/): the Kaggle notebook boots its server once with every
kernel already compiled, from a cache built by an earlier run of the same build. A cache that does not cover this
exact build would compile at boot instead, and a long compile on one GPU while the others spin in sm_reduce's
all-reduce trips its 50M-poll trap (sm_reduce_kernel.cu:406-408, 497-502). So the boot trusts the seed only when this
check passes, and otherwise falls back to today's warm boot on an EMPTY cache.

    python jit_seed_check.py fingerprint --out FILE --args-file server_args.txt   (first launch, before any boot)
    python jit_seed_check.py pack   --out DIR --args-file server_args.txt [--fingerprint FILE] [--dir PATH ...]
    python jit_seed_check.py verify --seed DIR --args-file server_args.txt [--root /]

Run it with the server's own interpreter (/tmp/sgl/bin/python) and environment (inside launch_server.sh, after the
exports): it imports the fork's own JIT-cache code for two checks, so they are exactly the server's.

`pack` writes DIR/jit_seed.tar (paths relative to /) and DIR/jit_seed_manifest.json:
- every file with its size and sha256 (lock files and in-progress build directories left out);
- the fingerprint of everything the caches' keys depend on (below);
- a summary per cache.

`verify` (after `tar -C / -xmf DIR/jit_seed.tar`) exits 0 when the seed covers this build, else 3, and prints one
JSON line. It fails on any HARD field that differs, any seeded file missing or changed, or any SGLang JIT build whose
recorded dependencies (sources, CUDA and system headers) no longer hash to its own name. SOFT fields (driver, CPU
model, host) are reported only: no cache we seed is keyed on them, except the CUDA driver's own PTX cache, which a
different driver merely refills.

HARD fingerprint fields:
- gpu: the GPUs' names and compute capabilities (Triton, inductor, TileLang, SGLang JIT all key on the arch);
- cxx: `c++ --version` (the SGLang JIT key, sglang/kernels/jit/utils/compile/cache.py:241-245; host code of every
  nvcc build, and Triton's launcher stubs);
- nvcc: `$CUDA_HOME/bin/nvcc --version`;
- python: the interpreter's version;
- venv: every installed distribution's RECORD (file list with sha256), so any package difference shows;
- sglang_tree: every file of the installed, patched sglang package (the diff, the JIT sources, sm_reduce's sources);
- env: the server switches and cache paths the launch exports, every cache path by its effective value (CPU pinning
  lists and the allocator's loader-only `pinned_max_round_threshold_mb` left out);
- server_args: the server's command line, without the loader-only switches (LOADER_ONLY_FLAGS);
- sgl_jit_env: the fork's own `_environment_fingerprint()` (target, both compilers, package versions);
- cpu_isa: sm_reduce's ISA tag, part of its extension's name (a different ISA builds that extension anew: 33 s at
  rank 0 before any all-reduce runs, so it is SOFT for safety but reported).
"""

from __future__ import annotations

import argparse
import glob
import hashlib
import json
import os
import platform
import re
import subprocess
import sys
import tarfile
import time
from pathlib import Path

SEED_TAR = "jit_seed.tar"
MANIFEST = "jit_seed_manifest.json"
# 2 (2026-10-24): the cache paths SGLang's import sets are hashed by their effective value, and the loader-only
# switches below are left out. A format-1 seed is refused (a fallback, never a wrong hit).
FORMAT = 2
DEFAULT_DIRS = ["/tmp/cache"]
# Server switches that steer only how weights are read or host memory is pinned: no compiled artifact depends on them
# (cache-key study docs/reports/1024_1137_jit_cache_seeding/), so tuning them must not invalidate the seed.
LOADER_ONLY_FLAGS = {"--weight-loader-prefetch-checkpoints": 0, "--model-loader-extra-config": 1}   # flag -> values
LOADER_ONLY_ALLOC = ("pinned_max_round_threshold_mb",)   # parts of PYTORCH_CUDA_ALLOC_CONF
# What `import sglang` sets with os.environ.setdefault (fork environ.py third_party_cache_defaults): hashed by the value
# in force either way, so a caller that imported sglang (and so set them) fingerprints the same as one that did not.
SGLANG_CACHE_DEFAULTS = {"TRITON_CACHE_DIR": "triton", "TORCHINDUCTOR_CACHE_DIR": "inductor", "CUDA_CACHE_PATH": "nv",
                         "TILELANG_CACHE_DIR": "tilelang", "FLASHINFER_WORKSPACE_BASE": ""}
# Never seeded: the warm-boot marker (the boot decides that), torch's build baton (a stale `lock` makes
# cpp_extension.load wait for a builder that does not exist), SGLang JIT staging dirs (builds in progress).
SKIP_NAMES = {"warm_done", "lock"}
SKIP_PREFIXES = (".staging-",)
ENV_KEYS_EXTRA = ("CUDA_HOME", "NVCC_APPEND_FLAGS", "TRITON_CACHE_DIR", "TORCHINDUCTOR_CACHE_DIR",
                  "SGLANG_JIT_CACHE_DIR", "FLASHINFER_WORKSPACE_BASE", "XDG_CACHE_HOME", "TORCH_EXTENSIONS_DIR",
                  "SGLANG_CACHE_DIR", "TILELANG_CACHE_DIR", "CUDA_CACHE_PATH", "CUTE_DSL_CACHE_DIR", "PYTORCH_CUDA_ALLOC_CONF",
                  "ARC_SLOTS", "ARC_MAX_REQS", "ARC_SESSIONS", "HOME")
# Families whose variables enter cache keys (Triton's invalidating set, inductor's config, FLA's autotune cache,
# CuTe DSL, FlashInfer, TileLang) or the rank -> GPU mapping inductor keys carry (CUDA_VISIBLE_DEVICES).
ENV_PREFIXES = ("SGLANG_", "TRITON_", "TORCHINDUCTOR_", "TORCH_COMPILE", "TORCHDYNAMO", "FLA_", "PTXAS", "MLIR_",
                "LLVM_", "NVPTX_", "CUTE_DSL", "FLASHINFER_", "TILELANG_", "CUDA_VISIBLE_DEVICES")
# Switches that change no kernel: CPU pinning (differs per machine), the check's own diagnostics, and the PLE direct
# read's loader switches (fork agi2/feat/ple-direct: host-side preadv into the pinned tables, which hold the same bytes,
# and which files the prefetch skips; sub 13, 2026-10-25), so the PLE fix can be switched without a new seed.
ENV_SKIP = {"SGLANG_SM_REDUCE_CPUS", "SGLANG_CRASH_ON_JIT_COMPILE", "SGLANG_JIT_CACHE_DEBUG",
            "SGLANG_TRITON_SLOW_COMPILE_THRESHOLD_SECS",
            "SGLANG_AGI2_PLE_DIRECT_READ", "SGLANG_AGI2_PLE_DIRECT_THREADS", "SGLANG_AGI2_PREFETCH_EXCLUDE",
            # per-token confidence (agi2/feat/fn-conf): with it on, boot and graph capture are unchanged; its two Triton
            # kernels compile only at the first request that asks (the logprobs session, 2026-10-25). It is safe with
            # SGLANG_ENABLE_GRAPH_POOL_BORROW: no borrowed scope reaches its q stash (the earlier hazard was a misread;
            # the logprobs session, 10-25 ~10:00 IST). The borrow switch stays in the fingerprint for its own sake.
            "SGLANG_AGI2_FN_CONF", "SGLANG_AGI2_FN_CONF_CHECK"}
HARD = ("gpu", "cxx", "nvcc", "python", "venv", "sglang_tree", "env", "server_args", "sgl_jit_env")
SOFT = ("driver", "cpu_isa", "cpu_model", "glibc", "host")


def _sha(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _file_sha(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def _run(cmd: list[str]) -> str:
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=60).stdout.strip()
    except (OSError, subprocess.SubprocessError) as e:
        return f"ERROR {type(e).__name__}: {e}"


def _site() -> Path:
    import sysconfig

    return Path(sysconfig.get_paths()["purelib"])


def _venv_hash() -> str:
    lines = []
    for rec in sorted(glob.glob(str(_site() / "*.dist-info" / "RECORD"))):
        for line in open(rec, encoding="utf-8", errors="replace"):
            path = line.split(",", 1)[0]
            if "__pycache__" in path or path.endswith(("INSTALLER", "REQUESTED", "direct_url.json", "uv_cache.json")):
                continue
            lines.append(line.rstrip("\n"))
    return _sha("\n".join(lines).encode())


def _tree_hash(root: Path) -> tuple[str, int]:
    h, n = hashlib.sha256(), 0
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = sorted(d for d in dirnames if d != "__pycache__")
        for name in sorted(filenames):
            if name.endswith(".pyc"):
                continue
            p = os.path.join(dirpath, name)
            h.update(os.path.relpath(p, root).encode() + b"\0" + _file_sha(p).encode() + b"\n")
            n += 1
    return h.hexdigest(), n


def _cpu_isa() -> str:
    """sm_reduce's own rule (sm_reduce_allreduce.py `_cpu_isa`): the tag in its extension's name."""
    try:
        flags = set(Path("/proc/cpuinfo").read_text().split("flags", 2)[1].split("\n", 1)[0].split())
    except (OSError, IndexError):
        flags = set()
    if {"avx512f", "avx512bw", "avx512vl"} <= flags:
        return "avx512"
    if {"avx2", "fma"} <= flags:
        return "avx2"
    return "generic"


def _cpu_model() -> str:
    for line in Path("/proc/cpuinfo").read_text().splitlines():
        if line.startswith("model name"):
            return line.split(":", 1)[1].strip()
    return "?"


def _effective_env() -> dict[str, str]:
    env = {k: v for k, v in os.environ.items()
           if (k.startswith(ENV_PREFIXES) or k in ENV_KEYS_EXTRA) and k not in ENV_SKIP}
    base = os.path.expanduser(os.environ.get("SGLANG_CACHE_DIR", "~/.cache/sglang"))
    for k, sub in SGLANG_CACHE_DEFAULTS.items():
        env.setdefault(k, os.path.join(base, sub) if sub else base)
    if "PYTORCH_CUDA_ALLOC_CONF" in env:
        parts = [p for p in env["PYTORCH_CUDA_ALLOC_CONF"].split(",") if p.split(":")[0] not in LOADER_ONLY_ALLOC]
        env["PYTORCH_CUDA_ALLOC_CONF"] = ",".join(parts)
    return env


def _env_hash() -> tuple[str, list[str]]:
    env = _effective_env()
    items = [f"{k}={env[k]}" for k in sorted(env)]
    return _sha("\n".join(items).encode()), items


def _args_hash(args_file: str) -> str:
    """The server's command line (one argument a line), loader-only switches and their values left out."""
    args, out, skip = open(args_file, encoding="utf-8").read().splitlines(), [], 0
    for a in args:
        if skip:
            skip -= 1
        elif a in LOADER_ONLY_FLAGS:
            skip = LOADER_ONLY_FLAGS[a]
        else:
            out.append(a)
    return _sha("\n".join(out).encode())


def _sgl_jit_env() -> str:
    from sglang.kernels.jit.utils.compile import cache  # the fork's own key code

    return cache._environment_fingerprint()


def fingerprint(args_file: str) -> tuple[dict, dict]:
    """(fields, detail): the fields compared, and the readable detail behind the hashed ones."""
    t = time.time()
    smi = _run(["nvidia-smi", "--query-gpu=name,compute_cap,driver_version", "--format=csv,noheader"]).splitlines()
    cuda_home = os.environ.get("CUDA_HOME", "")
    env_h, env_items = _env_hash()
    tree_h, tree_n = _tree_hash(_site() / "sglang")
    fields = {
        "gpu": sorted({",".join(x.strip() for x in row.split(",")[:2]) for row in smi}) + [f"count={len(smi)}"],
        "cxx": _run(["c++", "--version"]),
        "nvcc": _run([os.path.join(cuda_home, "bin", "nvcc"), "--version"]).splitlines()[-2:],
        "python": sys.version,
        "venv": _venv_hash(),
        "sglang_tree": tree_h,
        "env": env_h,
        "server_args": _args_hash(args_file),
        "sgl_jit_env": _sgl_jit_env(),
        "driver": sorted({row.split(",")[2].strip() for row in smi if row.count(",") >= 2}),
        "cpu_isa": _cpu_isa(),
        "cpu_model": _cpu_model(),
        "glibc": _run(["ldd", "--version"]).splitlines()[:1],
        "host": platform.node(),
    }
    detail = {"env": env_items, "sglang_files": tree_n, "fingerprint_s": round(time.time() - t, 2)}
    return fields, detail


def _seed_files(dirs: list[str]) -> list[str]:
    out = []
    for d in dirs:
        for dirpath, dirnames, filenames in os.walk(d):
            dirnames[:] = sorted(x for x in dirnames if not x.startswith(SKIP_PREFIXES))
            for name in sorted(filenames):
                p = os.path.join(dirpath, name)
                if name in SKIP_NAMES or os.path.islink(p) or not os.path.isfile(p):
                    continue
                out.append(p)
    return out


def _jit_leaves(root: str) -> list[Path]:
    jit = Path(root + os.environ.get("SGLANG_JIT_CACHE_DIR", "/tmp/cache/sglang_jit"))
    return sorted(p.parent for p in jit.glob("*/*/build-*/deps-*/sgl_deps.json"))


def _check_leaf(leaf: Path) -> str | None:
    """None if the leaf is a hit as the server would see it (cache.find_prebuilt's test), else why not."""
    from sglang.kernels.jit.utils.compile import cache

    entries = cache._read_deps(leaf)
    if entries is None:
        return "unreadable sgl_deps.json"
    current, changed = cache._refresh(entries)
    if current is None:
        return f"dependency gone: {changed[0]}"
    if f"deps-{cache._deps_key(current)}" != leaf.name:
        return f"dependencies changed: {', '.join(changed[:3])}"
    module = leaf.parent.parent.name
    if not (leaf / f"{module}.so").is_file():
        return f"{module}.so missing"
    return None


def _summary(files: list[str]) -> dict:
    def size(prefix):
        return sum(os.path.getsize(f) for f in files if f.startswith(prefix))

    jit = os.environ.get("SGLANG_JIT_CACHE_DIR", "/tmp/cache/sglang_jit")
    return {
        "files": len(files), "bytes": sum(os.path.getsize(f) for f in files),
        "sglang_jit_modules": sorted({Path(f).relative_to(jit).parts[1] for f in files
                                      if f.startswith(jit + "/") and f.endswith(".so")}),
        "sglang_jit_leaves": len([f for f in files if f.startswith(jit + "/") and f.endswith("sgl_deps.json")]),
        "torch_extensions": sorted({Path(f).relative_to(os.environ.get("TORCH_EXTENSIONS_DIR", "/tmp/cache/torch_extensions")).parts[0]
                                    for f in files if f.startswith(os.environ.get("TORCH_EXTENSIONS_DIR", "/tmp/cache/torch_extensions") + "/")}),
        "bytes_by_dir": {d: size(d) for d in sorted({"/".join(f.split("/")[:4]) for f in files})},
    }


def _ninja_inputs(build_dir: str) -> tuple[list[str], list[str]]:
    """(inputs, outputs) of a torch extension's ninja build: explicit inputs from build.ninja plus every header its
    deps log lists (`ninja -t deps`), and the outputs (objects, the library)."""
    ins, outs = set(), set()
    text = open(os.path.join(build_dir, "build.ninja"), encoding="utf-8").read().replace("$\n", " ")
    for line in text.splitlines():
        if line.startswith("build "):
            # the outputs end at the first unescaped ':' ("$:" is ninja's escaped colon)
            lhs, rhs = re.split(r"(?<!\$):", line[len("build "):], maxsplit=1)
            unesc = lambda t: t.replace("$:", ":").replace("$$", "$")  # noqa: E731
            outs.update(os.path.join(build_dir, unesc(o)) for o in lhs.split())
            for tok in rhs.split()[1:]:   # after the rule name; skip the implicit / order-only markers
                if tok not in ("|", "||") and os.path.isabs(unesc(tok)):
                    ins.add(unesc(tok))
    deps = subprocess.run(["ninja", "-C", build_dir, "-t", "deps"], capture_output=True, text=True).stdout
    for ln in deps.splitlines():   # a target's line, then its dependencies indented (relative ones to the build dir)
        p = ln.strip()
        if ln.startswith((" ", "\t")) and p:
            ins.add(p if os.path.isabs(p) else os.path.normpath(os.path.join(build_dir, p)))
    return sorted(ins), sorted(o for o in outs if os.path.exists(o))


def torchext(a) -> int:
    """Make a seeded torch extension build directory current for ninja, then ask ninja (dry run) whether it would build.

    torch's cpp_extension.load runs ninja on every load, and ninja rebuilds when any input (the sources, every header in
    its deps log) is newer than the objects: a fresh install re-copies all of them with new mtimes. Extracting with
    `tar -m` does not help (the deps log then reads "stored deps info out of date"). So the seed keeps the builder's
    mtimes, and here every input newer than an hour before the oldest object is dated back to that time (ninja never
    hashes content; what the inputs contain is the fingerprint's job: venv, sglang_tree, cxx, nvcc). A stale `lock`
    would make load() wait forever, so it is removed."""
    root = os.environ.get("TORCH_EXTENSIONS_DIR", "/tmp/cache/torch_extensions")
    out = []
    for d in sorted(glob.glob(os.path.join(root, "*", "build.ninja"))):
        bd = os.path.dirname(d)
        lock = os.path.join(bd, "lock")
        if os.path.exists(lock):
            os.remove(lock)
        ins, outs = _ninja_inputs(bd)
        if not outs:
            out.append({"dir": bd, "ninja": "no outputs"})
            continue
        t_ref = min(os.path.getmtime(o) for o in outs) - 3600
        moved = 0
        for p in ins:
            if os.path.exists(p) and os.path.getmtime(p) > t_ref:
                os.utime(p, (t_ref, t_ref))
                moved += 1
        dry = subprocess.run(["ninja", "-C", bd, "-n", "-d", "explain"], capture_output=True, text=True)
        said = (dry.stdout + dry.stderr).strip().splitlines()
        current = any("no work to do" in ln for ln in said)
        out.append({"dir": os.path.basename(bd), "inputs": len(ins), "dated_back": moved, "current": current,
                    "ninja": said[-3:] if not current else "no work to do"})
    ok = all(r.get("current") for r in out)
    print(json.dumps({"torch_extensions": out, "ok": ok}), flush=True)
    return 0 if ok else 3


def save_fingerprint(a) -> int:
    """Taken by the first launch in a container before any boot, so a later `pack` records the state a reusing
    container is checked in (fresh install and patch, nothing run yet)."""
    fields, detail = fingerprint(a.args_file)
    with open(a.out, "w", encoding="utf-8") as f:
        json.dump({"fingerprint": fields, "detail": detail}, f, indent=1)
    print(json.dumps({"fingerprint": a.out, "s": detail["fingerprint_s"]}), flush=True)
    return 0


def pack(a) -> int:
    t0 = time.time()
    dirs = a.dir or DEFAULT_DIRS
    files = _seed_files(dirs)
    os.makedirs(a.out, exist_ok=True)
    if a.fingerprint and os.path.exists(a.fingerprint):
        saved = json.load(open(a.fingerprint, encoding="utf-8"))
        fields, detail = saved["fingerprint"], saved["detail"]
        now, _ = fingerprint(a.args_file)
        drift = [k for k in HARD if now.get(k) != fields.get(k)]
        if drift:   # the boots changed what the keys depend on: a reusing container would never match
            print(json.dumps({"pack": "refused", "changed_since_first_launch": drift}), flush=True)
            return 3
    else:
        fields, detail = fingerprint(a.args_file)
    leaves = _jit_leaves("")
    bad = {str(leaf): why for leaf in leaves if (why := _check_leaf(leaf))}
    if bad:
        print(json.dumps({"pack": "refused", "bad_leaves": bad}), flush=True)
        return 3
    entries = [{"path": f, "size": os.path.getsize(f), "sha256": _file_sha(f)} for f in files]
    # The torch extension as ninja sees it on the builder: what its objects depend on, and that it is current here.
    ext = []
    for d in sorted(glob.glob(os.path.join(os.environ.get("TORCH_EXTENSIONS_DIR", "/tmp/cache/torch_extensions"),
                                           "*", "build.ninja"))):
        bd = os.path.dirname(d)
        ins, outs = _ninja_inputs(bd)
        dry = subprocess.run(["ninja", "-C", bd, "-n"], capture_output=True, text=True)
        ext.append({"dir": os.path.basename(bd), "inputs": len(ins), "outputs": [os.path.basename(o) for o in outs],
                    "current_on_builder": "no work to do" in dry.stdout + dry.stderr})
    # GNU format: whole-second mtimes, rounded down. Ninja needs each restored object no newer than its deps-log record,
    # which a sub-second mtime rounded up (PAX floats) could break.
    with tarfile.open(os.path.join(a.out, SEED_TAR), "w", format=tarfile.GNU_FORMAT) as tar:
        for f in files:
            tar.add(f, arcname=f.lstrip("/"), recursive=False)
    manifest = {"format": FORMAT, "built": time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
                "dirs": dirs, "hard": HARD, "soft": SOFT, "fingerprint": fields, "detail": detail,
                "summary": {**_summary(files), "torch_extensions_ninja": ext}, "files": entries}
    with open(os.path.join(a.out, MANIFEST), "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=1)
    s = manifest["summary"]
    print(json.dumps({"pack": "ok", "files": s["files"], "MB": round(s["bytes"] / 2**20, 1),
                      "sglang_jit_modules": s["sglang_jit_modules"], "torch_extensions": s["torch_extensions"],
                      "leaves": len(leaves), "s": round(time.time() - t0, 1)}), flush=True)
    return 0


def verify(a) -> int:
    t0 = time.time()
    try:
        manifest = json.load(open(os.path.join(a.seed, MANIFEST), encoding="utf-8"))
    except (OSError, ValueError) as e:
        print(json.dumps({"ok": False, "why": [f"no manifest: {type(e).__name__}: {e}"]}), flush=True)
        return 3
    why, soft = [], {}
    if manifest.get("format") != FORMAT:
        why.append(f"manifest format {manifest.get('format')} != {FORMAT}")
    root = a.root.rstrip("/")
    missing = changed = 0
    for e in manifest["files"]:
        p = root + e["path"]
        if not os.path.isfile(p):
            missing += 1
        elif os.path.getsize(p) != e["size"] or _file_sha(p) != e["sha256"]:
            changed += 1
    if missing or changed:
        why.append(f"seeded files: {missing} missing, {changed} changed of {len(manifest['files'])}")
    fields, detail = fingerprint(a.args_file)
    seed_fp = manifest["fingerprint"]
    for k in HARD:
        if fields.get(k) != seed_fp.get(k):
            why.append(f"{k} differs: seed {str(seed_fp.get(k))[:120]!r} vs here {str(fields.get(k))[:120]!r}")
    for k in SOFT:
        if fields.get(k) != seed_fp.get(k):
            soft[k] = {"seed": seed_fp.get(k), "here": fields.get(k)}
    leaves = _jit_leaves(root)
    want = manifest["summary"]["sglang_jit_leaves"]
    if len(leaves) != want:
        why.append(f"SGLang JIT leaves: {len(leaves)} here vs {want} seeded")
    bad = {str(leaf): w for leaf in leaves if (w := _check_leaf(leaf))}
    if bad:
        why.append(f"SGLang JIT builds that would rebuild: {bad}")
    if "env" in [w.split(" ")[0] for w in why]:
        seed_env = set(manifest["detail"].get("env", []))
        here_env = set(detail["env"])
        why.append(f"env only in seed: {sorted(seed_env - here_env)}; only here: {sorted(here_env - seed_env)}")
    verdict = {"ok": not why, "why": why, "soft_diff": soft, "files": len(manifest["files"]),
               "leaves": len(leaves), "seed_built": manifest.get("built"), "s": round(time.time() - t0, 1)}
    print(json.dumps(verdict), flush=True)
    return 0 if not why else 3


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("fingerprint")
    p.add_argument("--out", required=True)
    p.add_argument("--args-file", required=True)
    p = sub.add_parser("pack")
    p.add_argument("--out", required=True)
    p.add_argument("--args-file", required=True)
    p.add_argument("--fingerprint", help="the fingerprint the first launch saved (default: take one now)")
    p.add_argument("--dir", action="append", help="a cache directory to seed (default /tmp/cache); repeatable")
    p = sub.add_parser("verify")
    p.add_argument("--seed", required=True, help="the directory holding jit_seed_manifest.json")
    p.add_argument("--args-file", required=True)
    p.add_argument("--root", default="/", help="where the seed was unpacked (default /)")
    sub.add_parser("torchext", help="date a seeded torch extension's inputs back so ninja finds it current")
    a = ap.parse_args()
    return {"fingerprint": save_fingerprint, "pack": pack, "verify": verify, "torchext": torchext}[a.cmd](a)


if __name__ == "__main__":
    sys.exit(main())


Writing jit_seed_check.py


**Start the server**, with host RAM logged from here on:


In [18]:
%%bash
# Host RAM every 10 s from here on (used and available MiB): the pinned tables and the page cache during the boot
nohup setsid bash -c 'while true; do echo "$(date +%H:%M:%S) $(free -m | grep Mem:)"; sleep 10; done' > host_mem.log 2>&1 < /dev/null &
echo $! > host_mem.pid
free -g | head -2


               total        used        free      shared  buff/cache   available
Mem:             188           3          14           0         171         184


In [19]:
%%bash
# %%bash, not `!`: `!` runs on a terminal that hangs up when the script ends, which killed the server before it started (17:28)
bash launch_server.sh

model: /tmp/model_qg
draft: /kaggle/input/models/carriem7/qwen3-8-flash-next-dflash2/transformers/agi2-h1-s1500-9a064a9068f9/1
ARC_GRAPH_BS=1 2 4 8 12 16 24 32 40 48 56 64
ARC_MAMBA_SLOTS=68
ARC_MAX_REQS=66
ARC_MEM_FRACTION=0.899
ARC_SESSIONS=1
ARC_SLOTS=64
ARC_SSM_DTYPE=bfloat16
PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True,pinned_max_round_threshold_mb:1024
SGLANG_AGI2_DFLASH_PINNED_NTNP=1
SGLANG_AGI2_DRAFT_SUCC_COMPACT=1
SGLANG_AGI2_DRAFT_VOCAB=/tmp/draft_vocab_32k.json
SGLANG_AGI2_DRAFT_WINDOW_KV_PAGED=1
SGLANG_AGI2_EAGER_CAP=1
SGLANG_AGI2_EARLY_PUBLISH=1
SGLANG_AGI2_EPLB_MAP_HOST=1
SGLANG_AGI2_EP_MAP_TRITON=1
SGLANG_AGI2_EXTEND_NOSYNC=1
SGLANG_AGI2_FAST_TOPK_EXACT=1
SGLANG_AGI2_FI_INT_WORKSPACE_BYTES=1048576
SGLANG_AGI2_FN_EXPERT_KEEP_ANY_ORDER=1
SGLANG_AGI2_FN_EXPERT_KEEP=/tmp/keep_384_code_bal4.json
SGLANG_AGI2_FN_VISION_HOST_BLOCKS=1
SGLANG_AGI2_HOST_EMBED=1
SGLANG_AGI2_LAZY_GRAPH_MASK=1
SGLANG_AGI2_MARLIN_INPLACE_REPACK=1
SGLANG_AGI2_MARLIN_NO_SWIZZLED=1
SGLANG_AGI2_MOE_EMP

In [20]:
import re, subprocess, requests

SERVER = 'http://127.0.0.1:30000'
log_pos = 0

def show_server_log():
    """Print the server log written since the last call; GPUs 1-3 repeat GPU 0's lines, so only GPU 0's are shown."""
    global log_pos
    with open('sglang.log', 'rb') as f:
        f.seek(log_pos)
        text = f.read().decode(errors='replace')
        log_pos = f.tell()
    for line in text.splitlines():
        if not re.search(r' TP[1-3][ \]]', line):
            print(line, flush=True)

t = time.time()
SERVER_UP = False
while time.time() - t < 3600:   # 60 min: /kaggle/input has read ~24 MB/s at times since 10-20
    show_server_log()
    try:
        SERVER_UP = requests.get(f'{SERVER}/health', timeout=5).status_code == 200
    except requests.RequestException:
        pass
    if SERVER_UP or subprocess.run(['pgrep', '-f', 'sglang.launch_server'], capture_output=True).returncode:
        break
    time.sleep(5)
show_server_log()
check(SERVER_UP, 'the server did not come up (its log is above)')
print(f'server up: {SERVER_UP} ({time.time() - t:.0f} s in this cell)')

/tmp/sgl/lib/python3.12/site-packages/sglang/launch_server.py:71: UserWarning: 'python -m sglang.launch_server' is still supported, but 'sglang serve' is the recommended entrypoint.
  Example: sglang serve --model-path <model> [options]
  warnings.warn(
[2026-10-31 21:18:28] Registered linear attn model: config=GigaChat35Config, backend=GDNAttnBackend, archs=['GigaChat35ForCausalLM', 'GigaChat35ForCausalLMNextN']
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section', 'mrope_interleaved'}
[transformers] Unrecognized keys in `rope_parameters` for 'rope_type'='default': {'mrope_section', 'mrope_interleaved'}
[2026-10-31 21:18:35] Setting page size to 64 for compressed QSA (full//ratio compressed addressing).
[2026-10-31 21:18:35] Attention backend not specified. Use flashinfer backend by default.
[2026-10-31 21:18:35] --enable-linear-replayssm-spec with --mamba-ssm-dtype=bfloat16: the compact checkpoint is materialized after each accepted verify

In [21]:
def gpu_mem():
    """MiB used on each GPU (0 for a line nvidia-smi did not give as a number, or when it does not answer in 20 s)."""
    out = get_ipython().getoutput('timeout 20 nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits')
    return [int(m) for m in out if m.strip().isdigit()] or [0]


def restart_server(timeout=900):
    """The server died mid-round (a CUDA OOM, a crash): stop what is left of it, start it again with the same
    switches (launch_server.sh keeps the old log) and wait for /health. Every harness conversation then rebuilds its
    session from its own history (ServerGuard, section 9)."""
    global log_pos
    subprocess.run(['pkill', '-f', 'sglang.launch_server'])
    t = time.time()
    while time.time() - t < 60 and max(gpu_mem()) > 2000:   # until the old processes let go of the GPUs
        time.sleep(2)
    # what still lives of the old server: its tensor-parallel workers and detokenizer ('sglang::...', they hold the GPUs
    # and its ports), and anything else holding a GPU (nvidia-smi's PIDs are this container's: checked 2026-10-19)
    subprocess.run(['pkill', '-9', '-f', 'sglang::'])
    for pid in get_ipython().getoutput('timeout 20 nvidia-smi --query-compute-apps=pid --format=csv,noheader'):
        pid = pid.strip()
        try:   # only ours, never PID 1: in a gVisor sandbox (Modal) nvidia-smi reports the container's init
            cmd = open(f'/proc/{pid}/cmdline', errors='replace').read() if pid.isdigit() and int(pid) > 1 else ''
        except OSError:   # it exited meanwhile
            cmd = ''
        if 'python' in cmd or 'sglang' in cmd:
            subprocess.run(['kill', '-9', pid])
    while time.time() - t < 120 and max(gpu_mem()) > 2000:
        time.sleep(2)
    log_pos = 0   # show_server_log reads the new sglang.log from its start
    subprocess.run(['bash', 'launch_server.sh'], check=True)
    while time.time() - t < timeout:
        try:
            if requests.get(f'{SERVER}/health', timeout=5).status_code == 200:
                return True
        except requests.RequestException:
            pass
        time.sleep(5)
    return False

In [22]:
# The boot facts, from the server log, failing closed in a commit (a scored rerun logs them and goes on)
if SERVER_UP:
    boot = open('sglang.log', errors='replace').read()
    pool = re.search(r'TP0[^\]]*\] KV Cache is allocated.*?#tokens: (\d+)', boot)
    keep = re.search(r'agi2 expert keep: (\d+) experts per layer kept', boot)
    draft_graph = ('disabled' if 'Disable DFLASH draft cuda graph' in boot else
                   'captured' if 'Capture draft verify CUDA graph end' in boot else 'unknown')
    free = re.findall(r'TP\d[^\]]*\] Capture draft verify CUDA graph begin.*?avail mem=([\d.]+) GB', boot)
    print(f"KV pool {int(pool[1]) if pool else 0:,} tokens a GPU; experts kept {keep[1] if keep else '?'}; "
          f"draft CUDA graph {draft_graph}; free before the draft graphs {free} GB")
    check(keep is not None and keep[1] == '384', 'the 384-expert keep list did not load')
    check(draft_graph == 'captured', f'the DFlash draft CUDA graph is {draft_graph} (needs >= 1.0 GB free)')
    check(pool is not None and int(pool[1]) > 600_000, 'the KV pool is smaller than the build gives (~4.7M with the block tier at 7.5x)')
    tier = re.search(r'TP0[^\]]*\] agi2 QSA block tier plan: .*?-> index slots (\d+) .*?device rows (\d+), host rows (\d+)', boot)
    print('QSA block tier: ' + (f'{int(tier[1]):,} index slots, {int(tier[2]):,} GPU rows, {int(tier[3]):,} host rows'
                                if tier else 'NOT ON'))
    check(tier is not None and 'OFF (host-NUMA pages unavailable' not in boot,
          'the QSA block tier did not switch on (host-NUMA pages refused?): 64 slots would park')


KV pool 3,467,136 tokens a GPU; experts kept 384; draft CUDA graph captured; free before the draft graphs ['1.34', '1.32', '1.32', '1.32'] GB
QSA block tier: 3,467,136 index slots, 374,656 GPU rows, 3,092,480 host rows


In [23]:
# What the seed did: the extension step, the check (ok, or why it was refused), and what the boot compiled anyway
print(open('/tmp/precache.log').read().strip() or 'precache still running' if os.path.exists('/tmp/precache.log') else 'no precache log')
for f in ('seed_torchext.json', 'seed_check.json'):
    print(f, open(f).read().strip() if os.path.exists(f) else '(none: no seed was used)')
new = get_ipython().getoutput("find /tmp/cache -type f -newer /tmp/jit_seed_boot_marker -not -path '*/tilelang/tmp/*' -not -path '*/.staging*' | wc -l")
print('files compiled into /tmp/cache by this boot:', new[0] if new else '?')
!free -g | head -2 && echo "peak used MiB: $(sort -k4 -n host_mem.log | tail -1)"
print(f'{time.time() - T0:.0f} s since the first cell')


precache done in 155 s
seed_torchext.json {"torch_extensions": [{"dir": "sglang_sm_reduce_avx512", "inputs": 212, "dated_back": 60, "current": true, "ninja": "no work to do"}], "ok": true}
seed_check.json {"ok": true, "why": [], "soft_diff": {"host": {"seed": "28fceab509b4", "here": "24d371fc480c"}}, "files": 1671, "leaves": 9, "seed_built": "2026-10-31 21:10:57 UTC", "s": 8.4}
files compiled into /tmp/cache by this boot: 0
               total        used        free      shared  buff/cache   available
Mem:             188         108           7          51         125          79
peak used MiB: 21:23:20 Mem:          193221      111470        7596       52783      128689       81751
526 s since the first cell


## 7. Check the prompt the model really sees

One tiny request, rendered by the server itself. The chat template must add the `xhigh` effort line; if it didn't, every request would silently run at the default effort.

In [24]:
from tokenizers import Tokenizer

if SERVER_UP:
    MODEL_DIR = os.path.dirname([p for p in glob.glob('/kaggle/input/models/**/config.json', recursive=True) if 'flash-next-nvfp4' in p][0])
    try:   # the scored rerun must reach its file even if the server dies right after its health check
        r = requests.post(f'{SERVER}/v1/chat/completions', timeout=300, json={
            'model': 'qwen3.8-27b', 'messages': [{'role': 'user', 'content': 'ping'}], 'max_tokens': 1,
            'chat_template_kwargs': {'reasoning_effort': 'xhigh'}, 'return_input_ids_in_sglext': True}).json()
        rendered = Tokenizer.from_file(f'{MODEL_DIR}/tokenizer.json').decode(r['sglext']['input_ids'],
                                                                           skip_special_tokens=False)
        print(rendered)
        check('Reasoning effort is set to xhigh' in rendered, 'the xhigh effort line is missing')
    except (requests.RequestException, KeyError, TypeError, ValueError) as e:
        check(False, f'the prompt check could not be run: {type(e).__name__}: {e}')

<|im_start|>system
Reasoning effort is set to xhigh. Please think carefully through the task, validate key assumptions, consider plausible alternatives, and prioritize correctness, consistency, and clarity in the final answer.<|im_end|>
<|im_start|>user
ping<|im_end|>
<|im_start|>assistant
<think>



## 8. The task text, and reading a grid

- **`PROMPT`** is ARC Prize's own task text (from its benchmarking harness). The harness sends it with its last instruction replaced by the tools (section 9), and checks there that the two agree.
- **`parse_grid`** reads a grid from the model's text: a `\boxed{…}` grid first, else the last JSON list of rows. Closing brackets are tried from the end until one parses as a grid, so a note in brackets after the grid ("colours used: [1, 2]") does not hide it.
- **`kaggle_grid`** is the format Kaggle scores: 1–30 rows of equal length, 1–30 integers 0–9 each.
- **Sampling** uses the model card's thinking defaults (temperature 1.0, top-p 0.95, top-k 20).

In [25]:
PROMPT = (
    "You are participating in a puzzle solving competition. You are an expert at solving puzzles.\n\n"
    "Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or "
    "transformation in the training examples that maps the input to the output, then apply that pattern "
    "to the test input to give a final output.\n\n"
    "Respond in the format of the training output examples\n\n"
    "--Training Examples--\n{training_examples}\n--End of Training Examples--\n\n"
    "--Test Input--\n{test_input}\n--End of Test Input--\n\n"
    "Your response:")

def build_prompt(task, test_input, template=PROMPT):
    examples = ''.join(f'--Example {i}-- \n\n INPUT: \n\n{json.dumps(p["input"])}\n\n'
                       f'OUTPUT: \n\n{json.dumps(p["output"])}\n\n' for i, p in enumerate(task['train']))
    return template.format(training_examples=examples, test_input=json.dumps(test_input))

def is_grid(g):
    return isinstance(g, list) and len(g) > 0 and all(isinstance(r, list) for r in g)

def parse_grid(text):
    m = re.search(r'\\boxed\{(.*?)\}', text, re.DOTALL)
    if m:
        try:
            g = json.loads(m.group(1).strip())
            if is_grid(g):
                return g
        except json.JSONDecodeError:
            pass
    # the last bracketed JSON grid: each closing bracket from the end, scanned back to its opening, until one parses as
    # a grid (a grid followed by a bracketed note, "Colours used: [1, 2]", is still found)
    end = len(text)
    for _ in range(400):
        end = text.rfind(']', 0, end)
        if end < 0:
            return None
        depth, start = 1, end - 1
        while start >= 0 and depth:
            depth += (text[start] == ']') - (text[start] == '[')
            start -= 1
        if depth:
            continue   # no opening bracket for this one: try an earlier one
        try:
            g = json.loads(text[start + 1:end + 1])
        except json.JSONDecodeError:
            continue
        if is_grid(g):
            return g
    return None

def kaggle_grid(g):
    return (is_grid(g) and 1 <= len(g) <= 30 and 1 <= len(g[0]) <= 30
            and all(len(r) == len(g[0]) and all(type(v) is int and 0 <= v <= 9 for v in r) for r in g))

SAMPLING = dict(temperature=1.0, top_p=0.95, top_k=20, min_p=0.0)   # the model card's thinking defaults

## 9. The skills harness

Submission 6's harness, with the fixes its failure study points to (`docs/harness_flaws/README.md`, "What the cases point to"):
- **The skills toolkit** (`arc_skills.py`, below): `summary()` (exact per-pair facts), `match`, `objects` with holes and shape keys, `separators`/`panels`, `symmetries`, `period`, fills, `try_all` (many ideas at once), `try_library` (whole-task transforms, exact fits only) and `recipe()` (how to investigate; it names no likely rules). Failing submissions also say, object by object, what is missing, extra, misplaced or recoloured. The model calls `summary()` first in nearly every attempt.
- **Grading from the worker's own copy of the task**, so a python call that rebinds `test` or edits a grid cannot change what `submit` grades.
- **Crash recovery** (`ServerGuard`): a server that dies mid-run is restarted with the same switches (up to 3 times), and every conversation continues from its own history.
- **Session park** behind the KV gate (below).

**The sandbox:** one worker process per attempt, the task preloaded as numpy arrays, state kept between calls, 60 s per call (then a restart, which loses the Python state), 4 GB of memory, 12,000 characters of output shown to the model. New here:
- the worker clips what a call printed to 200,000 characters before replying (an 80 MB print used to overflow the reply line and end the attempt);
- every request carries an id that the worker's reply echoes, so the parent skips anything the model's code wrote to the worker's stdout itself, text or JSON;
- a worker that dies between calls (a broken pipe) is restarted like one that crashes during a call;
- the worker runs in its own process group: closing it kills whatever the model's code started too, and waits at most 5 s.

In [26]:
%%writefile arc_sandbox_worker.py
"""The Python the model's tool calls run in: one worker process per attempt, state kept between calls.

Reads one JSON request per line on stdin, answers one JSON line on stdout:
  {"op": "init", "task": {...}}    load the task: training pairs and test inputs as numpy arrays
  {"op": "exec", "code": "..."}    run code in the shared namespace, return what it printed (and any error)
  {"op": "submit", "code": "..."}  run code, then check `transform` on every training pair and apply it to the tests
  {"op": "apply", "inputs": [...]}  the harness's call, not the model's: the current `transform` on more inputs
"""
import contextlib
import io
import json
import sys
import time
import traceback

import numpy as np

np.set_printoptions(legacy="1.25")   # printed numbers stay plain: (1, 16), not (np.int64(1), np.int64(16))
NS = {}
TASK = {}   # the task as the harness keeps it (set by init). Grading, check() and the skills read this, never the
            # model's namespace, where `test = test_inputs[0]` or an in-place edit would change what gets graded
PICS = []   # pictures asked for by show_image during the current call: lists of grids
LISTS = False   # how the last submit handed grids to transform; the harness's apply hands them the same way
SKILLS_ON = False   # set by init: the skills arm (arc_skills.py in the namespace, object-level submit feedback)


PIC_MAX_CELLS, PIC_MAX_PER_ATTEMPT = 1800, 8   # every cell is one visual token; the server encodes pictures together
PIC_COUNT = 0   # pictures this attempt has shown (init resets it)


def show_image(*grids):
    """Show up to 4 grids (each at most 30x30, 1,800 cells together) to yourself as one picture, side by side; it
    comes back with this call's output. An attempt can show 8 pictures."""
    global PIC_COUNT
    arrs = [np.asarray(g) for g in grids[:4]]
    if not arrs or len(PICS) >= 3:
        print("[show_image: nothing shown (no grids, or 3 pictures already in this call)]")
        return
    bad = [str(a.shape) for a in arrs if a.ndim != 2 or not (1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30)]
    if bad:
        print(f"[show_image: nothing shown: each grid must be 1-30 x 1-30 cells, got {', '.join(bad)}. Every cell is "
              "already drawn large, so there is no need to magnify]")
        return
    cells = sum(a.size for a in arrs)
    if cells > PIC_MAX_CELLS:
        print(f"[show_image: nothing shown: {cells} cells; one picture holds at most {PIC_MAX_CELLS}, so show fewer grids]")
        return
    if PIC_COUNT >= PIC_MAX_PER_ATTEMPT:
        print(f"[show_image: nothing shown: this attempt has used its {PIC_MAX_PER_ATTEMPT} pictures]")
        return
    PIC_COUNT += 1
    gs = [a.astype(int).tolist() for a in arrs]
    PICS.append(gs)
    print(f"[picture {len(PICS)}: " + ", ".join(f"{len(g)}x{len(g[0])}" for g in gs) + "]")


def show(grid):
    """Print a grid as digit rows with row and column indices."""
    g = np.asarray(grid)
    print("    " + "".join(str(c % 10) for c in range(g.shape[1])) + f"   ({g.shape[0]}x{g.shape[1]})")
    for r, row in enumerate(g):
        print(f"{r:>3} " + "".join(str(int(v)) for v in row))


def diff(a, b):
    """Cells where two same-shape grids differ, as (row, col, a, b)."""
    a, b = np.asarray(a), np.asarray(b)
    if a.shape != b.shape:
        return f"shapes differ: {a.shape} vs {b.shape}"
    return [(int(r), int(c), int(a[r, c]), int(b[r, c])) for r, c in np.argwhere(a != b)]


def objects(grid, bg=None, diagonal=False, multicolor=False):
    """Connected components: dicts with color (or colors), cells, bbox (r0, c0, r1, c1), size, and shape (the
    object's own cells cropped to its bbox, other cells -1). bg defaults to the most common colour."""
    g = np.asarray(grid)
    if bg is None:
        vals, counts = np.unique(g, return_counts=True)
        bg = int(vals[np.argmax(counts)])
    seen = np.zeros(g.shape, bool)
    steps = [(1, 0), (-1, 0), (0, 1), (0, -1)] + ([(1, 1), (1, -1), (-1, 1), (-1, -1)] if diagonal else [])
    out = []
    for r0 in range(g.shape[0]):
        for c0 in range(g.shape[1]):
            if seen[r0, c0] or g[r0, c0] == bg:
                continue
            stack, cells = [(r0, c0)], []
            seen[r0, c0] = True
            while stack:
                r, c = stack.pop()
                cells.append((r, c))
                for dr, dc in steps:
                    rr, cc = r + dr, c + dc
                    if (0 <= rr < g.shape[0] and 0 <= cc < g.shape[1] and not seen[rr, cc] and g[rr, cc] != bg
                            and (multicolor or g[rr, cc] == g[r0, c0])):
                        seen[rr, cc] = True
                        stack.append((rr, cc))
            rs, cs = [p[0] for p in cells], [p[1] for p in cells]
            box = (min(rs), min(cs), max(rs), max(cs))
            shape = np.full((box[2] - box[0] + 1, box[3] - box[1] + 1), -1, int)
            for r, c in cells:
                shape[r - box[0], c - box[1]] = g[r, c]
            colors = sorted({int(g[r, c]) for r, c in cells})
            out.append({"color": colors[0] if len(colors) == 1 else colors, "cells": sorted(cells), "bbox": box,
                        "size": len(cells), "shape": shape})
    return out


def _grid_text(g):
    return "\n".join("".join(str(int(v)) for v in row) for row in np.asarray(g))


def _diff_text(expected, got, max_cells=900):
    """The expected grid, and the program's output with correct cells shown as '.'."""
    e, g = np.asarray(expected), np.asarray(got)
    if e.shape != g.shape:
        return (f"Wrong output SIZE: expected {e.shape[0]}x{e.shape[1]}, your program returned "
                f"{g.shape[0]}x{g.shape[1] if g.ndim == 2 else '?'}.\nExpected:\n{_grid_text(e)}")
    wrong = int((e != g).sum())
    if e.size > max_cells:
        return f"{wrong} of {e.size} cells are wrong (grid too large to print the diff)."
    marked = "\n".join("".join("." if e[r, c] == g[r, c] else str(int(g[r, c])) for c in range(e.shape[1]))
                       for r in range(e.shape[0]))
    return (f"{wrong} of {e.size} cells are wrong.\nExpected ({e.shape[0]}x{e.shape[1]}):\n{_grid_text(e)}\n"
            f"Your output, with CORRECT cells shown as '.':\n{marked}")


def _as_grid(x):
    a = np.asarray(x)
    if a.ndim != 2 or not (1 <= a.shape[0] <= 30 and 1 <= a.shape[1] <= 30):
        raise ValueError(f"not a 1-30 x 1-30 grid (got shape {a.shape})")
    if a.dtype.kind == "f" and np.isfinite(a).all() and (a == np.round(a)).all():
        a = a.astype(int)  # whole-number floats (np.zeros((h, w)) is float) are unambiguous
    if a.dtype.kind not in "iub":
        raise ValueError(f"cells must be integers (got dtype {a.dtype})")
    if a.min() < 0 or a.max() > 9:
        raise ValueError("cell values must be 0-9")
    return a.astype(int)


def check(fn, verbose=True):
    """Run fn on every training input; report per pair whether it matches exactly and which cells differ."""
    ok = 0
    for i, (x, y) in enumerate(TASK["train"]):
        try:
            got = _as_grid(fn(x.copy()))
        except Exception as e:  # noqa: BLE001 - report any error the model's code raises
            if verbose:
                print(f"Example {i + 1}: ERROR {type(e).__name__}: {e}")
            continue
        if got.shape == y.shape and (got == y).all():
            ok += 1
            if verbose:
                print(f"Example {i + 1}: OK")
        elif verbose:
            print(f"Example {i + 1}: WRONG\n{_diff_text(y, got)}")
    if verbose:
        print(f"{ok} of {len(TASK['train'])} examples reproduced")
    return ok == len(TASK["train"])


D4 = {"rotated 90 degrees": lambda a: np.rot90(a, 1), "rotated 180 degrees": lambda a: np.rot90(a, 2),
      "rotated 270 degrees": lambda a: np.rot90(a, 3), "flipped upside down": np.flipud,
      "flipped left-right": np.fliplr, "transposed": lambda a: a.T, "anti-transposed": lambda a: np.rot90(a, 2).T}


def _rules_broken(train, x, y):
    """Shape and colour rules every training pair obeys that the test output `y` (for test input `x`) breaks."""
    out = []
    pairs = [(np.asarray(i), np.asarray(o)) for i, o in train]
    cols = lambda a: set(np.unique(a).tolist())
    if all(o.shape == i.shape for i, o in pairs) and y.shape != x.shape:
        out.append(f"every training output has its input's size, but yours is {y.shape[0]}x{y.shape[1]} for a "
                   f"{x.shape[0]}x{x.shape[1]} input")
    ratios = {(o.shape[0] / i.shape[0], o.shape[1] / i.shape[1]) for i, o in pairs}
    if len(ratios) == 1 and (y.shape[0] / x.shape[0], y.shape[1] / x.shape[1]) != next(iter(ratios)) and not out:
        r = next(iter(ratios))
        out.append(f"every training output is {r[0]:g}x the input's height and {r[1]:g}x its width, but yours is not")
    if all(cols(o) <= cols(i) for i, o in pairs) and not cols(y) <= cols(x):
        out.append(f"no training output uses a colour its input lacks, but yours adds {sorted(cols(y) - cols(x))}")
    new_ok = set().union(*[cols(o) - cols(i) for i, o in pairs])
    if not (cols(y) - cols(x)) <= new_ok and not all(cols(o) <= cols(i) for i, o in pairs):
        out.append(f"yours adds colours {sorted(cols(y) - cols(x) - new_ok)} that no training output adds")
    if all(cols(o) == cols(i) for i, o in pairs) and cols(y) != cols(x):
        out.append(f"every training output uses exactly its input's colours, but yours uses {sorted(cols(y))} for an "
                   f"input with {sorted(cols(x))}")
    if all(o.shape == i.shape for i, o in pairs) and y.shape == x.shape:
        fr = [float((o != i).mean()) for i, o in pairs]
        f = float((y != x).mean())
        if f < 0.5 * min(fr) - 1e-9 or f > 2 * max(fr) + 1e-9:
            out.append(f"the training outputs change {min(fr):.0%}-{max(fr):.0%} of their input's cells, yours "
                       f"{f:.0%}")
    return out


def _overfit_flags(fn, arg, x_test, y_test, budget_s=15.0):
    """Signs that an accepted program fits the training pairs without having the general rule. Returns notes.
    The traced and transformed runs stop after `budget_s` seconds: the whole submit call must stay inside the tool's
    60 s limit, or the acceptance itself would be lost."""
    t0 = time.time()
    train = TASK["train"]
    notes = _rules_broken(train, x_test, np.asarray(y_test))
    # code only the test input runs (frames of the model's own code: <submit> and earlier <tool> calls)
    seen, cur = {}, [None]

    def ltrace(frame, event, a):
        if event == "line":
            seen[cur[0]].add((frame.f_code.co_filename, frame.f_code.co_name, frame.f_lineno))
        return ltrace

    def gtrace(frame, event, a):
        return ltrace if frame.f_code.co_filename in ("<submit>", "<tool>") else None

    def traced(tag, g):
        cur[0], seen[tag] = tag, set()
        sys.settrace(gtrace)
        try:
            fn(arg(g))
        except Exception:  # noqa: BLE001 - only the lines matter here
            pass
        finally:
            sys.settrace(None)

    for k, (x, _) in enumerate(train):
        traced(k, x)
        if time.time() - t0 > budget_s / 2:   # tracing is slow on a heavy program: skip the rest of the checks
            return notes
    traced("test", x_test)
    only = sorted({ln for f, n, ln in seen["test"] - set().union(*[seen[k] for k in range(len(train))]) if f == "<submit>"})
    if only:
        notes.append(f"lines {only} of your program run on the test input but on no training input, so no example "
                     "checks them")

    # symmetries kept on every training input but not on the test input
    def commutes(g, t):
        try:
            a, b = _as_grid(fn(arg(D4[t](g)))), D4[t](_as_grid(fn(arg(g))))
            return a.shape == b.shape and bool((a == b).all())
        except Exception:  # noqa: BLE001
            return False
    broken = []
    for t in D4:
        if time.time() - t0 > budget_s:
            break
        if all(commutes(x, t) for x, _ in train) and not commutes(x_test, t):
            broken.append(t)
    if broken:
        notes.append("on every training input your program gives the same result whether the grid is " + ", ".join(broken)
                     + " first or not, but not on the test input")
    return notes


def _submit(code, lists=False, overfit=False):
    """Run the code, then judge `transform` on every training pair (feedback for at most 2 failing ones) and the tests.
    `transform` gets each grid as a numpy array, or as a list of lists with `lists` (what the gridprog prompt promises).
    With `overfit`, an accepted program also gets the overfit checks (below)."""
    global LISTS
    exec(compile(code, "<submit>", "exec"), NS)  # noqa: S102 - this process exists to run it
    arg = (lambda g: g.tolist()) if lists else (lambda g: g.copy())
    LISTS = lists
    fn = NS.get("transform")
    if not callable(fn):
        return {"n_ok": 0, "n": len(TASK["train"]), "feedback": "Your code does not define transform(grid).",
                "test_outputs": None}
    n_ok, lines, fail_pair = 0, [], None
    for i, (x, y) in enumerate(TASK["train"]):
        try:
            got = _as_grid(fn(arg(x)))
            if got.shape == y.shape and (got == y).all():
                n_ok += 1
                continue
            msg = _diff_text(y, got)
            if SKILLS_ON and got.shape == y.shape:   # the skills arm: also what is wrong, object by object
                import arc_skills
                obj = arc_skills.object_diff(y, got)
                if obj:
                    msg += "\nObject by object:\n" + "\n".join("- " + s for s in obj[:6]) + \
                           (f"\n- ... and {len(obj) - 6} more object differences" if len(obj) > 6 else "")
            if fail_pair is None:   # the first failing example: expected and produced, for a picture
                fail_pair = [i + 1, y.tolist(), got.tolist()]
        except Exception as e:  # noqa: BLE001
            msg = f"Error: {type(e).__name__}: {e}"
        if len(lines) < 2:
            lines.append(f"## Example {i + 1} failed\n{msg}")
    tests, test_err = [], None
    for t in TASK["test"]:
        try:
            tests.append(_as_grid(fn(arg(t))).tolist())
        except Exception as e:  # noqa: BLE001
            tests.append(None)
            test_err = f"{type(e).__name__}: {e}"
    n = len(TASK["train"])
    flags = []
    if n_ok == n and test_err is None:
        fb = f"Accepted: reproduces all {n} examples. Its output on the test input is your answer."
        if overfit:
            try:
                flags = _overfit_flags(fn, arg, TASK["test"][0], tests[0])
            except Exception as e:  # noqa: BLE001 - a failed check must not cost the acceptance
                flags = []
                print(f"[harness: the overfit check failed: {type(e).__name__}]")
    elif n_ok == n:
        fb = f"Reproduces all {n} examples but FAILS on a test input: {test_err} -- fix it and submit again."
    else:
        fb = f"Reproduced {n_ok} of {n} examples.\n" + "\n".join(lines) + "\nFix the rule and submit again."
    return {"n_ok": n_ok, "n": n, "feedback": fb, "test_outputs": tests, "fail_pair": fail_pair, "overfit": flags}


OUT_MAX = 200_000   # characters of one call's output sent back (the harness shows the model 12,000 of them)


def _clip_out(text):
    """Keep the reply line small whatever the code printed: the head and the tail, and how much was cut between."""
    if len(text) <= OUT_MAX:
        return text
    head, tail = text[:OUT_MAX * 2 // 3], text[-(OUT_MAX // 3):]
    return f"{head}\n... [{len(text) - len(head) - len(tail)} characters cut] ...\n{tail}"


def _reply(obj):
    sys.stdout.write(json.dumps(obj) + "\n")
    sys.stdout.flush()


def _err_text(e):
    """The model's own frames only (filenames <tool>/<submit>); a compile error has none, so just its message."""
    tb = e.__traceback__
    while tb is not None and not tb.tb_frame.f_code.co_filename.startswith("<"):
        tb = tb.tb_next
    if tb is None:
        return "".join(traceback.format_exception_only(type(e), e))
    return "".join(traceback.format_exception(type(e), e, tb))


for line in sys.stdin:
    req = json.loads(line)
    if req["op"] == "init":
        t = req["task"]
        tr = [(np.array(p["input"], dtype=int), np.array(p["output"], dtype=int)) for p in t["train"]]
        TASK.clear()
        TASK.update(train=tr, test=[np.array(p["input"], dtype=int) for p in t["test"]])
        PIC_COUNT = 0
        mine = [(x.copy(), y.copy()) for x, y in tr]   # the model's own copies, free to rebind or edit
        NS.clear()
        NS.update(np=np, show=show, diff=diff, objects=objects, check=check, show_image=show_image, train=mine,
                  train_inputs=[x for x, _ in mine], train_outputs=[y for _, y in mine],
                  test_inputs=[x.copy() for x in TASK["test"]])
        NS["test"] = NS["test_inputs"]
        SKILLS_ON = bool(req.get("skills"))
        if SKILLS_ON:   # the skills module (arc_skills.py beside this file), its functions in the namespace; it
            import arc_skills   # reads the task from copies of TASK, like the grader
            arc_skills.NS = {"train": [(x.copy(), y.copy()) for x, y in tr], "test_inputs": [x.copy() for x in TASK["test"]]}
            NS.update({k: getattr(arc_skills, k) for k in arc_skills.SKILL_NAMES})
        _reply({"ok": True, "id": req.get("id")})   # the request's id: the parent takes only its own reply
    elif req["op"] == "apply":   # the harness's call: the session's transform on a task's other test inputs
        outs, fn = [], NS.get("transform")
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            for x in req.get("inputs") or []:
                try:
                    a = np.array(x, dtype=int)
                    outs.append(_as_grid(fn(a.tolist() if LISTS else a.copy())).tolist())
                except Exception:  # noqa: BLE001 - no grid for this input
                    outs.append(None)
        _reply({"outputs": outs, "id": req.get("id")})
    elif req["op"] in ("exec", "submit"):
        PICS.clear()
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            try:
                if req["op"] == "exec":
                    exec(compile(req["code"], "<tool>", "exec"), NS)  # noqa: S102
                    res = {"error": None}
                else:
                    res = _submit(req["code"], req.get("lists", False), req.get("overfit", False))
            except Exception as e:  # noqa: BLE001
                res = {"error": _err_text(e)}
        res["out"] = _clip_out(buf.getvalue())
        res["pictures"] = list(PICS)
        res["id"] = req.get("id")
        _reply(res)

Writing arc_sandbox_worker.py


In [27]:
%%writefile arc_skills.py
"""ARC skills: tested helpers preloaded in the harness sandbox. The prompt shows only SKILLS_INDEX; `help(name)` prints a
skill's full description with an example. Every function takes and returns numpy int arrays (lists also accepted)."""
from collections import Counter

import numpy as np

# ----------------------------------------------------------------------------------------------------------- basics


def _g(x):
    return np.asarray(x, dtype=int)


def bg(g):
    """The background colour: the most common colour of g."""
    v, c = np.unique(_g(g), return_counts=True)
    return int(v[np.argmax(c)])


D4_NAMES = ("identity", "rot90", "rot180", "rot270", "flip_lr", "flip_ud", "transpose", "anti_transpose")


def d4(g, name):
    """g rotated, flipped or transposed. name: rot90 (counter-clockwise), rot180, rot270, flip_lr (mirror left-right),
    flip_ud (upside down), transpose (main diagonal), anti_transpose (other diagonal), identity."""
    g = _g(g)
    return {"identity": g, "rot90": np.rot90(g, 1), "rot180": np.rot90(g, 2), "rot270": np.rot90(g, 3),
            "flip_lr": np.fliplr(g), "flip_ud": np.flipud(g), "transpose": g.T,
            "anti_transpose": np.rot90(g, 2).T}[name].copy()


def symmetries(g, ignore=None):
    """The D4 names (other than identity) that leave g unchanged. With `ignore` (a colour), cells of that colour
    match anything, e.g. a damaged symmetric pattern whose damage is painted in one colour."""
    g = _g(g)
    out = []
    for n in D4_NAMES[1:]:
        t = d4(g, n)
        if t.shape != g.shape:
            continue
        eq = t == g
        if ignore is not None:
            eq |= (t == ignore) | (g == ignore)
        if eq.all():
            out.append(n)
    return out


def period(g, ignore=None):
    """The smallest tile (rows, cols) that repeats to make g: g[r, c] == g[r % rows, c % cols] everywhere. With `ignore`
    (a colour), cells of that colour match anything. Returns (rows, cols); (H, W) means no repetition."""
    g = _g(g)
    H, W = g.shape
    known = np.ones_like(g, bool) if ignore is None else g != ignore
    best = (H, W)
    for pr in range(1, H + 1):
        for pc in range(1, W + 1):
            if pr * pc >= best[0] * best[1]:
                break
            ok = True
            for r0 in range(pr):
                for c0 in range(pc):
                    vals = g[r0::pr, c0::pc][known[r0::pr, c0::pc]]
                    if len(vals) and (vals != vals[0]).any():
                        ok = False
                        break
                if not ok:
                    break
            if ok:
                best = (pr, pc)
                break
    return best


def fill_by_period(g, ignore):
    """Repaint the cells of colour `ignore` from the smallest repeating tile of the other cells (period with ignore).
    Returns the repaired grid, or g unchanged when some tile cell is never seen."""
    g = _g(g).copy()
    pr, pc = period(g, ignore)
    known = g != ignore
    for r0 in range(pr):
        for c0 in range(pc):
            vals = g[r0::pr, c0::pc][known[r0::pr, c0::pc]]
            if len(vals) == 0:
                return _g(g)
            g[r0::pr, c0::pc] = vals[0]
    return g


def fill_by_symmetry(g, ignore, which=None):
    """Repaint the cells of colour `ignore` from their mirror images under the symmetries `which` (default: those
    symmetries(g, ignore) finds). Cells no mirror covers keep `ignore`."""
    g = _g(g).copy()
    which = which or symmetries(g, ignore)
    for _ in range(3):
        for n in which:
            t = d4(g, n)
            if t.shape != g.shape:
                continue
            m = (g == ignore) & (t != ignore)
            g[m] = t[m]
    return g


# ----------------------------------------------------------------------------------------------------------- objects


def objects(g, bg=None, diagonal=False, multicolor=False):
    """Connected components of non-background cells. Each is a dict: color (or sorted colors with multicolor), cells
    [(r, c)], bbox (r0, c0, r1, c1) inclusive, size, shape (the object cropped to its bbox, other cells -1), holes
    (number of enclosed background regions) and key (a shape signature: same key = same shape and colours, any
    position). bg defaults to the most common colour."""
    g = _g(g)
    if bg is None:
        bg = globals()["bg"](g)
    H, W = g.shape
    seen = np.zeros(g.shape, bool)
    steps = [(1, 0), (-1, 0), (0, 1), (0, -1)] + ([(1, 1), (1, -1), (-1, 1), (-1, -1)] if diagonal else [])
    out = []
    for r0 in range(H):
        for c0 in range(W):
            if seen[r0, c0] or g[r0, c0] == bg:
                continue
            stack, cells = [(r0, c0)], []
            seen[r0, c0] = True
            while stack:
                r, c = stack.pop()
                cells.append((r, c))
                for dr, dc in steps:
                    rr, cc = r + dr, c + dc
                    if (0 <= rr < H and 0 <= cc < W and not seen[rr, cc] and g[rr, cc] != bg
                            and (multicolor or g[rr, cc] == g[r0, c0])):
                        seen[rr, cc] = True
                        stack.append((rr, cc))
            rs, cs = [p[0] for p in cells], [p[1] for p in cells]
            box = (min(rs), min(cs), max(rs), max(cs))
            shape = np.full((box[2] - box[0] + 1, box[3] - box[1] + 1), -1, int)
            for r, c in cells:
                shape[r - box[0], c - box[1]] = g[r, c]
            colors = sorted({int(g[r, c]) for r, c in cells})
            out.append({"color": colors[0] if len(colors) == 1 else colors, "cells": sorted(cells), "bbox": box,
                        "size": len(cells), "shape": shape, "holes": _holes(shape),
                        "key": (shape.shape, tuple(shape.flatten().tolist()))})
    return out


def _holes(shape):
    """Background regions inside an object's bbox (cells -1) that do not touch the bbox border."""
    m = shape == -1
    H, W = m.shape
    seen = np.zeros_like(m)
    n = 0
    for r0 in range(H):
        for c0 in range(W):
            if not m[r0, c0] or seen[r0, c0]:
                continue
            stack, border = [(r0, c0)], False
            seen[r0, c0] = True
            while stack:
                r, c = stack.pop()
                border |= r in (0, H - 1) or c in (0, W - 1)
                for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    rr, cc = r + dr, c + dc
                    if 0 <= rr < H and 0 <= cc < W and m[rr, cc] and not seen[rr, cc]:
                        seen[rr, cc] = True
                        stack.append((rr, cc))
            n += not border
    return n


def match(a, b, bg=None):
    """Pair the objects of grid a with those of grid b (same-size or not). Returns lines describing each: 'moved by
    (dr, dc)', 'rotated/flipped (name)' (and moved), 'recoloured c1->c2', objects only in a ('gone') or only in b
    ('new'), then 'same' (same place); more than 8 'same' objects become one line. A quick way to see what a training
    pair does to its objects."""
    a, b = _g(a), _g(b)
    oa = objects(a, bg)
    ob = objects(b, bg)
    used, lines, same = set(), [], []
    for i, x in enumerate(oa):
        best, how = None, None
        for j, y in enumerate(ob):
            if j in used:
                continue
            if x["key"] == y["key"]:
                dr, dc = y["bbox"][0] - x["bbox"][0], y["bbox"][1] - x["bbox"][1]
                cand = ("same" if (dr, dc) == (0, 0) else f"moved by ({dr}, {dc})")
                if best is None or (dr, dc) == (0, 0):
                    best, how = j, cand
            elif best is None and x["shape"].shape == y["shape"].shape and ((x["shape"] == -1) == (y["shape"] == -1)).all() \
                    and x["bbox"] == y["bbox"]:
                best, how = j, f"recoloured {x['color']}->{y['color']}"
            elif best is None and x["size"] == y["size"] and x["size"] > 1:
                turned = [n for n in D4_NAMES[1:] if d4(x["shape"], n).shape == y["shape"].shape
                          and (d4(x["shape"], n) == y["shape"]).all()]
                if turned:
                    best, how = j, f"{turned[0]} and moved to bbox {y['bbox']}"
        if best is None:
            lines.append(f"gone: colour {x['color']} size {x['size']} at bbox {x['bbox']}")
        else:
            used.add(best)
            (same if how == "same" else lines).append(f"{how}: colour {x['color']} size {x['size']} at bbox {x['bbox']}")
    lines.sort(key=lambda s: s.startswith("gone"))   # what changed first; the 12,000-character cut keeps the head
    for j, y in enumerate(ob):
        if j not in used:
            lines.append(f"new: colour {y['color']} size {y['size']} at bbox {y['bbox']}")
    if len(same) > 8:
        counts = Counter(int(s.split("colour ")[1].split()[0]) for s in same)
        same = [f"same place: {len(same)} objects (" + ", ".join(f"colour {c} x{n}" for c, n in sorted(counts.items())) + ")"]
    return lines + same


# ----------------------------------------------------------------------------------------------------------- layout


def separators(g):
    """Full rows and full columns of one non-background colour: (rows, cols, colour) for the most common such colour,
    or ([], [], None). Lines of a colour found nowhere else usually border panels; a full line that is part of a
    larger shape usually does not."""
    g = _g(g)
    H, W = g.shape
    b = bg(g)
    rows = {r: int(g[r, 0]) for r in range(H) if (g[r] == g[r, 0]).all() and g[r, 0] != b}
    cols = {c: int(g[0, c]) for c in range(W) if (g[:, c] == g[0, c]).all() and g[0, c] != b}
    if not rows and not cols:
        return [], [], None
    colour = Counter(list(rows.values()) + list(cols.values())).most_common(1)[0][0]
    return (sorted(r for r, v in rows.items() if v == colour), sorted(c for c, v in cols.items() if v == colour),
            colour)


def panels(g):
    """Split g by its separator lines (see separators) into panels: a list of (r0, c0, panel grid), row-major. Without
    separators, [(0, 0, g)]."""
    g = _g(g)
    rs, cs, colour = separators(g)
    if colour is None:
        return [(0, 0, g.copy())]
    out = []
    rb = [-1] + rs + [g.shape[0]]
    cb = [-1] + cs + [g.shape[1]]
    for i in range(len(rb) - 1):
        for j in range(len(cb) - 1):
            r0, r1, c0, c1 = rb[i] + 1, rb[i + 1], cb[j] + 1, cb[j + 1]
            if r1 > r0 and c1 > c0:
                out.append((r0, c0, g[r0:r1, c0:c1].copy()))
    return out


def bbox(g, colours=None, bg=None):
    """(r0, c0, r1, c1), inclusive, of the cells whose colour is in `colours` (default: every non-background cell);
    None if there are none."""
    g = _g(g)
    m = (g != (globals()["bg"](g) if bg is None else bg)) if colours is None else np.isin(g, list(colours))
    if not m.any():
        return None
    rs, cs = np.where(m)
    return int(rs.min()), int(cs.min()), int(rs.max()), int(cs.max())


def crop(g, box):
    """g[r0..r1, c0..c1] for an inclusive box (r0, c0, r1, c1)."""
    r0, c0, r1, c1 = box
    return _g(g)[r0:r1 + 1, c0:c1 + 1].copy()


def find(g, sub):
    """Every (r, c) where grid `sub` appears exactly inside g."""
    g, sub = _g(g), _g(sub)
    H, W = g.shape
    h, w = sub.shape
    return [(r, c) for r in range(H - h + 1) for c in range(W - w + 1) if (g[r:r + h, c:c + w] == sub).all()]


# ----------------------------------------------------------------------------------------------------------- painting


def scale(g, k):
    """Each cell becomes a k x k block (k may be a pair (kr, kc))."""
    kr, kc = (k, k) if np.isscalar(k) else k
    return np.kron(_g(g), np.ones((kr, kc), int))


def downscale(g, k):
    """The inverse of scale: one cell per k x k block (k may be a pair); None if a block is not one colour."""
    g = _g(g)
    kr, kc = (k, k) if np.isscalar(k) else k
    H, W = g.shape
    if H % kr or W % kc:
        return None
    blocks = g.reshape(H // kr, kr, W // kc, kc)
    first = blocks[:, :1, :, :1]
    if (blocks != first).any():
        return None
    return blocks[:, 0, :, 0].copy()


def tile(g, rows, cols):
    """g repeated rows x cols times."""
    return np.tile(_g(g), (rows, cols))


def recolor(g, mapping):
    """Every colour a in `mapping` (a dict a -> b) repainted b; other colours kept."""
    g = _g(g)
    out = g.copy()
    for a, b in mapping.items():
        out[g == a] = b
    return out


def flood(g, r, c, colour, diagonal=False):
    """Flood-fill the region of g[r, c]'s colour that contains (r, c) with `colour`."""
    g = _g(g).copy()
    old = g[r, c]
    if old == colour:
        return g
    stack = [(r, c)]
    steps = [(1, 0), (-1, 0), (0, 1), (0, -1)] + ([(1, 1), (1, -1), (-1, 1), (-1, -1)] if diagonal else [])
    while stack:
        y, x = stack.pop()
        if 0 <= y < g.shape[0] and 0 <= x < g.shape[1] and g[y, x] == old:
            g[y, x] = colour
            stack.extend((y + dy, x + dx) for dy, dx in steps)
    return g


def fill_holes(g, colour, bg=None):
    """Paint every background region that does not touch the grid's border (an enclosed hole) with `colour`."""
    g = _g(g).copy()
    b = globals()["bg"](g) if bg is None else bg
    outside = np.zeros(g.shape, bool)
    H, W = g.shape
    stack = [(r, c) for r in range(H) for c in (0, W - 1)] + [(r, c) for r in (0, H - 1) for c in range(W)]
    while stack:
        r, c = stack.pop()
        if 0 <= r < H and 0 <= c < W and not outside[r, c] and g[r, c] == b:
            outside[r, c] = True
            stack.extend(((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)))
    g[(g == b) & ~outside] = colour
    return g


def gravity(g, direction, bg=None):
    """Let every non-background cell fall as far as it can: direction 'down', 'up', 'left' or 'right' (cells keep their
    order along the line)."""
    g = _g(g)
    b = globals()["bg"](g) if bg is None else bg
    k = {"down": 0, "left": 1, "up": 2, "right": 3}[direction]   # np.rot90 turns counter-clockwise: left edge -> bottom
    t = np.rot90(g, k).copy()   # make it fall 'down'
    for c in range(t.shape[1]):
        col = t[:, c]
        cells = col[col != b]
        t[:, c] = np.concatenate([np.full(len(col) - len(cells), b), cells])
    return np.rot90(t, -k).copy()


def draw_line(g, r0, c0, r1, c1, colour):
    """A straight line (horizontal, vertical or 45-degree diagonal) from (r0, c0) to (r1, c1), inclusive."""
    g = _g(g).copy()
    n = max(abs(r1 - r0), abs(c1 - c0))
    dr = (r1 > r0) - (r1 < r0)
    dc = (c1 > c0) - (c1 < c0)
    for i in range(n + 1):
        r, c = r0 + dr * i, c0 + dc * i
        if 0 <= r < g.shape[0] and 0 <= c < g.shape[1]:
            g[r, c] = colour
    return g


# ----------------------------------------------------------------------------------------------------------- summary


def _relations(x, y):
    """How output y relates to input x, as short phrases (exact checks only)."""
    out = []
    if x.shape == y.shape:
        if (x == y).all():
            return ["output = input"]
        for n in D4_NAMES[1:]:
            if d4(x, n).shape == y.shape and (d4(x, n) == y).all():
                out.append(f"output = {n}(input)")
        diff = x != y
        out.append(f"same size; {int(diff.sum())} of {x.size} cells change")
        trans = Counter((int(a), int(b)) for a, b in zip(x[diff], y[diff]))
        out.append("changes: " + ", ".join(f"{a}->{b} x{n}" for (a, b), n in trans.most_common(8)))
    else:
        for n in D4_NAMES:
            t = d4(x, n)
            if t.shape == y.shape and (t == y).all():
                out.append(f"output = {n}(input)")
        if y.shape[0] <= x.shape[0] and y.shape[1] <= x.shape[1]:
            pos = find(x, y)
            if pos:
                out.append(f"output is the input's sub-grid at {pos[:3]}" + (" (and more)" if len(pos) > 3 else ""))
        for k in range(2, 6):
            if y.shape == (x.shape[0] * k, x.shape[1] * k) and (scale(x, k) == y).all():
                out.append(f"output = scale(input, {k})")
            if x.shape == (y.shape[0] * k, y.shape[1] * k) and downscale(x, k) is not None and (downscale(x, k) == y).all():
                out.append(f"output = downscale(input, {k})")
        if y.shape[0] % x.shape[0] == 0 and y.shape[1] % x.shape[1] == 0 and y.shape != x.shape:
            tr, tc = y.shape[0] // x.shape[0], y.shape[1] // x.shape[1]
            blocks = [[y[i * x.shape[0]:(i + 1) * x.shape[0], j * x.shape[1]:(j + 1) * x.shape[1]] for j in range(tc)]
                      for i in range(tr)]
            names = []
            for row in blocks:
                for bl in row:
                    hit = [n for n in D4_NAMES if d4(x, n).shape == bl.shape and (d4(x, n) == bl).all()]
                    names.append(hit[0] if hit else "?")
            if "?" not in names:
                out.append(f"output = {tr}x{tc} tiles of the input: " + ", ".join(names))
        out.append(f"size {x.shape[0]}x{x.shape[1]} -> {y.shape[0]}x{y.shape[1]}")
    return out


def _grid_facts(g, name):
    rs, cs, colour = separators(g)
    objs = objects(g)
    sym = symmetries(g)
    pr, pc = period(g)
    facts = [f"{name}: {g.shape[0]}x{g.shape[1]}, background {bg(g)}, colours "
             + ", ".join(f"{int(v)}:{int(n)}" for v, n in zip(*np.unique(g, return_counts=True)))]
    if colour is not None:   # the lines and how much of their colour they hold; whether they border panels is not a fact
        on = np.zeros(g.shape, bool)
        on[rs, :] = True
        on[:, cs] = True
        n_on, n_all = int((on & (g == colour)).sum()), int((g == colour).sum())
        facts.append(f"  full lines of colour {colour}: rows {rs} cols {cs} ("
                     + (f"all {n_all} of its cells" if n_on == n_all else f"{n_on} of its {n_all} cells")
                     + f"; panels(g) cuts along them into {len(panels(g))} panels)")
    facts.append(f"  {len(objs)} objects (4-connected, one colour), sizes " + str(sorted((o['size'] for o in objs),
                                                                                             reverse=True)[:12]))
    if sym:
        facts.append(f"  symmetric under {sym}")
    rows_rep, cols_rep = pr <= g.shape[0] // 2, pc <= g.shape[1] // 2   # at least two repeats that way
    if rows_rep and cols_rep:
        facts.append(f"  repeats as a tile of {pr}x{pc}")
    elif rows_rep:
        facts.append(f"  rows repeat every {pr}")
    elif cols_rep:
        facts.append(f"  columns repeat every {pc}")
    return facts


def summary(train=None, test=None):
    """Facts about every training pair and the test input(s), all computed exactly: sizes, colour counts, separators
    and panels, objects, symmetries, periods, and how each output relates to its input (identity, flip/rotation,
    sub-grid, scale, tiling, which colours change into which). Use it first to see the task, then check ideas in code."""
    train = NS["train"] if train is None else train
    test = NS["test_inputs"] if test is None else test
    lines = []
    for i, (x, y) in enumerate(train):
        x, y = _g(x), _g(y)
        lines.append(f"Example {i + 1}:")
        lines += ["  " + s for s in _grid_facts(x, "input")]
        lines += ["  " + s for s in _grid_facts(y, "output")]
        lines += ["  " + s for s in _relations(x, y)]
    sizes = {(_g(y).shape) for _, y in train}
    if len(sizes) == 1:
        lines.append(f"All training outputs are {next(iter(sizes))[0]}x{next(iter(sizes))[1]}.")
    if all(_g(x).shape == _g(y).shape for x, y in train):
        lines.append("Every training output has its input's size.")
    for i, t in enumerate(test):
        lines += _grid_facts(_g(t), f"Test input {i + 1}" if len(test) > 1 else "Test input")
    print("\n".join(lines))


# ----------------------------------------------------------------------------------------------------------- search


def try_all(fns):
    """Run each candidate transform (a function grid -> grid, or a dict name -> function) on every training input.
    Prints, per candidate, how many training examples it reproduces exactly and the first error; returns the names
    of the candidates that reproduce all of them. Lets you test many hypotheses in one call."""
    train = NS["train"]
    items = list(fns.items()) if isinstance(fns, dict) else [(getattr(f, "__name__", str(i)), f) for i, f in enumerate(fns)]
    good = []
    for name, f in items:
        ok, err = 0, None
        for x, y in train:
            try:
                got = _g(f(_g(x).copy()))
                ok += got.shape == _g(y).shape and bool((got == _g(y)).all())
            except Exception as e:  # noqa: BLE001
                err = err or f"{type(e).__name__}: {e}"
        print(f"{name}: {ok}/{len(train)}" + (f"  (error: {err})" if err else ""))
        if ok == len(train):
            good.append(name)
    return good


def _largest_object(g):
    """The largest multicolour object, cropped to its bbox, its outside cells in the background colour."""
    objs = objects(g, multicolor=True)
    if not objs:
        return None
    s = max(objs, key=lambda o: o["size"])["shape"]
    return np.where(s == -1, bg(g), s)


def _library():
    """Whole-task transforms with no free parameters, or parameters read from the training pairs."""
    lib = {f"d4 {n}": (lambda n: lambda g: d4(g, n))(n) for n in D4_NAMES[1:]}
    for k in (2, 3, 4):
        lib[f"scale {k}"] = (lambda k: lambda g: scale(g, k))(k)
        lib[f"downscale {k}"] = (lambda k: lambda g: downscale(g, k))(k)
    for r, c in ((1, 2), (2, 1), (2, 2), (3, 3)):
        lib[f"tile {r}x{c}"] = (lambda r, c: lambda g: tile(g, r, c))(r, c)
    lib["mirror right"] = lambda g: np.hstack([_g(g), np.fliplr(_g(g))])
    lib["mirror down"] = lambda g: np.vstack([_g(g), np.flipud(_g(g))])
    lib["mirror 2x2"] = lambda g: np.vstack([np.hstack([_g(g), np.fliplr(_g(g))]),
                                             np.hstack([np.flipud(_g(g)), np.rot90(_g(g), 2)])])
    lib["crop to content"] = lambda g: crop(g, bbox(g))
    lib["largest object"] = _largest_object
    for d in ("down", "up", "left", "right"):
        lib[f"gravity {d}"] = (lambda d: lambda g: gravity(g, d))(d)
    return lib


def try_library():
    """Try a bank of common whole-task transforms (flips and rotations, scaling, tiling, mirroring, cropping to the
    content or the largest object, gravity, filling enclosed holes, colour maps, period and symmetry repair) on every
    training pair. Prints the ones that reproduce all of them, with their output on the test input. Costs no thinking;
    worth one call at the start."""
    fits, n = _library_fits(NS["train"], NS["test_inputs"])
    if not fits:
        print(f"none of {n} library transforms reproduces every training example")
        return []
    for name, outs, err in fits:
        if err:
            print(f"{name}: reproduces every training example but fails on the test input: {err}")
        else:
            print(f"{name}: reproduces every training example; its test output(s): {outs}")
    return [name for name, _, _ in fits]


def _library_fits(train, tests):
    """[(name, test outputs, error)] for the library transforms that reproduce every training pair, and the number
    tried."""
    lib = _library()
    # transforms whose parameters come from the training pairs
    pairs = [(_g(x), _g(y)) for x, y in train]
    cmap = {}
    if all(x.shape == y.shape for x, y in pairs):
        ok = True
        for x, y in pairs:
            for a, b in zip(x.flatten(), y.flatten()):
                if cmap.setdefault(int(a), int(b)) != b:
                    ok = False
        if ok and any(a != b for a, b in cmap.items()):
            lib["colour map " + str(cmap)] = lambda g, m=dict(cmap): recolor(g, m)
        new = set().union(*[set(np.unique(y)) - set(np.unique(x)) for x, y in pairs])
        for colour in new:
            lib[f"fill enclosed holes with {colour}"] = (lambda c: lambda g: fill_holes(g, c))(int(colour))
        gone = set().union(*[set(np.unique(x)) - set(np.unique(y)) for x, y in pairs])
        for colour in gone:
            lib[f"repair colour {colour} by period"] = (lambda c: lambda g: fill_by_period(g, c))(int(colour))
            lib[f"repair colour {colour} by symmetry"] = (lambda c: lambda g: fill_by_symmetry(g, c))(int(colour))
    fits = []
    for name, f in lib.items():
        try:
            if not all((lambda got: got is not None and _g(got).shape == y.shape and (_g(got) == y).all())(f(x.copy()))
                       for x, y in pairs):
                continue
        except Exception:  # noqa: BLE001
            continue
        try:
            fits.append((name, [_g(f(_g(t).copy())).tolist() for t in tests], None))
        except Exception as e:  # noqa: BLE001
            fits.append((name, None, f"{type(e).__name__}: {e}"))
    return fits, len(lib)


# ----------------------------------------------------------------------------------------------------------- feedback


def object_diff(expected, got, bg=None):
    """What a wrong output gets wrong, object by object (same-size grids): objects of the expected output that yours
    misses, has in another place or another colour, and objects yours has that the expected output does not. Returns
    lines; [] when the grids are equal."""
    e, o = _g(expected), _g(got)
    if e.shape != o.shape:
        return [f"wrong size: expected {e.shape[0]}x{e.shape[1]}, yours {o.shape[0]}x{o.shape[1]}"]
    if (e == o).all():
        return []
    b = globals()["bg"](e) if bg is None else bg
    oe, og = objects(e, b), objects(o, b)
    free = list(range(len(og)))
    lines = []
    for x in oe:
        same = [j for j in free if og[j]["key"] == x["key"] and og[j]["bbox"] == x["bbox"]]
        if same:
            free.remove(same[0])
            continue
        moved = sorted((j for j in free if og[j]["key"] == x["key"]),   # the nearest copy of the same shape
                       key=lambda j: abs(og[j]["bbox"][0] - x["bbox"][0]) + abs(og[j]["bbox"][1] - x["bbox"][1]))
        recol = [j for j in free if og[j]["bbox"] == x["bbox"] and og[j]["size"] == x["size"]]
        if moved:
            free.remove(moved[0])
            lines.append(f"misplaced: colour {x['color']} size {x['size']} belongs at bbox {x['bbox']}, yours is at "
                         f"{og[moved[0]]['bbox']}")
        elif recol:
            free.remove(recol[0])
            lines.append(f"wrong colour: the object at bbox {x['bbox']} should be {x['color']}, yours is "
                         f"{og[recol[0]]['color']}")
        else:
            lines.append(f"missing: colour {x['color']} size {x['size']} at bbox {x['bbox']}")
    lines += [f"extra: colour {og[j]['color']} size {og[j]['size']} at bbox {og[j]['bbox']}" for j in free]
    return lines


# ----------------------------------------------------------------------------------------------------------- recipes

# How to investigate with the tools. They name no likely rules: the training examples decide the rule.
RECIPES = {
    "testing ideas": (
        "Write each idea about the rule as a small function and run them together with try_all([...]). Keep only "
        "those that reproduce every training example. If several do and their outputs on the test input differ, the "
        "examples leave the rule open: look for what in the examples decides between them."),
    "objects": (
        "Make a table: for every training pair, each input object (objects(x)) with what happened to it in the "
        "output (match(x, y)) and its measurable properties (size, colour, bbox, holes, key, position). A rule must "
        "explain every row of every pair; test it with try_all before submitting."),
    "panels": (
        "When separators(g) finds lines whose colour appears nowhere else (summary() then says 'all N of its cells'), "
        "split each grid with panels(g) and treat the panels as grids of their own: compare them with each other and "
        "with the output using exact checks (np.array_equal, colour counts, diff, symmetries). Lines whose colour also "
        "appears elsewhere may be part of a shape. Check every observation in code on all pairs before relying on it."),
    "hidden pattern": (
        "When symmetries(g, ignore=c) or period(g, ignore=c) finds more structure with one colour c ignored than "
        "without it, fill_by_symmetry(g, c) and fill_by_period(g, c) repaint c from that structure. Compare the "
        "result with each training output before treating it as the rule."),
    "output size": (
        "When output sizes differ from the inputs or from each other, list each pair's input size, output size and "
        "candidate quantities (bboxes and counts from objects(), panel sizes, colour counts) side by side, and find "
        "a quantity that matches in every pair. A wrong size can never be right, so settle it first."),
    "a failing submission": (
        "submit lists the failing examples with a cell diff and, object by object, what is missing, extra, misplaced "
        "or recoloured. Read which objects are wrong before changing the program; one wrong object usually points to "
        "one wrong step."),
}


def recipe(name=None):
    """How-to guides for investigating a task with the tools (they suggest no rule): recipe() lists them,
    recipe('name') prints one."""
    if name is None:
        print("Recipes: " + "; ".join(RECIPES) + ".\nrecipe('name') prints one.")
        return
    key = name.lower().strip()
    hits = [k for k in RECIPES if key in k]
    if not hits:
        print(f"no recipe {name!r}; recipe() lists them")
        return
    print(f"{hits[0]}:\n{RECIPES[hits[0]]}")


# ----------------------------------------------------------------------------------------------------------- help

SKILL_NAMES = ("summary", "try_library", "try_all", "objects", "match", "separators", "panels", "symmetries", "period",
               "fill_by_period", "fill_by_symmetry", "bg", "bbox", "crop", "find", "d4", "scale", "downscale", "tile",
               "recolor", "flood", "fill_holes", "gravity", "draw_line", "object_diff", "recipe", "help")

SKILLS_INDEX = """Skills (preloaded in `python`; each one is tested; call help('name') for its full description and an example):
- summary()                exact facts about every training pair and the test input: sizes, colours, separators and
                           panels, objects, symmetries, periods, and how each output relates to its input. Start here.
- try_library()            tries ~40 common whole-task transforms on the training pairs and shows any that fit.
- try_all(fns)             runs several of your candidate transforms on every training pair at once.
- objects(g, ...)          connected components: colour, cells, bbox, size, shape, holes, key (shape signature).
- match(a, b)              which objects of a stay, move, change colour, appear or disappear in b.
- separators(g), panels(g) full lines of one colour, and the panels they cut g into.
- symmetries(g, ignore)    which flips/rotations leave g unchanged; period(g, ignore): the smallest repeating tile.
- fill_by_period(g, c), fill_by_symmetry(g, c)   repaint the cells of colour c from the pattern.
- bg(g), bbox(g, colours), crop(g, box), find(g, sub)
- d4(g, name), scale(g, k), downscale(g, k), tile(g, r, c), recolor(g, {a: b}), flood(g, r, c, colour),
  fill_holes(g, colour), gravity(g, direction), draw_line(g, r0, c0, r1, c1, colour)
- object_diff(expected, got) what a wrong output gets wrong, object by object (submit uses it for failing examples).
- recipe()                 how-to guides for investigating with these tools (testing ideas, objects, panels, hidden
                           patterns, output size, a failing submission); recipe('name') prints one."""

_EXAMPLES = {
    "summary": "summary()   # prints e.g. 'Example 1: input: 10x10, background 0, colours 0:88, 4:12 ... output = flip_lr(input)'",
    "try_library": "found = try_library()   # e.g. ['mirror right'] and its test output",
    "try_all": ("def a(g): return np.fliplr(g)\ndef b(g): return np.flipud(g)\ntry_all([a, b])   # 'a: 3/3', 'b: 1/3' -> ['a']"),
    "objects": "for o in objects(train_inputs[0]): print(o['color'], o['size'], o['bbox'], o['holes'])",
    "match": "print('\\n'.join(match(train_inputs[0], train_outputs[0])))",
    "panels": "for r0, c0, p in panels(train_inputs[0]): show(p)",
    "symmetries": "symmetries(train_outputs[0])   # e.g. ['flip_lr', 'rot180']",
    "period": "period(test_inputs[0], ignore=0)   # e.g. (3, 4)",
}


def help(name=None):  # noqa: A001 - the model calls help('name')
    """help() lists the skills; help('name') prints one skill's full description and an example."""
    if name is None:
        print(SKILLS_INDEX)
        return
    f = globals().get(name)
    if not callable(f):
        print(f"no skill named {name!r}; help() lists them")
        return
    import inspect
    print(f"{name}{inspect.signature(f)}\n{inspect.getdoc(f)}")
    if name in _EXAMPLES:
        print("Example:\n" + _EXAMPLES[name])

Writing arc_skills.py


The parent side, and a check of the sandbox on its own before any model is involved, with the skills loaded as the harness loads them: a grid shown with indices, a wrong program's feedback, a 200,000-character and an 80,000,000-character print (both clipped), a call that writes text, a JSON object and a number to the worker's stdout behind its back, a submit by reference, and a runaway loop (timed out). It also draws one `show_image` picture: the worker can, though this notebook never tells the model about pictures.

In [28]:
import asyncio, contextlib, signal, sys

WORKER = os.path.abspath('arc_sandbox_worker.py')
TOOL_TIMEOUT = 60.0     # seconds per tool call
TOOL_OUT = 12000        # characters of tool output the model sees


def _limits():  # in the worker process: cap its memory so a runaway array cannot touch the server
    import resource
    resource.setrlimit(resource.RLIMIT_AS, (4 << 30, 4 << 30))


def clip(text, limit=TOOL_OUT):
    """Keep the head (2/3) and the tail (1/3): the tail holds the conclusion or the error."""
    if len(text) <= limit:
        return text
    head, tail = text[: limit * 2 // 3], text[-limit // 3:]
    return f'{head}\n... [{len(text) - len(head) - len(tail)} characters cut] ...\n{tail}'


class Sandbox:
    """One worker process per attempt; its Python state persists between the model's calls."""

    def __init__(self, task, timeout=TOOL_TIMEOUT, skills=False):
        self.task, self.timeout, self.proc, self.calls, self.timeouts = task, timeout, None, 0, 0
        self.pics = []   # the last call's show_image pictures (lists of grids)
        self.skills = skills   # the skills module's functions in the namespace (arc_skills.py)
        self.seq = 0     # request ids: the worker echoes each one, so only its own reply is taken as the answer

    async def start(self):
        self.proc = await asyncio.create_subprocess_exec(
            sys.executable, '-u', WORKER, stdin=asyncio.subprocess.PIPE, stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.DEVNULL, preexec_fn=_limits,
            start_new_session=True,   # its own process group: close() kills whatever the model's code started too
            limit=64 << 20)   # a reply is one JSON line, clipped only here (the probe: 64 KiB killed an attempt)
        return await self._ask({'op': 'init', 'task': self.task, 'skills': self.skills})

    async def _ask(self, req):
        self.seq += 1
        self.proc.stdin.write((json.dumps(dict(req, id=self.seq)) + '\n').encode())
        await self.proc.stdin.drain()
        deadline = time.time() + self.timeout
        while True:
            line = await asyncio.wait_for(self.proc.stdout.readline(), max(0.1, deadline - time.time()))
            if not line:
                raise RuntimeError('sandbox worker exited')
            try:
                r = json.loads(line)
            except ValueError:   # the model's code wrote to the worker's stdout itself: not a reply, so skip it
                continue
            if isinstance(r, dict) and r.get('id') == self.seq:
                return r         # anything else on that stream (a stray JSON line) is not this request's reply

    async def _call(self, op, code, **extra):
        self.calls += 1
        try:
            return await self._ask({'op': op, 'code': code, **extra})
        except (asyncio.TimeoutError, RuntimeError, ValueError, OSError) as e:
            # ValueError: a reply line past the limit; OSError: the worker died between calls (a broken pipe)
            self.timeouts += 1
            await self.close()
            await self.start()
            why = f'timed out after {self.timeout:.0f} s' if isinstance(e, asyncio.TimeoutError) else 'crashed'
            return {'error': f'[the code {why}; the Python state was reset: earlier variables and functions are '
                             f'gone, the task data is reloaded]', 'out': ''}

    async def run(self, code):
        """A `python` call: the text the model sees."""
        r = await self._call('exec', code)
        self.pics = r.get('pictures') or []
        out = r.get('out', '') + (r.get('error') or '')
        return clip(out) if out.strip() else '[no output]'

    async def submit(self, code):
        """A `submit` call: (text the model sees, result with n_ok / n / test_outputs, or None on an error)."""
        r = await self._call('submit', code)
        self.pics = r.get('pictures') or []
        if r.get('error'):
            return clip(r.get('out', '') + r['error']), None
        text = (r['out'].strip() + '\n\n' if r['out'].strip() else '') + r['feedback']
        return clip(text), r

    async def apply(self, inputs):
        """The harness's own call, which the model never sees: the session's current `transform` on `inputs` (a
        task's other test inputs), converted as `submit` converts them. A grid, or None, per input."""
        r = await self._call('apply', '', inputs=inputs)
        outs = r.get('outputs')
        return outs if isinstance(outs, list) and len(outs) == len(inputs) else [None] * len(inputs)

    async def close(self):
        """Kill the worker and everything in its process group, then wait at most 5 s: a process the model's code
        started keeps the worker's pipes open, and an unbounded wait would hang until it exits."""
        if self.proc and self.proc.returncode is None:
            with contextlib.suppress(AttributeError, OSError):   # no process groups (not Linux), or already gone
                os.killpg(self.proc.pid, signal.SIGKILL)
            with contextlib.suppress(ProcessLookupError):
                self.proc.kill()
            with contextlib.suppress(asyncio.TimeoutError):
                await asyncio.wait_for(self.proc.wait(), 5)


# the sandbox on its own, before any model: the first task of this run, a wrong program, a long print, an endless
# loop (a 5 s limit here, so the check is quick), and a call that writes to the worker's stdout behind its back
async def _sandbox_selftest():
    task = next(iter(todo.values()))
    sb = Sandbox({'train': task['train'], 'test': task['test'][:1]}, timeout=5, skills=True)
    await sb.start()
    print(await sb.run('show(train_inputs[0]); print(len(objects(train_inputs[0])), "objects")'))
    print(await sb.run('show_image(train_inputs[0], train_outputs[0])'), '->', len(sb.pics), 'picture(s)')
    pictured = len(sb.pics) == 1 and len(sb.pics[0]) == 2
    text, res = await sb.submit('def transform(grid):\n    return grid')
    print(text[:600], '| first failing example for a picture:', res['fail_pair'][0] if res and res['fail_pair'] else None)
    other = task['train'][0]['input']   # the accepted program on another input, as a sibling test input gets it
    applied = await sb.apply([other, [[1, 2], [3]]])
    print('apply:', 'the identity on another input' if applied[0] == other else applied[0], '| a ragged input:', applied[1])
    n = len(await sb.run('print("x" * 200000)'))
    print(n, 'characters back from a 200,000-character print')
    big = await sb.run('print("y" * 80_000_000)')   # past the reply-line limit unless the worker clips it
    print(len(big), 'characters back from an 80,000,000-character print:', big[-120:].strip())
    stray = await sb.run('import os; os.write(1, b"stray line\\n{\\"out\\": \\"stray\\", \\"id\\": 0}\\n7\\n"); '
                         'print("after the stray write")')   # plain text, a JSON object and a JSON number
    print(stray)
    by_ref, _ = await sb.submit('')   # submit by reference: the transform the earlier submit defined
    print(by_ref[:200])
    looped = await sb.run('while True: pass')
    print(looped)
    await sb.close()
    return (n <= TOOL_OUT + 100 and len(big) <= TOOL_OUT + 100 and 'characters cut' in big and 'crashed' not in big
            and 'after the stray write' in stray and 'does not define' not in by_ref and 'timed out' in looped
            and pictured and applied[0] == other)

try:   # a worker that cannot even start must fail the check, not the notebook (the scored rerun must reach its file)
    SANDBOX_OK = await _sandbox_selftest()
except Exception as e:  # noqa: BLE001
    print(f'the sandbox self-test raised {type(e).__name__}: {e}')
    SANDBOX_OK = False
check(SANDBOX_OK, 'the sandbox self-test failed')

    012345678901234567890123456789   (30x30)
  0 353366541499439999349941456633
  1 533366454199349119439914546633
  2 113554669114994554994119664553
  3 115345661941914444191491665435
  4 699935334399926996299934335399
  5 969953333491999669991943333599
  6 996911359944699229964499531196
  7 999611539154969999694519351169
  8 149143995572432442342755993419
  9 411934914527344224437254194391
 10 991499456455244334425546549941
 11 994191444545423443245454441914
 12 439999695977557227557795969999
 13 349129969577452772547759699219
 14 994469997759545555459577988884
 15 915496297795464554645977988885
 16 915496297795464554645977988885
 17 994469997759545555459577988884
 18 349129969577452772547759688889
 19 439999695977557227557795988889
 20 994191444545423443245454488884
 21 991499456455244334425546588881
 22 411934914527344224437254188881
 23 149143995572432442342755993419
 24 999611539154969999694519351169
 25 996911359944699229964499531196
 26 969953333491999669991943333599
 27 699935

**The harness's prompt, tools and notes.**
- **The prompt** is ARC Prize's task text, with the tools and the skills index where it says "Respond in the format of the training output examples". It ends with the attempt's token budget, and says that the best-scoring submission is kept as an answer if no program is accepted. The cell checks the task text against ARC Prize's format.
- **The tools:** `python` (state kept between calls) and `submit` (the full source of `transform(grid)`, or an empty `code` to grade the `transform` the `python` session already holds).
- **Notes the harness adds** (wording adapted from a public ARC-AGI-2 REPL agent, and from the failure study):
  - after six `python` calls without a submission: submitting is free, and the best submission is kept;
  - at 50, 75 and 90 % of the budget: how many tokens are left, and to submit the best candidate now;
  - after the first accepted program: look for a different rule that would change the test output, starting from the readings set aside and from what only the test input shows;
  - a second accepted program with the same test output but different code: it is not a second answer (once per attempt);
  - a reply with a tool call that cannot be read: resend it (at most twice per attempt);
  - a turn cut at the length limit, or a repetition loop removed: stop analysing and test the hypothesis in code;
  - the final answer turn: why the attempt ended, and how to give the answer.
- **Parsing:** the model's tool calls are Qwen3-Coder XML. The `<tool_call>` opener is optional, and so is a stray `<|im_start|>` before `function=`; a call whose `<parameter>` tag is missing or unclosed takes its body as the code. A turn whose text compresses below 1 % is a repetition loop.

In [29]:
import zlib
from collections import Counter

# ARC Prize's task text, with the tools in place of "Respond in the format of the training output examples"
HEAD = ("You are participating in a puzzle solving competition. You are an expert at solving puzzles.\n\n"
        "Below is a list of input and output pairs with a pattern. Your goal is to identify the pattern or "
        "transformation in the training examples that maps the input to the output, then apply that pattern "
        "to the test input to give a final output.\n\n")
EXAMPLES = ("--Training Examples--\n{training_examples}\n--End of Training Examples--\n\n"
            "--Test Input--\n{test_input}\n--End of Test Input--\n\n")
check(HEAD + "Respond in the format of the training output examples\n\n" + EXAMPLES + "Your response:" == PROMPT,
      "the harness's task text differs from ARC Prize's")


def task_parts(task, test_input, head, tail):
    """The ARC task text, as build_prompt writes it, between `head` and `tail`."""
    ex = ''.join(f'--Example {i}-- \n\n INPUT: \n\n{json.dumps(p["input"])}\n\nOUTPUT: \n\n{json.dumps(p["output"])}\n\n'
                 for i, p in enumerate(task['train']))
    return (head + '--Training Examples--\n' + ex + '\n--End of Training Examples--\n\n'
            '--Test Input--\n' + json.dumps(test_input) + '\n--End of Test Input--\n\n' + tail)


import contextlib, importlib
import arc_skills   # the skills module written above: its index goes into the prompt
importlib.reload(arc_skills)   # as section 9 last wrote it (a kernel keeps its first import otherwise)


def repl_tail(budget):
    """The tools part of the harness prompt: the two tools, the skills index, and the attempt's budget."""
    return (
        "You have two tools.\n"
        "- `python` runs Python. The task is preloaded as numpy int arrays: `train_inputs`, `train_outputs`, `train` (the "
        "list of (input, output) pairs) and `test_inputs` (the test input above). `show(g)` prints a grid with row and "
        "column indices and `diff(a, b)` lists the cells where two same-shape grids differ as (row, col, value in a, "
        "value in b).\n" + arc_skills.SKILLS_INDEX + "\n"
        "Calling `summary()` first gives you exact facts about the task to reason from; `help('name')` shows a "
        "skill's full description and an example. Variables and functions persist between calls.\n"
        "- `submit` takes the full source of `def transform(grid)` (or an empty `code`, to submit the `transform` your "
        "`python` calls defined last), runs it on every training input (a numpy int array, like `train_inputs[i]`), "
        "and tells you which examples it reproduces, with a cell-level diff for the ones it does not. When it "
        "reproduces all of them it is accepted, and its output on the test input is your final answer.\n\n"
        "Think as long as you need about what the rule could be. For anything that must be exact (reading, counting or "
        "indexing cells, checking a hypothesis against every training example), use `python`, where the grids are already "
        "loaded: exact work done by eye is where mistakes happen. Submit as soon as you have a candidate: a failing "
        "submission shows exactly which examples and cells are wrong, and if no program is accepted, your best-scoring "
        "submission is kept as an answer. If you cannot find a program that reproduces every example, finish with your "
        "best output grid for the test input in the format of the training output examples. "
        f"You have {budget:,} tokens for this puzzle over all your turns; the harness tells you as they run out.")


TOOLS = [
    {'type': 'function', 'function': {
        'name': 'python',
        'description': ('Run Python code in the task sandbox (numpy and the task grids are preloaded; state persists '
                        'between calls). Print what you need; the output is cut at about 12,000 characters.'),
        'parameters': {'type': 'object', 'properties': {'code': {'type': 'string', 'description': 'Python code'}},
                       'required': ['code']}}},
    {'type': 'function', 'function': {
        'name': 'submit',
        'description': ('Submit a complete program defining transform(grid) -> grid, where grid is a 2-D numpy int '
                        'array; an empty code submits the transform your python calls defined last. It is run on every '
                        'training input; you are told which examples it reproduces. When it reproduces all of them it '
                        'is accepted and its output on the test input is your answer.'),
        'parameters': {'type': 'object', 'properties': {
            'code': {'type': 'string', 'description': 'Full Python source defining transform(grid), or empty.'}},
            'required': ['code']}}},
]

# Notes the harness adds in the conversation (wording from a public ARC-AGI-2 REPL agent, adapted, and from the
# study of submission 6's harness failures, docs/harness_flaws/)
THINKCAP_HINT = ('[harness] Your last turn was cut off before any tool call. Stop analysing. In your next reply, call '
                 '`python` to test your current best hypothesis in code, or `submit` a transform implementing it.')
NUDGE = ('\n\n[harness] {n} python calls since your last submission. Submitting is free and returns a cell-level diff '
         'for up to two failing examples (every example is checked), and if no program is accepted, your best-scoring submission is kept as an answer: a '
         'candidate transform is worth submitting even before it is finished.')
BUDGET_NOTE = ('\n\n[harness] Budget: {used:,} of the {total:,} tokens for this puzzle are used, {left:,} are left. If you '
               'have a candidate `transform`, submit it now: its diff shows what is still wrong, and if no program is '
               'accepted, your best-scoring submission is kept as an answer. If you have none yet, submit one for the '
               'part of the rule you are sure of.')
REVIEW_NOTE = ('\n\n[harness] Accepted: this program reproduces every training example, and its output on the test input (below) is your first answer. You also get a second answer. Training examples often leave the rule open: look for a different rule that also fits every training example but would change the test output (another reading of what is kept, counted, copied or recoloured, another tie-break, another order). Start from the readings you considered and set aside while solving, and from what the test input shows that no training input does. If you find one, submit its program: if it reproduces every training example and its test output differs, it becomes your second answer and the attempt ends. If you are confident no other rule fits, submit the same program again to finish.\nTest input: {ih}x{iw}, colours {icol}. Your program\'s output on it: {oh}x{ow}, colours {ocol}:\n{grid}')
SAME_NOTE = ('\n\n[harness] This program also reproduces every training example, but its output on the test input is the '
             'same as your first answer, so it is not a second answer. If it was meant as a different rule, that rule '
             'does not change the test output: look for one that does. If you are confident no other rule fits, submit '
             'your first program again to finish.')
MALFORMED_NOTE = ('[harness] Your reply has a tool call the harness could not read, so nothing ran. Resend it in the '
                  'tool-call format, with `<function=NAME>`, `<parameter=code>` and their closing tags.')
COMMIT = ('[harness] {why} Give your final answer now, in this reply: the output grid for the test input as a JSON list '
          'of rows, or one `submit` call with your best `transform` (its test output becomes your answer), or one '
          '`python` call that prints the grid with `print(grid.tolist())`.')
COMMIT_WHY = {'spent': 'The budget for this puzzle is used up.',
              'context': 'The conversation is too long to go on.',
              'turns': 'The turn limit for this puzzle is reached.',
              'ended': 'Your last reply had no tool call and no valid output grid.',
              'error': 'The harness hit an error and cannot continue this puzzle.',
              'time': 'The time for this puzzle is up.'}


def review_note(test_input, grid):
    """The accepted program's test output, shown back as a grid with indices and colour counts."""
    colours = lambda g: dict(sorted(Counter(v for r in g for v in r).items()))
    rows = '\n'.join(f'{r:>3} ' + ''.join(map(str, row)) for r, row in enumerate(grid))
    head = '    ' + ''.join(str(c % 10) for c in range(len(grid[0])))
    return REVIEW_NOTE.format(ih=len(test_input), iw=len(test_input[0]), icol=colours(test_input), oh=len(grid),
                              ow=len(grid[0]), ocol=colours(grid), grid=head + '\n' + rows)


# Tool calls, tolerantly: the <tool_call> opener is optional (one held-out attempt wrote <function=...> alone and was
# taken as finished), and so is a stray <|im_start|> before function= (E8's 269e22fb test 1 wrote
# <|im_start|>function=python>, was taken as finished, and lost the answer its code computes)
CALL_RE = re.compile(r'(?:<tool_call>\s*(?:<\|im_start\|>|<)?|<\|im_start\|>|<)function=([\w.-]+)>(.*?)</function>'
                     r'(?:\s*</tool_call>)?', re.DOTALL)
PARAM_RE = re.compile(r'<parameter=([\w.-]+)>\n?(.*?)\n?</parameter>', re.DOTALL)
CALL_MARK = re.compile(r'<tool_call>|function=[\w.-]+>')   # a reply that tried to call a tool


def xml_tool_calls(content):
    """The model's Qwen3-Coder XML tool calls, and the text around them. A call without a closed <parameter> tag takes
    its body as the code."""
    calls = []
    for i, m in enumerate(CALL_RE.finditer(content)):
        params = dict(PARAM_RE.findall(m.group(2)))
        if not params:
            body = re.sub(r'^\s*<parameter=[\w.-]+>\n?', '', m.group(2)).strip()
            params = {'code': body} if body else {}
        calls.append({'id': f'xml_{i}', 'type': 'function',
                      'function': {'name': m.group(1), 'arguments': json.dumps(params)}})
    return calls, CALL_RE.sub('', content).strip()


def call_code(c):
    """A parsed call's tool name and code (its one parameter, whatever the model named it)."""
    try:
        args = json.loads(c['function']['arguments'] or '{}')
    except json.JSONDecodeError:
        args = {}
    code = args.get('code', next(iter(args.values())) if len(args) == 1 else '')
    return c['function']['name'], code if isinstance(code, str) else ''


def degenerate(text):
    """A repetition loop: zlib squeezes it below 1 % (prose is ~20 %, an ASCII board ~1.6 %)."""
    b = text.encode()
    return len(b) >= 500 and len(zlib.compress(b)) / len(b) < 0.01


# the harness prompt's task text is ARC Prize's, to the character
first_task = next(iter(todo.values()))
x0 = first_task['test'][0]['input']
check(task_parts(first_task, x0, HEAD, '') == build_prompt(first_task, x0, HEAD + EXAMPLES),
      "the harness prompt's task text differs from ARC Prize's format")
print(len(task_parts(first_task, x0, HEAD, repl_tail(harness_tokens))), 'characters in a harness prompt (the first task)')

18049 characters in a harness prompt (the first task)


**A harness conversation is an SGLang streaming session.** The server keeps its KV cache and recurrent state between turns, so a turn sends, and prefills, only its new tokens: the tool replies, then the next assistant header. They are written exactly as the chat template writes them, and this cell checks that against the server's own rendering of turn 1. It also renders a turn 1 without the tool list, for the picture look (section 9).

A turn cut at the budget continues in its own session when the server ends turns exactly (`EXACT_TURNS`, section 5). The next turn sends only the turn's close and the new message. Measured at a near-certain next token, the continued state sits at the floor of a state built by prefill alone. Without it, DFlash2's last verify step could commit up to 7 tokens past the cut. A session trims those from the KV cache but not from the GDN state, so a cut turn was continued in a new session with the whole history prefilled again (150–180k tokens at a spent budget, a ~2-min stall for every other conversation).

**The final answer turn** is sent with the thinking block already closed (`<think>\n\n</think>`, as the template writes a turn without thinking), so the whole reply is the answer. The probe's harness split that reply at a `</think>` that never comes, took all of it for reasoning, and so never read a final turn.

Without a server (the scored rerun after a failed start), this cell skips the template and the solve below is skipped too, so the notebook still reaches its fallback file; a server that fails during the template check is only noted there. The template is split at the last `ping`, the user message, so a `ping` anywhere in the system or tools text cannot shift it.

In [30]:
import json, zlib
CONTEXT = 262144   # the server's context length; a conversation finishes before it outgrows it
MAX_TURNS = 200    # only a guard against runaway loops; the budget and the context are the real limits
TPREFIX = TSUFFIX = None   # turn 1 of a harness conversation, rendered once by the server (set_repl_template)
LOOK_TEMPLATE = None   # the picture look's turn 1: the same render without the tool list (set_repl_template)
# Every /generate request of this notebook holds one of these: more than SLOTS requests in flight crash this server
# (the DFLASH draft sampler's buffers are sized to the largest decode CUDA graph, 24 here; dflash_worker_v2.py:844).
INFLIGHT = asyncio.Semaphore(SLOTS)
import aiohttp, random


def healthy():
    try:
        return requests.get(f'{SERVER}/health', timeout=5).status_code == 200
    except requests.RequestException:
        return False


class ServerGuard:
    """Recovers from a server that dies mid-round (E5 lost 7 attempts to a vision-tower OOM). The first conversation
    whose request fails at the connection restarts it (restart_server, section 5); the others wait. At most
    `max_restarts` per kernel. `epoch` counts restarts: a session opened before the last one is gone, so its
    conversation rebuilds from the history it keeps here."""

    def __init__(self, max_restarts=3):
        self.epoch, self.restarts, self.max_restarts = 0, 0, max_restarts
        self.lock, self.calm_until = asyncio.Lock(), 0.0

    async def recover(self, epoch, why):
        async with self.lock:
            if self.epoch != epoch or await asyncio.to_thread(healthy):   # restarted meanwhile, or only a dropped link
                return
            if self.restarts >= self.max_restarts:
                raise RuntimeError(f'the server died again ({why}) after {self.restarts} restarts')
            self.restarts += 1
            t = time.time()
            print(f"{time.strftime('%H:%M:%S')} [guard] the server is down ({why}): restart {self.restarts}", flush=True)
            if not await asyncio.to_thread(restart_server):
                raise RuntimeError('the server did not come back after a restart')
            print(f"{time.strftime('%H:%M:%S')} [guard] server back after {time.time() - t:.0f} s", flush=True)
            self.epoch += 1
            self.calm_until = time.time() + 60   # rebuilds with pictures spread over a minute: one ViT batch each


GUARD = ServerGuard()


def ids(text):
    return TOK.encode(text, add_special_tokens=False).ids


def set_repl_template():
    """The chat template's turn 1 with the tools: the system turn (effort line + tool list) and the assistant header
    around the user's prompt, rendered by the server itself. True if local tokenization gives the server's ids."""
    global TPREFIX, TSUFFIX, LOOK_TEMPLATE
    r = requests.post(f'{SERVER}/v1/chat/completions', timeout=300, json={
        'model': 'qwen3.8-27b', 'messages': [{'role': 'user', 'content': 'ping'}], 'tools': TOOLS, 'max_tokens': 1,
        'chat_template_kwargs': {'reasoning_effort': 'xhigh', 'preserve_thinking': True},
        'return_input_ids_in_sglext': True}).json()
    TPREFIX, TSUFFIX = TOK.decode(r['sglext']['input_ids'], skip_special_tokens=False).rsplit('ping', 1)
    ok = ids(TPREFIX + 'ping' + TSUFFIX) == r['sglext']['input_ids'] and 'Reasoning effort is set to xhigh' in TPREFIX
    check(ok, 'the harness template: local tokenization differs from the server, or the xhigh line is missing')
    # the picture look's turn 1 without the tool list: offered tools, a look called python in ~60 % of draws and
    # wrote no description (look probe 1026_2040: 34 of 34 empty looks; none of 65 without tools)
    try:
        r = requests.post(f'{SERVER}/v1/chat/completions', timeout=300, json={
            'model': 'qwen3.8-27b', 'messages': [{'role': 'user', 'content': 'ping'}], 'max_tokens': 1,
            'chat_template_kwargs': {'reasoning_effort': 'xhigh', 'preserve_thinking': True},
            'return_input_ids_in_sglext': True}).json()
        pre, suf = TOK.decode(r['sglext']['input_ids'], skip_special_tokens=False).rsplit('ping', 1)
        if (ids(pre + 'ping' + suf) == r['sglext']['input_ids'] and 'Reasoning effort is set to xhigh' in pre
                and '<tools>' not in pre):
            LOOK_TEMPLATE = (pre, suf)
    except (requests.RequestException, KeyError, TypeError, ValueError):   # the look keeps the tools template
        LOOK_TEMPLATE = None
    return ok


RS_HOMES = {}   # agi2 request split: a task's home (0/1), alternating as tasks first appear


def rs_home(task):
    """The task's home for the server's QSA request split, sent as extra_key on every turn of its conversations."""
    k = zlib.crc32(json.dumps(task['train'], sort_keys=True).encode())
    if k not in RS_HOMES:
        RS_HOMES[k] = len(RS_HOMES) % 2
    return RS_HOMES[k]


class SessionConv:
    """A harness conversation as an SGLang streaming session: the server keeps its KV and recurrent state between
    turns, so a turn sends, and prefills, only its new tokens (the tool replies or a note, then the next assistant
    header), written exactly as the chat template writes them. The history is the prompt plus exactly the ids the
    model wrote.

    Pictures: the history holds one placeholder per picture (the server expands it), and `pics` every picture in
    placeholder order. A request carries the pictures whose placeholders it sends: the new ones on a normal turn, all of
    them when a new session gets the whole history."""

    def __init__(self, http, prompt, pics=(), template=None, fork_at=None):
        self.http, self.sid, self.hist, self.queue = http, None, [], []
        self.fork_at = fork_at   # agi2 session fork: the first turn's shared-prefix length (None: no fork)
        self.home = 0   # the task's home (rs_home), set by the attempt that opens it
        pre, suf = template or (TPREFIX, TSUFFIX)   # another turn 1's (prefix, suffix): the picture look's
        self.first = ids(pre + prompt.strip() + suf)
        self.close_with, self.out_at, self.rebuild = '', 0, False
        self.parks, self.parked_s = 0, 0.0   # how often the KV gate (below) parked it, and for how long
        self.pics, self.sent = [uri for uri, _ in pics], 0   # every picture, and how many the session already holds
        self.pic_tokens = sum(n for _, n in pics)            # visual tokens the placeholders stand for (KV gate)
        self.epoch = GUARD.epoch                             # the server its session lives on (ServerGuard)

    async def _post(self, path, body):
        async with self.http.post(f'{SERVER}{path}', json=body) as r:
            return await (r.json() if path != '/close_session' else r.text())

    def _pending(self, thinking):
        """What the template writes after the model's last turn: its end if it was cut, the queued tool replies (one
        user turn) and notes, then the next assistant header."""
        s, in_tools = self.close_with, False
        for kind, text in self.queue:
            if kind == 'tool':   # a picture's placeholder is part of its note's text (PICTURE_NOTE, FAIL_NOTE, IMG)
                s += ('' if in_tools else '\n<|im_start|>user') + f'\n<tool_response>\n{text.strip()}\n</tool_response>'
                in_tools = True
            else:
                s += ('<|im_end|>' if in_tools else '') + f'\n<|im_start|>user\n{text.strip()}<|im_end|>'
                in_tools = False
        s += ('<|im_end|>' if in_tools else '') + '\n<|im_start|>assistant\n<think>\n' + ('' if thinking else '\n</think>\n\n')
        self.queue, self.close_with = [], ''
        return ids(s)

    async def turn(self, max_tokens, thinking=True):
        if self.sid is not None and self.epoch != GUARD.epoch:   # its session died with the server (ServerGuard)
            self.sid, self.rebuild = None, True
        if GATE is not None:
            await GATE.before_turn(self)   # may park it first (host RAM, or closed and rebuilt from its history later)
        kept = (self.close_with, list(self.queue))   # put back if this turn fails, so nothing queued for it is lost
        send = self.first if not self.hist else self._pending(thinking)
        base = self.hist   # what the session holds before this turn: the history changes only when a turn succeeds
        try:
            for left in (2, 1, 0):   # retries: after a dead server's restart, or over a pooled connection it had closed
                try:
                    if self.sid is not None and self.epoch != GUARD.epoch:   # restarted while it waited (gate, a slot)
                        self.sid, self.rebuild = None, True
                    if self.rebuild:   # cut, a removed loop, closed by the gate, a restarted server, a failed turn: a
                        if self.sid is not None:   # new session gets the whole history again, once
                            await self.close()
                        send, base, self.rebuild = base + send, [], False
                        if self.pics and time.time() < GUARD.calm_until:   # after a restart: one picture batch at a time
                            await asyncio.sleep(random.uniform(0, GUARD.calm_until - time.time()))
                    if self.sid is None and FORK and self.fork_at and not base and send is self.first \
                            and len(send) > self.fork_at > 0:
                        self.sid = await FORKS.open_fork(self.http, list(send[:self.fork_at]),
                                                         getattr(self, 'home', None))
                        if self.sid is not None:   # a new session, as the open below: it holds no pictures
                            self.sent, self.epoch = 0, GUARD.epoch
                    if self.sid is None:
                        self.sid = await self._post('/open_session', {'capacity_of_str_len': CONTEXT, 'streaming': True})
                        if not isinstance(self.sid, str):
                            raise RuntimeError(f'open_session failed: {str(self.sid)[:200]}')
                        self.sent, self.epoch = 0, GUARD.epoch   # a new session holds no pictures yet: all go now
                    body = {'input_ids': send, 'session_params': {'id': self.sid}, 'extra_key': f'agi2h{self.home}',
                            'sampling_params': {**SAMPLING, 'max_new_tokens': max_tokens, 'no_stop_trim': True,
                                                'skip_special_tokens': False, 'spaces_between_special_tokens': False}}
                    if len(self.pics) > self.sent:
                        body['image_data'] = self.pics[self.sent:]
                    async with INFLIGHT:
                        data = await self._post('/generate', body)
                    if left and isinstance(data, dict) and 'session' in str(data.get('error', '')):   # it lost it
                        self.sid, self.rebuild = None, True
                        continue
                    break
                except aiohttp.ClientConnectionError as e:   # the server went away (or the link dropped) mid-turn
                    if not left:
                        raise
                    await GUARD.recover(self.epoch, type(e).__name__)
                    if self.epoch != GUARD.epoch:             # restarted: its session is gone, rebuild from the history
                        self.sid, self.rebuild = None, True   # kept here (a live server gets the same request again)
            if not isinstance(data, dict) or 'meta_info' not in data:
                raise RuntimeError(f'server error: {str(data)[:300]}')
            fin = data['meta_info'].get('finish_reason') or {}
            if fin.get('type') == 'abort':   # the server dropped the turn (and the session's KV with it)
                raise RuntimeError(f"turn aborted: {str(fin.get('message'))[:300]}")
        except BaseException:
            # The turn did not happen: what was queued for it waits for the next turn, which sends the whole history,
            # as it stood before this one, to a new session (this session's state is unknown). Before this, a failed
            # rebuild lost the history, and the attempt's next turn started again from its first prompt.
            self.close_with, self.queue = kept
            self.rebuild = True
            raise
        mi = data['meta_info']
        self.hist = base + send
        self.sent = len(self.pics)
        self.out_at = len(self.hist)
        self.hist += data['output_ids']
        stopped = fin.get('type') == 'stop'
        text = data['text']
        for eos in ('<|im_end|>', '<|endoftext|>'):
            text = text.removesuffix(eos)
        if thinking:
            reasoning, content = text.split('</think>', 1) if '</think>' in text else (text, '')
        else:   # the thinking block was closed before the reply: all of it is the answer (submission 6's harness
            reasoning, content = '', text   # split it at a </think> that never comes, and so never read a final turn)
        if not stopped:   # cut at max_tokens: the next turn first closes what is still open, in this session when
            self.close_with = ('' if '</think>' in text or not thinking else '\n</think>\n\n') + '<|im_end|>'
            self.rebuild = not EXACT_TURNS   # the server holds it exactly at the cut (section 5), else in a new one
        calls, rest = xml_tool_calls(content) if CALL_MARK.search(content) else ([], content)
        return dict(content=rest.strip(), reasoning=reasoning.strip(), calls=calls,
                    finish='length' if not stopped else 'tool_calls' if calls else 'stop',
                    prompt=mi.get('prompt_tokens', 0), tokens=mi.get('completion_tokens', 0),
                    cached=mi.get('cached_tokens') or 0,
                    bare=bool(calls) and '<tool_call>' not in content)   # a call written without its opener

    def drop_notes(self):
        """Forget the harness notes queued for the next turn (the tool replies stay: every call needs its reply)."""
        self.queue = [q for q in self.queue if q[0] == 'tool']

    def drop_loop(self):
        self.hist = self.hist[:self.out_at] + ids('\n</think>\n\n[a repetition loop, removed by the harness]<|im_end|>')
        self.close_with, self.rebuild = '', True

    def tool(self, call, out, pics=()):
        """A tool reply; `pics` are (uri, tokens) whose placeholders `out` already contains, in order."""
        self.queue.append(('tool', out))
        self.pics += [uri for uri, _ in pics]
        self.pic_tokens += sum(n for _, n in pics)

    def user(self, text):
        self.queue.append(('user', text))

    async def park(self):
        """Move this idle session's KV, recurrent state and draft ring to host RAM (session park, section 4); its next
        turn, to the same session id, copies them back. False if the server refuses (host budget) or lacks it."""
        try:
            res = await self._post('/park_session', {'session_id': self.sid})
        except Exception:  # noqa: BLE001 - a refusal or an old server: the caller closes as before
            return False
        return isinstance(res, dict) and bool(res.get('success'))

    async def close(self):
        if self.sid is not None:
            await self._post('/close_session', {'session_id': self.sid})
            self.sid = None


HARNESS_OK = False
if SERVER_UP:   # without a server the scored rerun skips the solve and ends with the fallback file, so nothing may raise
    TOK = Tokenizer.from_file(f'{MODEL_DIR}/tokenizer.json')
    try:
        HARNESS_OK = set_repl_template() and SANDBOX_OK
    except (requests.RequestException, KeyError, TypeError, ValueError) as e:   # the server went away, or a bad answer
        check(False, f'the harness template could not be rendered: {type(e).__name__}: {e}')
    if TPREFIX is not None:
        print('turn 1 template:', len(ids(TPREFIX)), '+', len(ids(TSUFFIX)), 'tokens around the prompt')
        print('picture look template:', f'{len(ids(LOOK_TEMPLATE[0]))} + {len(ids(LOOK_TEMPLATE[1]))} tokens, '
              'without the tool list' if LOOK_TEMPLATE else 'the tools template (the toolless render did not check)')

import contextlib, time, uuid

FORK = True         # agi2 session fork (needs the server's SGLANG_AGI2_SESSION_FORK=1): attempts share a prefill
FORK_MAX = 48        # templates kept (parked on the server, ~0.1 GB host a GPU each), least recently used dropped first
FORK_MARGIN = 4      # tokens cut before the tools tail, so a token merging across that boundary stays out
FORK_TTL = 7200      # s: the server reaps a template idle this long (a backup to the LRU and to closes)
FORK_STATS = {'templates': 0, 'forks': 0, 'template_fail': 0, 'template_boundary': 0, 'evicted': 0}


class ForkTemplates:
    """Parked template sessions by their prefix ids (server: SGLANG_AGI2_SESSION_FORK)."""

    def __init__(self):
        self.by_key, self.locks, self.epoch, self.n = {}, {}, None, 0
        self.pending = set()   # cleanup tasks, kept referenced until done

    async def _post(self, http, path, body):
        async with http.post(f'{SERVER}{path}', json=body) as r:
            return await (r.json() if path != '/close_session' else r.text())

    def _close_later(self, http, sid):
        """Close a session without waiting (survives the caller's cancellation); errors are ignored."""
        async def go():
            with contextlib.suppress(Exception):
                await self._post(http, '/close_session', {'session_id': sid})
        task = asyncio.get_running_loop().create_task(go())
        self.pending.add(task)
        task.add_done_callback(self.pending.discard)

    async def _make(self, http, prefix, home, key):
        """Prefill `prefix` once in a template session and park it; (tsid, prefix), or None. A template that fails
        anywhere after its open (an error, an aborted turn, a park that does not hold exactly the prefix, a
        cancellation) is closed."""
        tsid = f'agi2.{key[0]:08x}.tmpl.{uuid.uuid4().hex[:8]}'
        owned = True   # an explicit id: a lost open response still gets a close (closing an unknown id is harmless)
        try:
            r = await self._post(http, '/open_session', {'capacity_of_str_len': CONTEXT, 'streaming': True,
                                                         'session_id': tsid, 'timeout': FORK_TTL})
            if r != tsid:
                FORK_STATS['template_fail'] += 1
                return None
            body = {'input_ids': prefix, 'session_params': {'id': tsid},
                    'sampling_params': {'max_new_tokens': 1, 'temperature': 0.0}}
            if home is not None:
                body['extra_key'] = f'agi2h{home}'
            async with INFLIGHT:
                d = await self._post(http, '/generate', body)
            fin = ((d.get('meta_info') or {}).get('finish_reason') or {}) if isinstance(d, dict) else {}
            if not isinstance(d, dict) or 'meta_info' not in d or fin.get('type') == 'abort':
                FORK_STATS['template_fail'] += 1
                return None
            p = await self._post(http, '/park_session', {'session_id': tsid})
            tokens = int(p.get('tokens') or 0) if isinstance(p, dict) else -1
            if not (isinstance(p, dict) and p.get('success') and tokens == len(prefix)):
                FORK_STATS['template_boundary' if tokens == len(prefix) + 1 else 'template_fail'] += 1
                return None
            owned = False   # the caller registers it
            return tsid, prefix, time.time()
        finally:
            if owned:
                self._close_later(http, tsid)

    async def open_fork(self, http, prefix, home=None):
        """A new session id forked from the template of `prefix` (made now if missing), or None: open as before.
        `home`: the task's home under the QSA request split (rs_harness); a template and its forks share it, so the
        restored KV sits in the buffers that home reads."""
        if self.epoch != GUARD.epoch:   # a restarted server lost every template
            self.by_key, self.locks, self.epoch = {}, {}, GUARD.epoch
        epoch = GUARD.epoch
        key = (zlib.crc32(bytes(str(prefix), 'ascii')), home)
        lock = self.locks.setdefault(key, asyncio.Lock())
        async with lock:
            if self.epoch != GUARD.epoch:   # restarted while this caller waited for the lock
                self.by_key, self.epoch = {}, GUARD.epoch
            epoch = GUARD.epoch
            ent = self.by_key.get(key)
            if ent is not None and time.time() - ent[2] > 0.9 * FORK_TTL:   # the server may reap it soon
                self.by_key.pop(key)
                self._close_later(http, ent[0])
                ent = None
            if ent is not None and ent[1] != prefix:   # a CRC collision: the displaced template goes
                self.by_key.pop(key)
                self._close_later(http, ent[0])
                ent = None
            if ent is None:
                ent = await self._make(http, prefix, home, key)
                if ent is None:
                    return None
                if GUARD.epoch != epoch or self.epoch != epoch:   # restarted meanwhile: it may live on the new server
                    self._close_later(http, ent[0])
                    return None
                self.by_key[key] = ent
                FORK_STATS['templates'] += 1
                while len(self.by_key) > FORK_MAX:   # least recently used first (dicts keep insertion order)
                    old = next(iter(self.by_key))
                    old_tsid = self.by_key.pop(old)[0]
                    FORK_STATS['evicted'] += 1
                    self._close_later(http, old_tsid)
            else:
                self.by_key[key] = self.by_key.pop(key)   # most recently used
            self.n += 1
            sid = f'{ent[0]}.fork.{self.n}'
            try:
                r = await self._post(http, '/open_session', {'capacity_of_str_len': CONTEXT, 'streaming': True,
                                                             'session_id': sid})
            except BaseException:
                self._close_later(http, sid)
                raise
            if r != sid:
                self._close_later(http, sid)
                return None
            FORK_STATS['forks'] += 1
            return sid


FORKS = ForkTemplates()


turn 1 template: 497 + 7 tokens around the prompt
picture look template: 45 + 7 tokens, without the tool list


**Keeping the KV pool from filling.** A session holds its whole conversation in the KV pool, also while it waits on a tool. Harness conversations grow to 80–250k tokens, so only ~6–10 of them fit a ~1.0M-token pool at once; the block tier's index (section 5, 6.0x here: ~3.47M slots a GPU)
holds about four times that, with the cold parts in host RAM, and this gate still guards it. In the probe, 18 conversations filled the pool in 24 min and stock SGLang could not recover (round 3). The fixes of section 5 let the server survive that; this gate keeps it from happening.

**How the gate works:**
- Every 2 s it asks the server how full the pool is (`/v1/loads`, counted by the scheduler when asked). An answer that has not come within 30 s is skipped, and any failed poll is counted and retried: the gate never stops polling.
- **No latch:** between answers the gate counts its own admissions, parks and returns. If the server has not answered for 2 min, it drops those counts and goes by the last answer, so a dead poll cannot stop admissions for good.
- **Parking:** above `GATE_HIGH` (90 %), a conversation about to send its next turn is parked first: its session moves to host RAM (session park), which frees the pool at once; if the server refuses (host budget), the session is closed instead.
- **Coming back:** a parked conversation returns as soon as its history fits under `GATE_HIGH - GATE_ROOM` (85 %) again, the oldest first; a restore from host RAM takes ~0.2 s, a closed session rebuilds from its exact history (10 s apart).
- **New attempts:** a harness attempt starts only when nothing parked is waiting and its first turn fits under the same mark.

In the probe's rounds 4–6 the gate parked 8–30 conversations per round; all of them came back, and the server never stalled. Submissions 6–8 ran their 11 h 35 min behind it at 82 %. At 90 % (`1020-kv-gate-probe`, 75 min of submission 8's workload) the pool ran at a median 83 % and peaked at 93 %, with no retraction; a retraction would free that conversation's session and re-read its history (section 5): a stall of a minute or two for the others, not a wedged server.

In [31]:
import aiohttp


class KVGate:
    """Keeps session histories from filling the KV pool (above). `usage` is the pool's use as the server's scheduler
    counts it when asked (`/v1/loads`; the `token_usage` gauge in /metrics is the max over the KV pool and the mamba
    slots, refreshed only with a log line), plus what was parked or brought back since that answer.

    Sizes, not fixed marks, decide who comes back (the probe's round 4: with fixed marks at 60 % and 65 %, parked
    conversations and the queue waited while the pool sat at 64-79 % and the batch drained from 18 to 5). A parked
    conversation returns as soon as its history fits under `high - room`, the oldest first; a new attempt starts when
    nothing parked is waiting and its first turn (about `first` of the pool) fits there too. Each admission is counted
    at once (round 4: ten queued attempts started in one poll and the server spent a minute only prefilling)."""

    def __init__(self, high=0.82, room=0.05, first=0.02, every=2.0, gap=10.0):
        self.high, self.room, self.first, self.every, self.gap = high, room, first, every, gap
        self.usage, self.pool, self.retracted = 0.0, None, 0
        self.waiting, self.parks, self.next_back = [], 0, 0.0   # waiting: parked conversations, oldest first
        self.host_parks = 0   # parks the server kept in host RAM (the rest were closed, to be re-read)
        self.answered, self.failed = time.time(), 0   # when the server last answered a poll; polls without an answer
        self.polled = 0.0   # the pool's use in the server's last answer, before this gate's own estimates

    @property
    def parked(self):
        return len(self.waiting)

    async def poll(self, http):
        while True:
            try:   # an answer that does not come within 30 s is skipped, so the gate never waits on a stuck request
                async with http.get(f'{SERVER}/v1/loads', timeout=aiohttp.ClientTimeout(total=30)) as r:
                    load = (await r.json())['loads'][0]
                pool = load['max_total_num_tokens']
                self.retracted = (load.get('queues') or {}).get('retracted', 0)
                self.polled = self.usage = load['num_used_tokens'] / pool
                self.pool, self.answered = pool, time.time()
            except Exception:  # noqa: BLE001 - whatever went wrong (a timeout too), the next poll asks again
                self.failed += 1
            await asyncio.sleep(self.every)

    def fits(self, share):
        if time.time() - self.answered > 120:   # no answer for 2 min: drop this gate's own estimates and go by the
            self.usage = self.polled            # last answer, so admissions cannot freeze on counts nothing corrects
        return self.usage + share <= self.high - self.room

    async def admit_attempt(self):
        while self.waiting or not self.fits(self.first):
            await asyncio.sleep(self.every)
        self.usage += self.first   # counted now, so workers waiting on the same poll do not all start at once

    async def before_turn(self, conv):
        if conv.sid is None or not self.pool or self.usage < self.high:
            return
        hosted = await conv.park()   # host RAM (session park): the next turn, same session id, restores it in ~0.2 s
        if hosted:
            self.host_parks += 1
        else:                   # refused or unavailable: close, and the next turn re-reads the whole history
            with contextlib.suppress(aiohttp.ClientConnectionError):   # a dead server took the session with it
                await conv.close()  # between turns nothing is in flight, so the server frees the whole session at once
            conv.sid, conv.rebuild = None, True
        share = (len(conv.hist) + conv.pic_tokens) / self.pool   # a picture's placeholder stands for its visual tokens
        self.usage -= share  # until the next answer from the server: the session's KV leaves the pool either way
        conv.parks, t = conv.parks + 1, time.time()
        me = object()
        self.waiting.append(me)
        self.parks += 1
        try:
            while self.waiting[0] is not me or not self.fits(share) or (not hosted and time.time() < self.next_back):
                await asyncio.sleep(self.every)
            self.usage += share   # its history comes back
            if not hosted:   # a re-read is a long prefill, so the next re-read waits `gap`; E6's restores took 0.2 s
                self.next_back = time.time() + self.gap
        finally:
            self.waiting.remove(me)
            conv.parked_s += time.time() - t


GATE = KVGate(high=GATE_HIGH, room=GATE_ROOM) if SESSIONS else None
print('KV gate:', {k: v for k, v in vars(GATE).items() if k in ('high', 'room', 'first', 'gap')} if GATE else 'off')

KV gate: {'high': 0.9, 'room': 0.05, 'first': 0.02, 'gap': 10.0}


**A first look at the pictures** (`VISION_LOOK`). Before its tool loop, every attempt opens a separate conversation that sees only the task as pictures, with no digits. It describes what it sees in at most 300 words: the objects, what stays and what changes, and two or three candidate rules. That description goes after the task in the attempt's own prompt, marked as possibly wrong ("the digits are exact"). The look's tokens come out of the attempt's 163,840.
- **The pictures:** ruled, with row and column numbers in the margins and a heavier line every 5 cells. One picture per training pair (input left, output right) and one for the test input.
- **The look turn:** capped at 12,288 tokens, and its turn 1 has no tool list. Offered the attempt's tools, a look called python in ~60 % of draws and wrote no description (look probe 1026_2040: 34 of 34 empty looks, none of 65 without tools). If it still writes nothing, it gets one 1,536-token answer turn with thinking off.
- **Why this way:** of the ways of showing the model pictures tried on 10-22, only this one helped, and it helped in each of three paired samples on one server. Pictures in the main prompt, a picture tool, pictures of failing outputs, and a pictured check all did not. Over the three samples, against submission 10's attempt on the same outputs, it fixed 26 and broke 13 (sign test p 0.053): 25 against 20 and 26 against 21 of 42 outputs on set v2b1, and 29 against 26 of 48 on set v2b2 (`experiments/10/22/1022_1326_…`). Pictures beside the digits hurt the model's cell-by-cell reading in the perception bench; a look with no digits keeps the two apart.
- **Attempts 2-4** take a look of their own, as in that measurement (with `TAIL_VIEW = 'transpose'` it would see the task
  transposed; here the view is plain).

In [32]:
%%writefile arc_vision.py
"""Pictures of ARC grids for Qwen3.8's vision tower (the vision plan, docs/reports/1022_0010_vision_harness_plan/).

Standalone (numpy + Pillow only), so a notebook cell can write it as `arc_vision.py` beside `arc_skills.py`.

The model sees an image as a grid of visual tokens of 32 x 32 pixels (16-pixel patches merged 2 x 2). A cell drawn at a
multiple of 32 pixels therefore covers whole tokens, at its own row and column of the model's 2-D positions: 32 px is
one token per cell (the 10-17 probe), 64 px four, 96 px nine. 16 px puts four cells in one token.

What a picture can carry besides the colours (all optional, per `Style`):
- `lines`: a thin dark line on each cell's top and left edge (10-17: without lines, same-colour neighbours merge);
- `major`: a heavier line every N cells, for counting;
- `rulers`: row and column indices in a one-cell margin on the top and left, so a position can be read off the picture
  and matched to the digits (a crop keeps the indices of the full grid);
- `digits`: the cell's digit drawn inside it;
- `marks`: cells outlined in a colour (e.g. the cells where two grids differ) and boxes (e.g. objects).

`tokens()` gives the number of visual tokens the image processor will make (Qwen2-VL's smart resize: sides rounded to
multiples of 32, the area kept within 65,536 to 16,777,216 pixels); `render()` raises the cell size itself when a
picture would be small enough to be enlarged, so cells stay on token boundaries.
"""

from __future__ import annotations

import base64
import io
import math
from dataclasses import dataclass, replace

import numpy as np
from PIL import Image, ImageDraw, ImageFont

# ARC's usual colours for 0-9, in the order the prompt's colour line names them
PALETTE = np.array([(0, 0, 0), (0, 116, 217), (255, 65, 54), (46, 204, 64), (255, 220, 0), (170, 170, 170),
                    (240, 18, 190), (255, 133, 27), (127, 219, 255), (135, 12, 37)], np.uint8)
COLOURS = '0 black, 1 blue, 2 red, 3 green, 4 yellow, 5 grey, 6 magenta, 7 orange, 8 light blue, 9 maroon'
DARK = {0, 1, 2, 6, 9}         # colours a digit is written on in white; the others get black
TOKEN_PX = 32                  # one visual token covers 32 x 32 pixels
MIN_PIXELS, MAX_PIXELS = 65536, 16777216   # Qwen3.8's preprocessor_config.json: shortest_edge / longest_edge (area)
WHITE, LINE, MAJOR, RULER_BG, RULER_FG = (255, 255, 255), (64, 64, 64), (24, 24, 24), (245, 245, 245), (40, 40, 40)


@dataclass(frozen=True)
class Style:
    cell: int = 32         # pixels per cell; a multiple of 32 keeps every cell on whole visual tokens
    lines: bool = True     # thin cell lines
    major: int = 0         # a heavier line every `major` cells (0: none)
    rulers: bool = False   # row / column indices in a one-cell margin on the top and left
    digits: bool = False   # each cell's digit drawn in it
    gap: int = 1           # blank cells between grids shown side by side


STYLES = {   # the perception bench's renderings (plan section 3)
    'c32': Style(32),
    'c64': Style(64),
    'c96': Style(96),
    'c32r': Style(32, major=5, rulers=True),
    'c64r': Style(64, major=5, rulers=True),
    'c64d': Style(64, digits=True),
    'c64rd': Style(64, major=5, rulers=True, digits=True),
}


def smart_resize(h: int, w: int, factor: int = TOKEN_PX, min_pixels: int = MIN_PIXELS,
                 max_pixels: int = MAX_PIXELS) -> tuple[int, int]:
    """The size the image processor resizes an h x w image to (Qwen2-VL's smart_resize)."""
    hb, wb = max(factor, round(h / factor) * factor), max(factor, round(w / factor) * factor)
    if hb * wb > max_pixels:
        beta = math.sqrt(h * w / max_pixels)
        hb, wb = max(factor, math.floor(h / beta / factor) * factor), max(factor, math.floor(w / beta / factor) * factor)
    elif hb * wb < min_pixels:
        beta = math.sqrt(min_pixels / (h * w))
        hb, wb = math.ceil(h * beta / factor) * factor, math.ceil(w * beta / factor) * factor
    return hb, wb


def tokens(img: np.ndarray) -> int:
    """Visual tokens the processor makes from an RGB array."""
    hb, wb = smart_resize(img.shape[0], img.shape[1])
    return (hb // TOKEN_PX) * (wb // TOKEN_PX)


def _font(px: int):
    try:
        return ImageFont.load_default(size=max(8, px))
    except TypeError:   # Pillow < 10.1 has no scalable default font
        return ImageFont.load_default()


def _text(draw: ImageDraw.ImageDraw, xy: tuple[float, float], s: str, size: int, fill) -> None:
    """`s` centred on `xy`."""
    font = _font(size)
    x0, y0, x1, y1 = draw.textbbox((0, 0), s, font=font)
    draw.text((xy[0] - (x1 - x0) / 2 - x0, xy[1] - (y1 - y0) / 2 - y0), s, font=font, fill=fill)


def _fit_cell(style: Style, cells_h: int, cells_w: int) -> Style:
    """Keep the picture inside the processor's area limits without resizing: raise the cell size (in steps of 32 px)
    while the picture would be enlarged, lower it while it would be shrunk. Either resize would move cells off token
    boundaries."""
    cell = style.cell
    while cells_h * cells_w * cell * cell < MIN_PIXELS:
        cell += TOKEN_PX if cell >= TOKEN_PX else cell
    while cell > TOKEN_PX and cells_h * cells_w * cell * cell > MAX_PIXELS:
        cell -= TOKEN_PX
    return replace(style, cell=cell) if cell != style.cell else style


def render(*grids, style: Style = Style(), origin: list[tuple[int, int]] | None = None,
           marks: list[dict] | None = None) -> np.ndarray:
    """Grids side by side (top-aligned, `style.gap` blank cells apart) as an RGB uint8 array whose sides are multiples
    of 32 pixels (padded white), so every cell sits on whole visual tokens.

    origin: per grid, the (row, column) its top-left cell has in a larger grid (a crop), for the rulers.
    marks: per grid, {'cells': {(r, c), ...}, 'colour': (r, g, b), 'boxes': [(r0, c0, r1, c1, (r, g, b)), ...]};
    cells are outlined, boxes drawn around inclusive cell ranges.
    """
    gs = [np.clip(np.asarray(g, dtype=np.int64), 0, 9) for g in grids]
    if not gs or any(g.ndim != 2 or g.size == 0 for g in gs):
        raise ValueError('render needs non-empty 2-D grids')
    margin = 1 if style.rulers else 0
    rows = max(g.shape[0] for g in gs) + margin
    cols = sum(g.shape[1] for g in gs) + style.gap * (len(gs) - 1) + margin
    style = _fit_cell(style, rows, cols)
    cell = style.cell
    H, W = rows * cell, cols * cell
    canvas = np.full((math.ceil(H / TOKEN_PX) * TOKEN_PX, math.ceil(W / TOKEN_PX) * TOKEN_PX, 3), WHITE, np.uint8)
    thin, heavy = max(1, cell // 32), max(2, cell // 16)
    placed = []   # (grid, y0, x0) in pixels
    x = margin
    for g in gs:
        y0, x0 = margin * cell, x * cell
        h, w = g.shape
        px = PALETTE[g].repeat(cell, 0).repeat(cell, 1)
        if style.lines:
            for k in range(thin):
                px[k::cell, :] = LINE
                px[:, k::cell] = LINE
        canvas[y0:y0 + h * cell, x0:x0 + w * cell] = px
        placed.append((g, y0, x0))
        x += w + style.gap
    img = Image.fromarray(canvas)
    draw = ImageDraw.Draw(img)
    for i, (g, y0, x0) in enumerate(placed):
        h, w = g.shape
        r_off, c_off = origin[i] if origin else (0, 0)
        if style.major:
            for r in range(0, h + 1):
                if (r + r_off) % style.major == 0:
                    draw.rectangle([x0, y0 + r * cell - heavy // 2, x0 + w * cell - 1, y0 + r * cell + (heavy - 1) // 2],
                                   fill=MAJOR)
            for c in range(0, w + 1):
                if (c + c_off) % style.major == 0:
                    draw.rectangle([x0 + c * cell - heavy // 2, y0, x0 + c * cell + (heavy - 1) // 2, y0 + h * cell - 1],
                                   fill=MAJOR)
        if style.digits:
            for r in range(h):
                for c in range(w):
                    v = int(g[r, c])
                    _text(draw, (x0 + (c + 0.5) * cell, y0 + (r + 0.5) * cell), str(v), int(cell * 0.55),
                          (255, 255, 255) if v in DARK else (0, 0, 0))
        if style.rulers:
            draw.rectangle([x0, 0, x0 + w * cell - 1, cell - 1], fill=RULER_BG)
            for c in range(w):
                _text(draw, (x0 + (c + 0.5) * cell, cell / 2), str(c + c_off), int(cell * 0.42), RULER_FG)
            if i == 0:
                draw.rectangle([0, y0, cell - 1, y0 + h * cell - 1], fill=RULER_BG)
                for r in range(h):
                    _text(draw, (cell / 2, y0 + (r + 0.5) * cell), str(r + r_off), int(cell * 0.42), RULER_FG)
        m = (marks or [None] * len(placed))[i] if marks else None
        if m:
            colour, wide = tuple(m.get('colour', (255, 0, 255))), max(2, cell // 10)
            for r, c in sorted(m.get('cells', ())):
                draw.rectangle([x0 + c * cell, y0 + r * cell, x0 + (c + 1) * cell - 1, y0 + (r + 1) * cell - 1],
                               outline=colour, width=wide)
            for r0, c0, r1, c1, col in m.get('boxes', ()):
                draw.rectangle([x0 + c0 * cell, y0 + r0 * cell, x0 + (c1 + 1) * cell - 1, y0 + (r1 + 1) * cell - 1],
                               outline=tuple(col), width=wide)
    return np.asarray(img)


def diff_cells(a, b) -> set[tuple[int, int]]:
    """Cells where two same-shape grids differ (empty set when the shapes differ: show both instead)."""
    a, b = np.asarray(a), np.asarray(b)
    if a.shape != b.shape:
        return set()
    return {(int(r), int(c)) for r, c in zip(*np.nonzero(a != b))}


def crop(grid, r0: int, c0: int, r1: int, c1: int):
    """Rows r0..r1 and columns c0..c1 (inclusive, clipped) and their origin, for render(..., origin=[...])."""
    g = np.asarray(grid)
    r0, c0 = max(0, r0), max(0, c0)
    r1, c1 = min(g.shape[0] - 1, r1), min(g.shape[1] - 1, c1)
    if r0 > r1 or c0 > c1:
        raise ValueError('empty crop')
    return g[r0:r1 + 1, c0:c1 + 1].tolist(), (r0, c0)


def data_uri(img: np.ndarray) -> str:
    """The array as a PNG data URI (what SGLang's OpenAI-style image_url and /generate image_data accept)."""
    buf = io.BytesIO()
    Image.fromarray(img).save(buf, 'PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()


def picture(*grids, style: Style = Style(), **kw) -> tuple[str, int]:
    """render() as (PNG data URI, visual tokens)."""
    img = render(*grids, style=style, **kw)
    return data_uri(img), tokens(img)

Writing arc_vision.py


In [33]:
import arc_vision as av   # written above: the grids drawn as pictures (numpy and Pillow, on the CPU)

IMG = '<|vision_start|><|image_pad|><|vision_end|>'   # a picture's placeholder in the prompt; the server expands it


def picture(*grids, style='c32r'):
    """Grids side by side as (PNG data URI, visual tokens), ruled: row and column numbers in the margins and a heavier
    line every 5 cells."""
    return av.picture(*grids, style=av.STYLES[style])


PIC_GUIDE = ('Each grid is drawn with its row and column numbers in the margins and a heavier line every 5 cells; '
             'the colours are ' + av.COLOURS + '. The digits above are exact; the pictures show shapes, objects and '
             'layout.')
GESTALT_TASK = ('Here are the training examples of a puzzle as pictures (each input left, its output right) and its test '
                'input. ' + PIC_GUIDE.split('. The digits above')[0] + '. Do not solve it in full. Look carefully and '
                'describe, concretely (positions, counts, shapes, colours): the objects in the inputs; what stays the '
                'same and what changes from each input to its output; and two or three different candidate rules that '
                'could explain every example. Keep it under 300 words.\n')
GESTALT_NOTE = ('\n\n[harness] A first look at the pictures, written by a separate pass that saw only the pictures, not '
                'the digits (it may be wrong; the digits are exact):\n{desc}')


async def picture_look(http, task, test_input, rec):
    """A fresh conversation that sees only the pictures (no digits) and describes the task: returns (text, tokens);
    what it did goes into `rec`, which the attempt merges into its own record. Any failure gives no text."""
    pics, lines = [], []
    for i, p in enumerate(task['train']):
        pics.append(picture(p['input'], p['output']))
        lines.append(f'Example {i}: {IMG}')
    pics.append(picture(test_input))
    lines.append(f'Test input: {IMG}')
    conv = SessionConv(http, GESTALT_TASK + '\n'.join(lines), pics, template=LOOK_TEMPLATE)   # no tool list
    conv.home = rs_home(task)
    desc, used = '', 0
    rec['gestalt_toolless'] = LOOK_TEMPLATE is not None
    try:
        g = await conv.turn(VISION_TOKENS)
        used = g['tokens']
        desc = (g['content'] or '').strip()[:3000]
        if not desc:   # a look can spend its whole turn thinking: one answer turn with thinking off
            conv.user('Write your description now, in at most 300 words, from what you have seen.')
            g2 = await conv.turn(1536, thinking=False)
            used += g2['tokens']
            desc = (g2['content'] or '').strip()[:3000]
            rec['gestalt_forced'] = True
    except Exception as e:  # noqa: BLE001 - the attempt goes on without the look
        rec['gestalt_error'] = f'{type(e).__name__}: {e}'[:200]
    with contextlib.suppress(Exception):
        await asyncio.wait_for(conv.close(), 30)
    rec['gestalt_tokens'] = used
    rec['gestalt'] = desc[:1500]
    return desc, used


if VISION_LOOK:   # a check that shows its result: one task's pictures, drawn here, with their visual tokens
    _t = next(iter(todo.values()))
    _p = [picture(p['input'], p['output']) for p in _t['train']] + [picture(_t['test'][0]['input'])]
    print(f'picture look on: {len(_p)} pictures for the first task, {sum(n for _, n in _p):,} visual tokens, '
          f'{sum(len(u) for u, _ in _p) // 1024:,} KB of PNG; the look turn is capped at {VISION_TOKENS:,} tokens')
else:
    print('picture look off: submission 10\'s attempt')

picture look on: 5 pictures for the first task, 5,549 visual tokens, 145 KB of PNG; the look turn is capped at 12,288 tokens


**One harness attempt** (`repl_attempt`):
- **No per-turn limit.** The model works with `python` and `submit`, its reasoning kept between turns. The limits are the attempt's budget (163,840 tokens) and the 262k context.
- **Acceptance:** a program that reproduces every training example is accepted, and its output on the test input is recorded at once. The harness also runs it, unseen by the model, on the task's other test inputs (`Sandbox.apply`, as `submit` would hand them over), and those outputs answer them (`SHARE`).
  - (First attempts only: attempts 2-4 end at their first accepted program, 10-30.) The model is then asked for a different rule that also fits every training example and would change the test output, because training examples often leave the rule open. If it finds one, that rule's output is the second answer (3 right second answers in the probe's 64 evaluation attempts).
  - A second accepted program with the same test output but different code gets one note that it is not a second answer; the attempt goes on.
- **Endings:**
  - the second accepted submission (a different rule, or the same program again);
  - a turn with no tool call: its grid is the answer (JSON, or rows written out);
  - the budget, the context or the turn limit (200) running out;
  - an error in the working loop (a server error, an aborted turn): recorded, and the attempt goes on to its final answer turn.
- **The final answer turn:** an attempt that ends with no accepted program and no grid gets one more reply, with thinking off (up to 8,192 tokens, outside the budget). Queued working notes are dropped, so the last word before it is the request for the answer. Its reply may be:
  - a grid: the answer (JSON, or rows written out);
  - a `submit` call: its test output is the answer, and if it reproduces every example it counts as accepted;
  - a `python` call: the harness runs it, and the last grid it printed is the answer (JSON on one line, `show(g)`, a numpy print, or one row per line; a label before it on the line is ignored). When every training pair fixes the output's size (the input's size, or one size for all), only a grid of that size counts, so a printed diagnostic is not taken for the answer.
- **What is kept:** the answer (the accepted program's output, else the final grid, else the best-scoring submission's output), then the second rule's output, the grid written after an acceptance, or the best-scoring submission's. The best submission is also written to `submission.json` the moment it improves, so an attempt cancelled at the stop keeps it.
- **Loops:** a repetition loop is cut out of the history.

**What earlier rounds measured** (this harness without this notebook's fixes):
- the probe's round 4 (33 evaluation outputs, one attempt each): 24 programs accepted, 13 of them right; a median 32–43 min and ~76k tokens per attempt;
- the 2026-10-18 bench on Modal 4 × L4 (32 outputs, one attempt each, no time limit): 24 solved, 2 budgets spent.

In [34]:
import ast


def spent(g):
    return {k: g[k] for k in ('prompt', 'tokens', 'cached')}


def same_code(a, b):
    """Two programs that differ only in layout or comments."""
    try:
        return ast.dump(ast.parse(a)) == ast.dump(ast.parse(b))
    except SyntaxError:
        return a.strip() == b.strip()


SHOW_HEAD = re.compile(r'^ {4}\d+ +\((\d+)x(\d+)\)$')   # show(g)'s header: column digits, then (HxW)
SHOW_ROW = re.compile(r'^ *\d+ (\d+)$')                 # show(g)'s rows: the row index, then the row's digits


def _row(line):
    """One printed row of a grid, and whether it was printed in brackets: its digits ('0120'), or single digits in
    brackets or between spaces or commas."""
    s = line.strip()
    bracketed = s.startswith('[') and s.endswith(']')
    s = s.strip('[]').replace(',', ' ')
    if not s or not re.fullmatch(r'[0-9 ]+', s):
        return None
    toks = s.split()
    if len(toks) == 1:
        toks = list(toks[0])
    elif any(len(t) > 1 for t in toks):
        return None
    return [int(t) for t in toks], bracketed


def expected_shape(task, test_input):
    """The test output's size when every training pair fixes it the same way (each output has its input's size, or
    all outputs have one size); None when the pairs do not."""
    size = lambda g: (len(g), len(g[0]))
    pairs = [(size(p['input']), size(p['output'])) for p in task['train']]
    if all(i == o for i, o in pairs):
        return size(test_input)
    return pairs[0][1] if len({o for _, o in pairs}) == 1 else None


def printed_grid(text, shape=None):
    """The last grid a python call printed: JSON on one line (`print(g.tolist())`), show(g)'s indexed rows, or one row
    per line. Rows printed bare must number at least two and be at least 2 wide (one number per line is a count, not a
    grid); rows in brackets (`print(row)`) may be one wide, and one bracketed row on the last line is a 1-row grid.
    With `shape`, only grids of that size count (the final turn: a diagnostic print such as a 3x3 pattern is not the
    answer to a task whose outputs are all 14x14). None if it printed none."""
    # a label before a printed grid ("Answer: [[1 2]" or "out = [[1, 2], [3, 4]]") is cut, so the grid's first row counts
    lines = [l[l.index('[['):] if '[[' in l else l for l in text.splitlines()]
    found, run = [], []   # found: (last line, 1 for JSON or show(g) else 0, grid)
    last = max([i for i, l in enumerate(lines) if l.strip()], default=-1)   # the last line with text
    alone = True   # the current run did not start right after another row (a ragged grid's last row is no grid)

    def flush(end):
        rows = [cells for cells, _ in run]
        if (len(rows) >= 2 and len(rows[0]) >= 2) or (rows and all(b for _, b in run)
                                                        and (len(rows) >= 2 or (end == last and alone))):
            found.append((end, 0, rows))

    for i, line in enumerate(lines):
        if line.strip().startswith('[['):
            with contextlib.suppress(ValueError):
                found.append((i, 1, json.loads(line.strip())))
        m = SHOW_HEAD.match(line)
        if m:
            h, w = int(m.group(1)), int(m.group(2))
            rows = [SHOW_ROW.match(l) for l in lines[i + 1:i + 1 + h]]
            if len(rows) == h and all(rows) and all(len(r.group(1)) == w for r in rows):
                found.append((i + h, 1, [[int(c) for c in r.group(1)] for r in rows]))
        r = _row(line)
        if r is not None and run and len(r[0]) == len(run[-1][0]) and r[1] == run[-1][1]:
            run.append(r)
            continue
        flush(i - 1)
        alone = not run
        run = [r] if r is not None else []
    flush(len(lines) - 1)
    found = [f for f in found if kaggle_grid(f[2]) and (shape is None or (len(f[2]), len(f[2][0])) == shape)]
    return max(found, key=lambda f: (f[0], f[1]))[2] if found else None


def text_grid(text):
    """The grid a reply writes: JSON (parse_grid), else rows written out as printed_grid reads them. None if neither."""
    g = parse_grid(text) if text.strip() else None
    return g if g is not None and kaggle_grid(g) else printed_grid(text)


def frugal_skips(acc_tokens):
    """Graded frugal (10-27): at level 2 every review goes; at level 1 only the reviews of acceptances made at
    <= TAIL_PLAUS_AT tokens (look included); at 0 every review runs. FRUGAL is set by the solve's pacer."""
    return FRUGAL[0] >= 2 or (FRUGAL[0] >= 1 and acc_tokens <= TAIL_PLAUS_AT)


def fork_cut(task, test_input):
    """agi2 session fork: the first turn's ids before the tools tail, less FORK_MARGIN (the same for every
    attempt of this task and test input)."""
    return max(0, len(ids(TPREFIX + task_parts(task, test_input, HEAD, '').strip())) - FORK_MARGIN)


async def repl_attempt(http, task, test_input, budget, live=None, on_accept=None, on_best=None, nudge_every=6,
                       siblings=(), on_shared=None, stop_after_accept=None):
    """One harness attempt: the tool loop on this attempt's sandbox, in a streaming session (above). `on_accept(grid)`
    gets every accepted program's test output at once, and `on_best(grid)` every better-scoring failing submission's;
    `live` names a file that gets one JSON line per turn (its tool calls, their code and what they returned).
    `on_shared(grids)` gets each accepted program's outputs on `siblings`, the task's other test inputs.
    `stop_after_accept()`, when given, ends the attempt at its next turn once a program is accepted and it returns
    true (a second attempt whose review could not change its pair)."""
    left, ctx, t0 = budget, 0, time.time()   # ctx: tokens the conversation holds (prompt + output, plus tool replies)
    marks = sorted(BUDGET_AT)                # fractions of the budget still to announce

    def log(**kw):
        if live:
            with open(live, 'a') as f:
                f.write(json.dumps(dict(kw, s=round(time.time() - t0))) + '\n')

    sb = Sandbox({'train': task['train'], 'test': [{'input': test_input}]}, skills=True)
    look = {}   # the picture look's record, merged into the attempt's
    if VISION_LOOK:
        desc, used = await picture_look(http, task, test_input, look)
        left -= used
        log(turn='look', tokens=used, forced=bool(look.get('gestalt_forced')), chars=len(desc),
            error=look.get('gestalt_error'))
        conv = SessionConv(http, task_parts(task, test_input, HEAD, repl_tail(budget - used)
                                            + (GESTALT_NOTE.format(desc=desc) if desc else '')),
                           fork_at=fork_cut(task, test_input))
        conv.home = rs_home(task)
    else:
        conv = SessionConv(http, task_parts(task, test_input, HEAD, repl_tail(budget)),
                           fork_at=fork_cut(task, test_input))
        conv.home = rs_home(task)
    rec = dict(requests=[], turns=0, texts=[], python_calls=0, submits=[], cut_turns=0, loops=0, committed=False,
               calls_in_reasoning=0, bare_calls=0, malformed=0, budget_notes=0, same_noted=False, final_from=None)
    rec.update(look)
    since_submit, accepted, best, final, pending = 0, None, None, None, None   # pending: accepted, asked for another rule
    why = 'ended'   # why the tool loop ended, for the final answer turn
    try:
        try:   # the working loop; an error in it (a server error, an aborted turn) still leads to the final answer turn
            await sb.start()
            while True:
                if rec['turns'] >= MAX_TURNS:
                    why = 'turns'
                    break
                if pending is not None and (frugal_skips(rec.get('first_accept_tokens') or 0)
                                            or (stop_after_accept is not None and stop_after_accept())):
                    break   # behind schedule, or a second attempt whose review cannot change its pair: the accepted program stands
                if left <= 512:
                    why = 'spent'
                    break
                if time.time() >= WRAP_AT:   # the run's end is near: no more thinking, the answer now
                    why = 'time'
                    break
                room = CONTEXT - ctx - 1024          # how far the conversation can still grow
                if room < 4096:
                    rec['context_full'], why = True, 'context'
                    log(turn=rec['turns'] + 1, note=f'context nearly full ({ctx:,} tokens): finishing')
                    break
                g = await conv.turn(min(left, room, clock_cap()))
                rec['turns'] += 1
                rec['requests'].append(spent(g))
                left -= g['tokens']
                ctx = g['prompt'] + g['tokens']
                calls, content = g['calls'], g['content']
                if not calls and not content.strip() and CALL_MARK.search(g['reasoning']):
                    calls, _ = xml_tool_calls(g['reasoning'])   # written inside <think>: still a call
                    rec['calls_in_reasoning'] += bool(calls)
                rec['bare_calls'] += bool(g.get('bare'))
                rec['texts'].append(dict(g, content=content, calls=calls))
                turn = dict(turn=rec['turns'], tokens=g['tokens'], prompt=g['prompt'], finish=g['finish'],
                            think_chars=len(g['reasoning']), content=content[:1500])
                note = ''   # the budget note for the reply to this turn: once per mark passed (the latest if several)
                while marks and budget - left >= marks[0] * budget:
                    marks.pop(0)
                    note = BUDGET_NOTE.format(used=budget - left, total=budget, left=max(left, 0))
                rec['budget_notes'] += bool(note)
                if degenerate(g['reasoning'] + content):
                    rec['loops'] += 1
                    log(**turn, note='repetition loop, removed')
                    conv.drop_loop()
                    conv.user(THINKCAP_HINT + note)
                    continue
                if calls:
                    seen = []
                    for k, c in enumerate(calls):
                        name, code = call_code(c)
                        if name == 'python':
                            rec['python_calls'] += 1
                            since_submit += 1
                            out = await sb.run(code)
                            if since_submit % nudge_every == 0:
                                out += NUDGE.format(n=since_submit)
                        elif name == 'submit':
                            since_submit = 0
                            out, res = await sb.submit(code)
                            sub = dict(n_ok=res['n_ok'], n=res['n'], test=(res['test_outputs'] or [None])[0]) if res \
                                else dict(n_ok=-1, n=len(task['train']), test=None)
                            rec['submits'].append(sub)
                            if res and res['n_ok'] == res['n'] and kaggle_grid(sub['test']):
                                if on_accept:
                                    on_accept(sub['test'])
                                if siblings and on_shared:   # the same program on the task's other test inputs
                                    on_shared(await sb.apply(list(siblings)))
                                if pending is None:   # the first acceptance: the first answer; ask for another rule
                                    pending, rec['first_accept'] = dict(sub, code=code), sub['test']
                                    rec['first_accept_tokens'] = budget - left   # sub 13: tokens at the first acceptance (the look included)
                                    if frugal_skips(budget - left):   # graded frugal: early reviews go first
                                        accepted = sub
                                    else:
                                        out += review_note(test_input, sub['test'])
                                elif (sub['test'] == pending['test'] and not same_code(code, pending['code'])
                                      and not rec['same_noted']):   # meant as another rule, with the same test output
                                    rec['same_noted'] = True
                                    out += SAME_NOTE
                                else:   # a different rule, or the same program again: the attempt ends
                                    accepted = sub
                            elif res and kaggle_grid(sub['test']) and (best is None or sub['n_ok'] > best['n_ok']):
                                best = sub   # the best-scoring failing submission: kept as an answer if nothing better
                                if on_best:
                                    on_best(sub['test'])
                        else:
                            out = f'[harness] There is no tool {name!r}; the tools are `python` and `submit`.'
                        if k == len(calls) - 1:
                            out += note
                        conv.tool(c, out)
                        ctx += len(out) + 16   # grids print about one token per character
                        seen.append(dict(name=name, code=code[:3000], out=out[:2500]))
                    log(**turn, calls=seen)
                    if accepted:
                        break
                    continue
                if g['finish'] == 'length':
                    rec['cut_turns'] += 1
                    log(**turn, note='cut at the budget')
                    conv.user(THINKCAP_HINT + note)
                    continue
                if CALL_MARK.search(content) and rec['malformed'] < 2:   # it tried to call a tool; none could be read
                    rec['malformed'] += 1
                    log(**turn, note='a tool call the harness could not read')
                    conv.user(MALFORMED_NOTE + note)
                    continue
                log(**turn, note='ended without a tool call')
                final = text_grid(content)
                break
        except Exception as e:  # noqa: BLE001 - the stop's CancelledError is not an Exception: it still ends the attempt
            rec['error'], why = f'{type(e).__name__}: {e}'[:300], 'error'
            log(turn=rec['turns'] + 1, note=f'error: {rec["error"]}')
        rec['spent'] = left <= 512 or bool(rec.get('context_full'))
        if accepted is None and pending is None and final is None:
            # The final answer turn, thinking off: a grid, or a submit or python call that builds it, run here. Notes
            # queued for a next working turn are dropped: the last word before it is the request for the answer.
            conv.drop_notes()
            conv.user(COMMIT.format(why=COMMIT_WHY[why]))
            try:
                g = await conv.turn(max(256, min(3072 if why == 'time' else 8192, CONTEXT - ctx - 512)), thinking=False)
            except Exception as e:  # noqa: BLE001 - no room left even for the answer, or the server is gone
                g = dict(content='', reasoning='', calls=[], finish=f'error: {e}'[:200], prompt=0, tokens=0, cached=0)
            rec['requests'].append(spent(g))
            rec['texts'].append(g)
            rec['committed'] = True
            got = text_grid(g['content'])
            how, seen = ('grid' if got is not None else None), []
            for c in g['calls'][:3]:
                name, code = call_code(c)
                out = ''
                try:
                    if name == 'submit':
                        out, res = await sb.submit(code)
                        test = (res['test_outputs'] or [None])[0] if res else None
                        if res and kaggle_grid(test):
                            got, how = test, 'submit'
                            if res['n_ok'] == res['n']:   # it reproduces every example: an accepted program after all
                                if on_accept:
                                    on_accept(test)
                                if siblings and on_shared:
                                    on_shared(await sb.apply(list(siblings)))
                                accepted = dict(n_ok=res['n_ok'], n=res['n'], test=test)
                    elif name == 'python':
                        out = await sb.run(code)
                        grid = printed_grid(out, expected_shape(task, test_input))
                        if grid is not None:
                            got, how = grid, 'python'
                except Exception as e:  # noqa: BLE001 - a sandbox that cannot run it: this call gives no grid
                    out = f'error: {type(e).__name__}: {e}'
                seen.append(dict(name=name, code=code[:3000], out=out[:2500]))
            rec['final_from'] = how
            final = None if accepted else got
            log(turn='final', why=why, tokens=g['tokens'], prompt=g['prompt'], finish=g['finish'],
                content=g['content'][:1500], calls=seen, grid=how)
    except asyncio.CancelledError:
        # The GPU turn is over. Before destroying this worker, checkpoint a program that
        # was defined in python but never submitted. This uses no model tokens and adds
        # at most two seconds to the existing bounded cancellation cleanup.
        if pending is None and accepted is None:
            async def checkpoint():
                _, res = await sb.submit('')
                g = (res.get('test_outputs') or [None])[0] if res else None
                if g is None or not kaggle_grid(g):
                    return
                verified = res['n_ok'] == res['n']
                if verified and on_accept:
                    on_accept(g)              # persist before trying the siblings
                elif on_best and (best is None or res['n_ok'] > best['n_ok']):
                    on_best(g)
                log(turn='checkpoint', tokens=0, verified=verified, grid=True)
                if verified and siblings and on_shared:
                    on_shared(await sb.apply(list(siblings)))
            with contextlib.suppress(Exception):
                await asyncio.wait_for(checkpoint(), 2.0)
        raise
    finally:
        rec['sandbox_calls'], rec['sandbox_timeouts'] = sb.calls, sb.timeouts
        rec['parks'], rec['parked_s'] = conv.parks, round(conv.parked_s)
        await sb.close()
        try:   # bounded, so a wedged server cannot hold an attempt (or the stop) here
            await asyncio.wait_for(conv.close(), 30)
        except Exception:  # noqa: BLE001 - a dropped connection must not hide the attempt's own outcome
            pass
    p = pending['test'] if pending else None
    if accepted:
        if p is not None and p != accepted['test']:   # a different rule after the note: the second answer
            return dict(rec, answer=p, second=accepted['test'], verified=True, source='program', alternative=True)
        return dict(rec, answer=accepted['test'], verified=True, source='program')
    if pending:   # accepted once, then the model ended without submitting again
        return dict(rec, answer=p, second=final if final and final != p else None, verified=True, source='program')
    if final:
        return dict(rec, answer=final, second=best['test'] if best and best['test'] != final else None,
                    verified=False, source=f'final-{rec["final_from"]}' if rec['committed'] else 'grid')
    return dict(rec, answer=best['test'] if best else None, verified=False, source='best-submit' if best else None)

## 10. Solve

- **One harness attempt per task,** by `SLOTS` (64) workers: first every task's first test output, in the order of the test file. An accepted program answers its task's other test outputs at once; those that stay unanswered (no program accepted) are attempted after every task's first, and an output whose task's attempt is still running waits for it, until the last stretch before the stop. A new conversation starts only when the KV gate lets it, so the cache, not a fixed number, sets how many run at once. A worker takes the next output the moment its attempt ends.
- **Attempts 2-4 in the idle end** (`TAIL`, `MAX_ATTEMPTS`; the 4-attempt schedule of 10-30,
  `docs/reports/1029_2028_attempt_scheduling/COMPARISON.md`). When a worker finds no first attempt it can start, it takes
  a waiting later attempt instead, and goes back to first attempts the moment one can start. Running later attempts are
  never stopped for a first attempt, and, apart from two-at-once below, a task never has two attempts running.
- **Two-at-once** (`PAR2`, `PAR2_AT`, sub 20): a free worker first looks (at most every 5 s) for a task whose 1st attempt
  is running alone, has used `PAR2_AT` tokens (64k, counted over its finished turns) and has accepted nothing; it starts
  that task's 2nd attempt beside it (tier X), ahead of any new task, only while the pacer has slack (frugal level 0)
  and not in the last `TAIL_MIN_LEFT`. The 3rd and 4th attempts follow by tier as below, each only when the task has
  nothing running. `RETRY_FIX`: a task whose own 1st attempt found no program always gets its 2nd attempt.
  - **Who gets attempt 2:** as in submissions 13-17, by the first attempt's class: U (no accepted program, or one only in
    its final answer turn), L (accepted after `TAIL_LATE_AT`), R (its review found a rival rule), P (after `TAIL_PLAUS_AT`),
    E (earlier). An errored attempt gets one retry, as tier U, counted against the four.
  - **Who gets attempts 3 and 4:** a task none of whose outputs is settled. An output is settled (AGREE) when two accepted
    attempts gave it the same grid; a task is settled when all its outputs are. Otherwise the next attempt's tier is the
    worst of its outputs': PARTIAL (fewer than two accepted attempts) ranks after the U, L and R second attempts, three or more candidates (DISTINCT3)
    with P, and two disagreeing answers that already fill both slots (DISTINCT2) after E. At most `MAX_ATTEMPTS` (4).
  - **Order:** one queue by (tier, attempts done, key): every second attempt of tiers U, L and R runs before any third
    attempt of a lower tier. Within the second attempts, U by the longest first attempt, the others by the latest
    acceptance. None in the last `TAIL_MIN_LEFT`.
  - **The attempt:** independent and with a full budget, never shown an earlier answer, seeing the task as given
    (`TAIL_VIEW = 'plain'`). It works on the task's first output that is not settled, with its other outputs as siblings,
    and **ends at its first accepted program** (no review: on the replayed records the review of a later attempt bought
    nothing for 0.14-0.26x of a first pass's tokens).
  - **Placement, by vote** (`pick2`): every attempt gives each output one vote, its accepted answer (else its final grid; sub 20, G3: one accepted
    vote per conversation, a sibling's shared program included, and first-pass candidates keep the earlier tie index);
    attempt 1 and attempt 2 are the grids with the most accepted votes, then the most votes, then the earliest. The first
    review's rival and the best failing submission are candidates without a vote, so a rival fills attempt 2 behind one
    agreed answer. An accepted program is placed the moment it is accepted, so an attempt cut at the stop keeps it.
    With only the first attempt, the placement is submission 17's.
- **The wrap-up** (`WRAP_S` before the stop): no attempt starts; each turn is capped to end by then (at `TOK_PER_S` tokens a second, lowered on a slow machine); an attempt without a program then gives its final answer, thinking off, in at most 3,072 tokens.
- **The pace line** (`PACE_EVERY`, from `PACE_START`): the token rate over the last 60 minutes (the sample just before the hour opens the window); the tokens the first pass still needs, with unstarted first attempts at the Kaplan-Meier mean and running ones at their expected remainder, both up to the whole budget plus the answer turn; the projected end of the first pass plus `DRAIN_S`; and the slack to the stop.
  - With `FRUGAL_AUTO`, graded (10-27): a slack under `FRUGAL_EARLY_ON` (30) min in 2 readings in a row drops the review of acceptances at <= 32,768 tokens, until the slack is back above `FRUGAL_EARLY_OFF` (45); a slack under 10 min in 2 readings drops every review, until it is back above 30. Once every task's first output is done, the switch is off (later attempts have no review anyway: each ends at its first accepted program). The pace line counts only first attempts as remaining work, so queued second attempts never switch it on.
- `submission.json` is rewritten after every finished attempt, every accepted program and every better submission.
- **At the stop time,** attempts still running are cancelled, keeping what they had written, and nothing new starts; a cancelled attempt's
  program that was written but never submitted is graded once (the cancellation checkpoint, at most 2 s). The cell waits at most 2 min for them to clean up, so one that cannot (a stuck process, a wedged connection) does not hold the notebook.
- If the harness checks failed, a commit stops here; the scored rerun notes it and runs the harness anyway.

**What it prints:**
- a status line every 5 minutes: attempts done, accepted, running and waiting; second attempts done and waiting; the rate so far and when the queue would empty at it; the KV gate; the server's latest decode line;
- a `PACE` line every 10 minutes after the first hour;
- one line per finished attempt.

**What it keeps:**
- `attempts.jsonl`: one line per attempt, with its kind (`harness` or `tail`);
- `attempts/<task>_<test>_<kind>.json`: everything the model wrote (`tail` for attempt 2, `tail3` and `tail4` after),
  and with `OWN_ANSWERS` the attempt's own answer(s) beside the output's pair;
- `attempts/live/`: one line per turn, with its tool calls;
- `pace.jsonl`.

In [35]:
from collections import Counter

STOP = DEADLINE if FULL else min(DEADLINE, time.time() + minutes * 60)
check(HARNESS_OK, 'the harness checks failed (the turn-1 template or the sandbox self-test)')
print(len(outputs), f'test outputs of {len(todo)} tasks,'
      + (" one harness attempt per task (its accepted program answers the task's other test outputs)" if SHARE
         else ' one harness attempt per test output'),
      '| solving stops at', time.strftime('%H:%M:%S', time.gmtime(STOP)),
      f'UTC, {(STOP - time.time()) / 3600:.2f} h from now')
LIVE = '/kaggle/working/attempts/live'
os.makedirs(LIVE, exist_ok=True)
# every task's first test output, then the others: a task's accepted program answers its other test outputs (SHARE),
# so those get a conversation of their own only when their task's attempt accepted no program
queue = [o for o in outputs if o[1] == 0] + [o for o in outputs if o[1] > 0] if SHARE else list(outputs)
shared = {}             # (task, test) -> the attempt whose accepted program answered it, with no conversation of its own
active = Counter()      # task -> its attempts running
LATE = min(90 * 60, (STOP - time.time()) / 2)   # this close to the stop, an output no longer waits for its task
cands = {}              # (task, test) -> what its attempt gave: answer, second, accepted, best, verified, source, tokens, tail
running = {}            # attempt name -> start time
done = []               # one record per finished or cancelled attempt
t_solve = time.time()

# --- v11: the end of the run (settings TAIL, TAIL_LATE, TAIL_MIN_LEFT, WRAP_S, FRUGAL_AUTO live in the settings cell) ---
wrap_s = min(WRAP_S, (STOP - time.time()) * 0.02)               # shorter rehearsals scale these down, as LATE is
tail_min_left = min(TAIL_MIN_LEFT, (STOP - time.time()) * 0.06)
WRAP_AT = STOP - wrap_s       # read by repl_attempt: from here every attempt without a program is asked for its answer
FORCE_TAIL = REHEARSAL == 'smoke' and not RERUN     # the smoke commit tails every task, to show the tail path in its log
FRUGAL = [0]                  # read by repl_attempt (frugal_skips): 0 every review runs, 1 early ones go, 2 all go; set by pacer()


def frugal_next(level, short, tight, slack):
    """The graded frugal's next level from this pace reading: `short` and `tight` count the readings in a row under
    10 min and under FRUGAL_EARLY_ON (this one included). Level 2 (every review goes) after 2 readings under 10, down
    above 30; level 1 (the early reviews go) after 2 under FRUGAL_EARLY_ON, down above FRUGAL_EARLY_OFF."""
    if short >= 2:
        return 2
    if level == 2 and slack > 30:
        level = 1 if slack <= FRUGAL_EARLY_OFF else 0
    if level == 0 and tight >= 2:
        return 1
    if level == 1 and slack > FRUGAL_EARLY_OFF:
        return 0
    return level
SAMPLING['temperature'] = TEMPERATURE   # sub 13: SessionConv reads SAMPLING at every request
print('sampling temperature', SAMPLING['temperature'], flush=True)


STREAM_TPS = [TOK_PER_S]      # a safe decode speed per stream; pacer() lowers it when the machine is slow


def clock_cap():
    """Tokens a turn may generate so that it ends by WRAP_AT (at STREAM_TPS per stream); read by repl_attempt."""
    return max(1024, int(STREAM_TPS[0] * (WRAP_AT - time.time())))
tails = []                    # later attempts waiting: (tier 0-5, attempts done, key, task); for second attempts tier 0-4 =
                              #   U, L, R, P, E and key -tokens: U by its first attempt's
                              #   tokens, the others by the tokens at its acceptance
tail_state = {}               # task -> 'queued' | 'running' | 'done'
tail_cls = {}                 # task -> its tier: 'U' no program | 'L' after 96k | 'R' a rival rule | 'P' after 32k | 'E' earlier
first_running = [0]           # first or sibling attempts running: each may still queue a tail
att_n = Counter()             # (4x schedule, 10-30) task -> attempts started, the first included (MAX_ATTEMPTS)
retried = set()               # (4x) tasks whose errored attempt got its one retry (counted against the four)
par2_scan = [0.0]             # (PAR2) when a worker last looked for a hard first attempt (at most every 5 s)


def pair_of(grids):
    """The first two different Kaggle grids of `grids`, as [attempt_1, attempt_2] (None where there is none)."""
    out = []
    for g in grids:
        if g is not None and kaggle_grid(g) and g not in out:
            out.append(g)
    return (out + [None, None])[:2]


def tail_place(cls, cur, best, t):
    """One output's [attempt_1, attempt_2] once a tail has run. cls: 'U' (it had no accepted program), 'R' (its review
    found a rival rule) or 'L' (accepted: tiers L, P and E alike); cur: its pair before the tail; best: its best
    failing submission; t: the tail's dict(ta, ts, tv)."""
    ta, ts, tv = t['ta'], t['ts'], t['tv']
    if cls == 'U' and tv:                          # the tail found a program: its answer first, the old first answer second
        return pair_of([ta, cur[0], ts, cur[1]])
    if cls == 'R':   # (10-27) a rival rule already fills attempt_2: a verified answer differing from both replaces it
        return ([cur[0], ta] if tv and ta is not None and kaggle_grid(ta) and ta != cur[0] and ta != cur[1]
                else list(cur))
    if cls in ('L', 'P'):                          # accepted late (or plausible, sub 13): the tail's answer second if it differs
        return pair_of([cur[0], ta, ts, cur[1]])
    if cls == 'U' and (cur[1] is None or cur[1] == best):   # neither accepted: only an empty or `best` slot is filled
        return pair_of([cur[0], ta, ts, cur[1]])
    return list(cur)


def votes_of(o):
    """(4x schedule, 10-30) One output's candidates: grid -> [accepted attempt-1 votes, attempt-1 votes, first seen,
    grid]. Each attempt gives one attempt-1 vote: its first accepted program's output (accepted), else its final grid.
    The first review's rival, a second accepted program of the same attempt, the final and best grids are candidates
    without a vote."""
    c = cands.get(o, {})
    acc = c.get('accepted', [])
    own = c.get('answer')
    first = own if own is not None and (c.get('verified') or not acc) else (acc[0] if acc else None)
    table = {}

    def add(g, vote, accepted):
        if g is None or not kaggle_grid(g):
            return
        k = json.dumps(g)
        if k not in table:
            table[k] = [0, 0, len(table), g]
        if vote:
            table[k][1] += 1
            table[k][0] += int(bool(accepted))
    if c.get('program_votes'):
        for g in c['program_votes'].values():
            add(g, True, True)
    else:
        add(first, True, bool(acc) or bool(c.get('verified')))
    # First-pass candidates get their tie indices before any later attempt.
    for g in [c.get('second')] + acc + [own, c.get('best')]:
        add(g, False, False)
    for e in c.get('later', []):
        add(e['ta'], True, e['tv'])
    return table


def pick2(o):
    """(4x) attempt_1 and attempt_2 by vote: most accepted attempt-1 votes, then most attempt-1 votes, then first seen
    (attempt order). A rival has no vote, so it fills attempt_2 behind one agreed answer."""
    ranked = sorted(votes_of(o).values(), key=lambda v: (-v[0], -v[1], v[2]))
    return ([v[3] for v in ranked] + [None, None])[:2]


def next_tier(t):
    """(4x) The tier of task t's next attempt, or None once it is settled (every output AGREE: two accepted attempts
    gave it the same grid). Per output: PARTIAL (fewer than two accepted attempts) ranks after the U, L and R second
    attempts (2, G6: every risky task gets its second attempt before any task gets a third); three or more
    candidates (DISTINCT3) with P (3); two disagreeing answers that already fill the two slots (DISTINCT2) last (5)."""
    tiers = []
    for m in range(len(todo[t]['test'])):
        table = list(votes_of((t, m)).values())
        if any(v[0] >= 2 for v in table):
            continue
        if sum(v[0] for v in table) < 2:
            tiers.append(2)   # PARTIAL: after every U, L and R second attempt (G6), before P
        else:
            tiers.append(3 if len(table) >= 3 else 5)
    return min(tiers) if tiers else None


def set_answers(o):
    """attempt_1 and attempt_2 of one test output: its own attempt's answer when verified (else its first accepted
    program, its own or its task's), its second answer, its other accepted programs, an unverified answer of its own,
    then its best-scoring submission; then, if a tail ran, tail_place."""
    c = cands.get(o, {})
    acc = c.get('accepted', [])
    own = c.get('answer')
    first = own if own is not None and (c.get('verified') or not acc) else (acc[0] if acc else None)
    grids = []
    for g in [first, c.get('second')] + acc + [own, c.get('best')]:
        if g is not None and kaggle_grid(g) and g not in grids:
            grids.append(g)
    cur = (grids + [None, None])[:2]
    c['primary_pair'] = list(cur)   # (PAR2) the pair from the first attempt's own candidates, before any vote
    if c.get('later') or len(c.get('program_votes', {})) > 1:   # (4x) attempts 2-4 hold evidence: the vote over every candidate
        cur = pick2(o)
    elif c.get('tail'):
        cur = tail_place(c['tail']['cls'], cur, c.get('best'), c['tail'])
    answers[o] = cur
    write_submission()


def record(rec, texts=None, own=None):
    rec.setdefault('at_min', round((time.time() - t_solve) / 60, 1))   # when it ended, in minutes of solving
    done.append(rec)
    print(time.strftime('%H:%M:%S'), json.dumps(rec), flush=True)
    with open('attempts.jsonl', 'a') as f:
        f.write(json.dumps(rec) + '\n')
    with open(f"/kaggle/working/attempts/{rec['task']}_{rec['test']}_{rec['kind']}{rec['att'] if rec.get('att', 2) > 2 else ''}.json", 'w') as f:
        json.dump(dict(rec, answers=answers.get((rec['task'], rec['test'])),
                       **({'own_answers': own} if OWN_ANSWERS and own is not None else {}), generations=texts), f)


def live_calls(name):
    """Turns, tokens and tool calls of a harness attempt so far, from its live log."""
    try:
        turns = [json.loads(l) for l in open(f'{LIVE}/{name}.jsonl')]
    except (FileNotFoundError, json.JSONDecodeError):
        return 0, 0, 0
    return len(turns), sum(t.get('tokens', 0) for t in turns), sum(len(t.get('calls', [])) for t in turns)


def tail_class(rec):
    """A first attempt's class for the tail: 'U' (no program reproduced every example), 'L' (accepted after TAIL_LATE
    tokens or more, with no second rule), 'R' when its review found a rival rule (TAIL_RIVAL, 10-27), 'P' after
    TAIL_PLAUS_AT, then 'E' for an earlier one (TAIL_EARLY, 10-27: on unseen tasks early answers were wrong 4 times in
    18, so they get a second attempt too, last). None for a tier that is switched off."""
    if not rec or rec.get('unfinished') or rec.get('error'):
        return None
    if FORCE_TAIL:
        return 'U'   # the smoke commit runs every task's tail, so its log shows the tail path end to end (never in the scored rerun)
    if not rec.get('verified') or not rec.get('first_accept'):   # a program accepted only in the final answer turn counts as none
        return 'U'
    if TAIL_TIERS:   # sub 13: by the tokens at the first acceptance; a rival rule already fills attempt_2
        if rec.get('alternative'):   # a rival rule fills attempt_2; tier R replaces it only on three different answers
            return 'R' if TAIL_RIVAL else None
        acc = rec.get('first_accept_tokens') or rec.get('tokens') or 0
        return 'L' if acc > TAIL_LATE_AT else ('P' if acc > TAIL_PLAUS_AT else ('E' if TAIL_EARLY else None))
    if (rec.get('tokens') or 0) >= TAIL_LATE and not rec.get('alternative'):
        return 'L'
    return None


def consider_tail(o, summary):
    t, i = o
    if not (TAIL and SHARE) or i != 0 or t in tail_state:
        return
    att_n[t] = max(att_n[t], 1)
    if summary.get('error') and t not in retried and MAX_ATTEMPTS > 1:   # (4x) an errored attempt: one retry, tier 0
        retried.add(t)
        tail_state[t], tail_cls[t] = 'queued', 'U'
        tails.append((0, 1, 0, t))
        tails.sort()
        return
    cls = tail_class(summary)
    if cls:
        tail_state[t], tail_cls[t] = 'queued', cls
        if TAIL_TIERS:   # sub 13 + 10-27: U (longest first attempt first), then L, R, P, E (latest acceptance first)
            acc = summary.get('first_accept_tokens') or summary.get('tokens') or 0
            tails.append(({'U': 0, 'L': 1, 'R': 2, 'P': 3, 'E': 4}[cls], 1,
                          -(summary.get('tokens') or 0) if cls == 'U' else -acc, t))
        else:
            tails.append((0 if cls == 'U' else 1, 1, -(summary.get('tokens') or 0), t))
        tails.sort()


async def run_harness(http_long, o):
    tid, i = o
    name, t = f'{tid}_{i}_harness', time.time()
    base = dict(task=tid, test=i, kind='harness')
    task, test_input = todo[tid], todo[tid]['test'][i]['input']
    c = cands.setdefault(o, {})
    sib = [(tid, j) for j in range(len(task['test']))
           if j != i and not cands.get((tid, j), {}).get('accepted')] if SHARE else []

    def on_shared(grids):   # this attempt's accepted program on the task's other test outputs
        for s, g in zip(sib, grids):
            if g is not None and kaggle_grid(g):
                cs = cands.setdefault(s, {})
                cs.setdefault('program_votes', {}).setdefault(name, g)
                if g not in cs.setdefault('accepted', []):
                    cs['accepted'].append(g)
                cs.setdefault('source', 'shared')
                if s not in shared:
                    shared[s] = name
                    print(time.strftime('%H:%M:%S'), f'{s[0]} test {s[1]}: answered by {name}', flush=True)
                set_answers(s)

    def on_accept(g):   # one primary accepted vote from this independent conversation
        c.setdefault('program_votes', {}).setdefault(name, g)
        if g not in c.setdefault('accepted', []):
            c['accepted'].append(g)
        set_answers(o)

    def on_best(g):     # so does the best-scoring failing submission, which a cancelled attempt then keeps
        c['best'] = g
        set_answers(o)

    running[name] = t
    active[tid] += 1
    first_running[0] += 1
    try:
        rec = await repl_attempt(http_long, task, test_input, harness_tokens, live=f'{LIVE}/{name}.jsonl',
                                 on_accept=on_accept, on_best=on_best,
                                 siblings=[task['test'][j]['input'] for _, j in sib], on_shared=on_shared)
        texts = rec.pop('texts')
        c.update(answer=rec.get('answer'), second=rec.get('second'), verified=bool(rec.get('verified')),
                 source=rec.get('source'), first_accept=rec.get('first_accept') is not None)
        summary = dict(tokens=sum(q['tokens'] for q in rec['requests']),
                       prefill_new=sum(q['prompt'] - q['cached'] for q in rec['requests']),
                       **{k: rec.get(k) for k in ('source', 'verified', 'alternative', 'turns', 'python_calls',
                                                  'cut_turns', 'loops', 'committed', 'final_from', 'spent',
                                                  'context_full', 'parks', 'parked_s', 'sandbox_timeouts',
                                                  'calls_in_reasoning', 'bare_calls', 'malformed', 'budget_notes',
                                                  'same_noted', 'error', 'gestalt_tokens', 'gestalt_forced',
                                                  'gestalt_error', 'gestalt', 'first_accept_tokens')},
                       submits=len(rec['submits']), first_accept=rec.get('first_accept') is not None)
    except asyncio.CancelledError:   # the stop time: what the attempt wrote into submission.json stays there
        n, tok, calls = live_calls(name)
        record(dict(base, seconds=round(time.time() - t), unfinished=True, turns=n, tokens=tok, tool_calls=calls,
                    grid=any(answers.get(o, [None]))))
        raise
    except Exception as e:  # noqa: BLE001 - one failed attempt must not stop the solve
        texts, summary = None, dict(error=f'{type(e).__name__}: {e}'[:300])
    finally:
        running.pop(name, None)
        active[tid] -= 1
        first_running[0] -= 1
    c['tokens'] = summary.get('tokens', 0)
    set_answers(o)
    if PAR2:   # (PAR2) the first pass's pair is the first attempt's own, even when its copy ran beside it
        for m in range(len(task['test'])):
            cm = cands.get((tid, m))
            if cm is not None and cm.get('primary_pair') is not None:
                cm['pair_first'] = list(cm['primary_pair'])
    record(dict(base, seconds=round(time.time() - t), grid=any(answers.get(o, [None])), **summary), texts,
           own=[c.get('answer'), c.get('second')])   # (OWN_ANSWERS) this attempt's own answer and its review's rival
    consider_tail(o, summary)


def tail_pair_fixed(t, i, sib, accs):
    """True once a second attempt's first accepted program fixes every output's pair under tail_place: its
    grid differs from the first pass's attempt 1 on each output, so the attempt's review could not change the
    submission."""
    if not accs or (sib and accs[0][1] is None):
        return False
    firsts = [accs[0][0]] + list(accs[0][1] or [])
    for k, m in enumerate([i] + sib):
        g = firsts[k] if k < len(firsts) else None
        cur0 = (cands.get((t, m), {}).get('pair_first') or [None])[0]
        if g is None or not kaggle_grid(g) or cur0 is None or g == cur0:
            return False
    return True


async def run_tail(http_long, t):
    """(4x schedule, 10-30) Attempt k = 2..MAX_ATTEMPTS on task `t`, independent (never shown an earlier answer), in
    the run's idle time, ending at its first accepted program (no review). Its answers join every output's candidates
    and pick2's vote places them; a task whose outputs are not all AGREE then queues its next attempt by tier."""
    task = todo[t]
    k = att_n[t] + 1
    # TAIL_VIEW (10-27): 'transpose' shows the task transposed and transposes every grid back; 'plain' (sub 18) does not
    view = (lambda g: [list(r) for r in zip(*g)] if g is not None else None) if TAIL_VIEW == 'transpose' else (lambda g: g)
    seen = task if TAIL_VIEW != 'transpose' else {
        'train': [{'input': view(p['input']), 'output': view(p['output'])} for p in task['train']],
        'test': [{'input': view(p['input'])} for p in task['test']]}
    n = len(task['test'])
    if k == 2:   # the second attempt: on the first output with no accepted program, else the first
        need = [m for m in range(n) if not cands.get((t, m), {}).get('accepted')]
        c0 = cands.get((t, 0), {})
        final_turn_only = c0.get('verified') and not c0.get('first_accept')   # its own program came only in the answer turn
        if not RETRY_FIX and tail_cls.get(t) == 'U' and not need and not final_turn_only and not FORCE_TAIL and t not in retried:
            tail_state[t] = 'done'   # a sibling's own attempt found a program that answers every output
            return
    else:        # a third or fourth attempt: on the first output that is not settled
        need = [m for m in range(n) if not any(v[0] >= 2 for v in votes_of((t, m)).values())]
        if not need:
            tail_state[t] = 'done'
            return
    att_n[t] = k
    i = need[0] if need else 0
    sib = [m for m in range(n) if m != i]
    name, t0 = f'{t}_{i}_tail' + (str(k) if k > 2 else ''), time.time()
    base = dict(task=t, test=i, kind='tail', att=k, view=TAIL_VIEW, tier=tail_cls.get(t))
    accs = []   # this attempt's accepted test output
    for m in range(n):   # each output's pair before the later attempts, for the finish's first-pass score
        cands.setdefault((t, m), {}).setdefault('pair_first', list(answers.get((t, m)) or [None, None]))

    def add(m, g, tv):   # this attempt's answer on output m joins its candidates at once (a cut attempt keeps it)
        if g is None or not kaggle_grid(g):
            return
        later = cands.setdefault((t, m), {}).setdefault('later', [])
        if any(e['att'] == k for e in later):
            return
        later.append(dict(att=k, ta=g, tv=tv))
        set_answers((t, m))

    def on_acc(g):
        accs.append(g)
        add(i, g, True)

    def on_sh(grids):
        if accs:
            for m, g in zip(sib, grids or []):
                add(m, g, True)

    tail_state[t] = 'running'
    running[name] = t0
    active[t] += 1
    try:
        rec = await repl_attempt(http_long, seen, seen['test'][i]['input'], harness_tokens, live=f'{LIVE}/{name}.jsonl',
                                 on_accept=lambda g: on_acc(view(g)), on_best=None,
                                 siblings=[seen['test'][m]['input'] for m in sib],
                                 on_shared=(lambda gs: on_sh([view(g) for g in gs])) if sib else None,
                                 stop_after_accept=lambda: bool(accs))   # (4x) attempts 2-4 end at their first acceptance
        texts = rec.pop('texts', None)
        rec.update(answer=view(rec.get('answer')))   # back to the task's own orientation
    except asyncio.CancelledError:   # the stop: what it accepted is placed already
        n_, tok, calls = live_calls(name)
        record(dict(base, seconds=round(time.time() - t0), unfinished=True, turns=n_, tokens=tok, tool_calls=calls))
        raise
    except Exception as e:  # noqa: BLE001 - a failed attempt leaves the task's answers as they were
        rec, texts = dict(error=f'{type(e).__name__}: {e}'[:300], requests=[], submits=[]), None
        if accs:   # it accepted a program before failing: that answer stands (added already)
            rec.update(answer=accs[0], verified=True)
    finally:
        running.pop(name, None)
        active[t] -= 1
        tail_state[t] = 'done'
    add(i, rec.get('answer'), bool(rec.get('verified')))   # no program: its final grid is a vote without acceptance
    tier = next_tier(t) if k < MAX_ATTEMPTS else None
    if rec.get('error') and t not in retried and k < MAX_ATTEMPTS:   # (4x) one retry, counted against the four
        retried.add(t)
        tier = 0
    if tier is not None:
        tails.append((tier, k, 0, t))
        tails.sort()
        tail_state[t] = 'queued'
    tokens = sum(q['tokens'] for q in rec.get('requests', []))
    record(dict(base, seconds=round(time.time() - t0), grid=any(answers.get((t, i), [None])), tokens=tokens,
                prefill_new=sum(q['prompt'] - q['cached'] for q in rec.get('requests', [])), verified=rec.get('verified'),
                source=rec.get('source'), spent=rec.get('spent'), committed=rec.get('committed'), error=rec.get('error'),
                context_full=rec.get('context_full'), parks=rec.get('parks'), parked_s=rec.get('parked_s'),
                sandbox_timeouts=rec.get('sandbox_timeouts'), turns=rec.get('turns'),
                first_accept_tokens=rec.get('first_accept_tokens'),
                next_tier=tier, gestalt_tokens=rec.get('gestalt_tokens'), gestalt_forced=rec.get('gestalt_forced'),
                gestalt_error=rec.get('gestalt_error'), gestalt=rec.get('gestalt'),
                placed={m: answers.get((t, m)) is not None for m in range(n)}), texts,
           own=[rec.get('answer')])   # (OWN_ANSWERS) this attempt's own answer


def next_output():
    """The next test output to attempt, or None for now. One its task's program answered needs no attempt; one whose
    task has an attempt running waits for it (that program may yet answer it), until LATE before the stop."""
    queue[:] = [o for o in queue if o not in shared]
    for k, o in enumerate(queue):
        if not SHARE or not active[o[0]] or STOP - time.time() < LATE:
            return queue.pop(k)
    return None


def next_tail():
    """A waiting tail whose task has no attempt running; none in the last TAIL_MIN_LEFT seconds (it could not finish).
    Called only when no first or sibling attempt can start: outputs blocked behind their own task never hold a tail back."""
    if not (TAIL and SHARE) or STOP - time.time() < tail_min_left:
        return None
    for k, (_, _, _, t) in enumerate(tails):
        if not active[t]:
            tails.pop(k)
            return t
    return None


async def worker(http_long):
    while time.time() < WRAP_AT:   # no new attempt in the last minutes: it could only be cut
        if not (queue or tails or running):   # (4x) nothing waits, and no running attempt can still queue one
            return
        if GATE is not None:
            await GATE.admit_attempt()   # a new conversation only when its first turn fits in the KV pool
        if (PAR2 and FRUGAL[0] == 0 and time.time() < WRAP_AT and STOP - time.time() >= tail_min_left
                and time.time() - par2_scan[0] >= 5):
            par2_scan[0] = time.time()   # (PAR2) a task's first attempt past PAR2_AT tokens, nothing accepted, alone:
            hard = []                    # its second attempt starts now, beside it, ahead of any new task
            for nm in list(running):
                tid = nm[:-len('_0_harness')] if nm.endswith('_0_harness') else None
                if (tid is None or tid in tail_state or active[tid] != 1 or att_n[tid] >= MAX_ATTEMPTS
                        or cands.get((tid, 0), {}).get('accepted')):
                    continue
                tok = live_calls(nm)[1]
                if tok >= PAR2_AT:
                    hard.append((tok, tid))
            if hard:
                tid = max(hard)[1]               # the longest-running one first
                att_n[tid] = max(att_n[tid], 1)   # its running first attempt is attempt 1; this one is attempt 2
                tail_cls[tid] = 'X'
                par2_scan[0] = 0.0               # the next free worker looks again at once
                await run_tail(http_long, tid)   # (no await before it marks the task running: no double start)
                continue
        o = next_output() if time.time() < WRAP_AT else None
        if o is not None:
            await run_harness(http_long, o)
            continue
        t = next_tail() if time.time() < WRAP_AT else None
        if t is not None:
            await run_tail(http_long, t)
            continue
        if GATE is not None:   # this admission started nothing: take back its provisional share (never below the last poll)
            GATE.usage = max(GATE.polled, GATE.usage - GATE.first)
        await asyncio.sleep(SLEEP_BLOCKED)


def decode_line():
    """The server's latest decode line (GPU 0)."""
    try:
        with open('sglang.log', 'rb') as f:
            f.seek(max(0, os.path.getsize('sglang.log') - 200_000))
            lines = [l for l in f.read().decode(errors='replace').splitlines() if 'TP0' in l and 'Decode batch' in l]
    except OSError as e:
        return f'no server log ({type(e).__name__})'
    return lines[-1].split('] ', 1)[-1] if lines else 'no decode line yet'


# --- the pace of the first pass: Kaplan-Meier token demand, the last hour's token rate, the end of the first pass ---
def km_curve(events, censored, prior_mean=85000.0, prior_w=6, cap=131500.0):
    """Survival of an attempt's total tokens from finished attempts (events) and running ones (censored at their tokens
    so far), with `prior_w` pseudo-events over 10k..cap so that the first hours are not noisy."""
    pts = [(x, 1) for x in events] + [(x, 0) for x in censored]
    pts += [(10000 + (cap - 10000) * (k + 0.5) / prior_w, 1) for k in range(prior_w)]
    pts.sort()
    n, s, xs, ss, i = len(pts), 1.0, [0.0], [1.0], 0
    while i < n:
        x, d, j = pts[i][0], 0, i
        while j < n and pts[j][0] == x:
            d += pts[j][1]
            j += 1
        if d:
            s *= 1 - d / (n - i)
        xs.append(x)
        ss.append(s)
        i = j
    return xs, ss


def km_integral(xs, ss, lo, hi):
    return sum(s * (min(b, hi) - max(a, lo)) for a, b, s in zip(xs, xs[1:] + [hi], ss) if min(b, hi) > max(a, lo))


def km_surv(xs, ss, x):
    s = 1.0
    for a, v in zip(xs, ss):
        if a > x:
            break
        s = v
    return s


tok_hist = []    # (time, tokens generated by all attempts so far)
KM_HI = harness_tokens + 3072   # remaining-work integrals run to the whole budget plus the answer turn (sub 13's 131,500 cut off longer attempts)


def tokens_now():
    return (sum((r.get('tokens') or 0) + (r.get('gestalt_tokens') or 0) for r in done)   # the picture looks too
            + sum(live_calls(n)[1] for n in list(running)))


def pace():
    """Where the first pass stands: the token rate over the last hour, the tokens still to be generated for the first
    attempts (unstarted ones at the Kaplan-Meier mean, running ones at their conditional remainder), the projected end of
    the first pass (+ DRAIN_S: the last attempts finish with the machine emptying), and the slack to the stop."""
    now = time.time()
    tok_hist.append((now, tokens_now()))
    past = [h for h in tok_hist if h[0] >= now - 3600]
    older = [h for h in tok_hist if h[0] < now - 3600]
    if older:   # the sample just before the hour: appends come every 600 s plus the work, so without it the window is 50 min
        past = [older[-1]] + past
    rate = (past[-1][1] - past[0][1]) / max(1.0, past[-1][0] - past[0][0]) if len(past) > 1 else 0.0
    fin = [r['tokens'] + (r.get('gestalt_tokens') or 0) for r in done
           if r.get('kind') == 'harness' and r.get('test') == 0 and not r.get('unfinished') and r.get('tokens')]
    run_first = [live_calls(n)[1] for n in list(running) if n.endswith('_0_harness')]
    unstarted = sum(1 for o in queue if o[1] == 0)
    xs, ss = km_curve(fin, run_first)
    if not unstarted and not run_first:   # the first pass is over: nothing left for the frugal switch to protect
        return dict(rate=rate, need=0.0, end=now, slack_min=None, mean_tokens=km_integral(xs, ss, 0.0, KM_HI),
                    first_pass_done=True)
    need = unstarted * km_integral(xs, ss, 0.0, KM_HI)
    for g in run_first:
        need += km_integral(xs, ss, g, KM_HI) / max(km_surv(xs, ss, g), 0.02)
    if rate <= 0:
        return dict(rate=rate, need=need, end=None, slack_min=None, mean_tokens=None)
    end = now + need / rate + DRAIN_S
    return dict(rate=rate, need=need, end=end, slack_min=(STOP - end) / 60, mean_tokens=km_integral(xs, ss, 0.0, KM_HI))


async def pacer(every=600):
    """Every 10 min after the first hour: print the pace, and (FRUGAL_AUTO) set the graded frugal level (frugal_next):
    the early reviews go after two readings in a row under FRUGAL_EARLY_ON min of slack, every review after two under
    10 min; back down above FRUGAL_EARLY_OFF and 30 min. The token count is sampled from the start, so the first
    projection already has the last hour's rate."""
    short = 0   # readings in a row with less than 10 min of slack (one noisy reading must not switch reviews off)
    tight = 0   # readings in a row with less than FRUGAL_EARLY_ON min of slack (the graded frugal's first stage)
    while True:
        p = pace()
        if time.time() - t_solve < PACE_START:
            await asyncio.sleep(every)
            continue
        if p['rate'] > 0 and running:
            STREAM_TPS[0] = min(TOK_PER_S, max(8.0, 0.75 * p['rate'] / len(running)))   # a slow machine: shorter turns near the end
        if FRUGAL_AUTO and p['slack_min'] is not None:
            short = short + 1 if p['slack_min'] < 10 else 0
            tight = tight + 1 if p['slack_min'] < FRUGAL_EARLY_ON else 0
            FRUGAL[0] = frugal_next(FRUGAL[0], short, tight, p['slack_min'])
        elif p.get('first_pass_done'):   # second attempts keep their review
            FRUGAL[0] = 0
        line = dict(t_min=round((time.time() - t_solve) / 60), rate_tok_min=round(p['rate'] * 60), need_Mtok=round(p['need'] / 1e6, 2),
                    mean_first_tokens=round(p['mean_tokens'] or 0), slack_min=None if p['slack_min'] is None else round(p['slack_min']),
                    frugal=FRUGAL[0], running=len(running), unstarted_firsts=sum(1 for o in queue if o[1] == 0),
                    tails_waiting=len(tails), parked=GATE.parked if GATE else 0,
                    first_pass_done=bool(p.get('first_pass_done')))
        print(time.strftime('%H:%M:%S'), 'PACE', json.dumps(line), flush=True)
        with open('pace.jsonl', 'a') as f:
            f.write(json.dumps(line) + '\n')
        await asyncio.sleep(every)


async def server_status(every=300):
    """Every 5 min: attempts done, accepted, running and waiting; the rate so far and when the queue would empty at it;
    the KV gate, and the server's latest decode line."""
    while True:
        await asyncio.sleep(every)
        fin = [r for r in done if not r.get('unfinished') and r.get('kind') == 'harness']
        rate = len(fin) / max(1.0, time.time() - t_solve) * 3600   # finished attempts per hour
        if not queue:
            when = 'none waiting'
        elif rate:
            when = f'the queue empties in {len(queue) / rate:.1f} h at this rate'
        else:
            when = 'no attempt finished yet'
        print(time.strftime('%H:%M:%S'), f'{len(fin)} attempts done ({sum(bool(r.get("verified")) for r in fin)} accepted,',
              f'{sum(bool(r.get("grid")) for r in fin)} with a grid), {len(running)} running, '
              f'{sum(o not in shared for o in queue)} waiting, {len(shared)} answered by their task\'s program;',
              f'tails {sum(r.get("kind") == "tail" and not r.get("unfinished") for r in done)} done / {len(tails)} waiting;',
              f'{rate:.1f} per hour, {when}; {(STOP - time.time()) / 3600:.2f} h left',
              (f'| KV {GATE.usage:.0%}, parked {GATE.parked}, parks {GATE.parks} ({GATE.host_parks} to host RAM),'
               f' polls unanswered {GATE.failed}' if GATE else ''), '| server:', decode_line(), flush=True)


async def solve():
    status = asyncio.create_task(server_status(STATUS_EVERY))
    pacing = asyncio.create_task(pacer(PACE_EVERY))
    try:  # the status and gate tasks must end with the solve, however the solve ends
        async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=None, sock_read=3600)) as http:
            # a harness turn is one request that stays silent until the turn ends, which can take over an hour
            async with aiohttp.ClientSession(timeout=aiohttp.ClientTimeout(total=None, sock_read=None)) as http_long:
                poll = asyncio.create_task(GATE.poll(http)) if GATE is not None else None
                workers = [asyncio.create_task(worker(http_long)) for _ in range(SLOTS)]
                try:
                    _, still = await asyncio.wait(workers, timeout=max(1, STOP - time.time()))
                finally:  # the stop time, or an interrupt of this cell: no worker may outlive it
                    in_flight = len(running)
                    for w in workers:
                        w.cancel()
                    # at most 2 min for the cancelled attempts to record themselves and clean up: one that cannot
                    # (a stuck process, a wedged connection) must not hold the notebook past Kaggle's limit
                    _, stuck = await asyncio.wait(workers, timeout=CANCEL_WAIT)
                    if stuck:
                        print(f'{len(stuck)} cancelled attempts did not finish cleaning up; left behind', flush=True)
                    if poll is not None:
                        poll.cancel()
                print(f'stop time reached: {in_flight} attempts cancelled, {len(queue)} never started, {len(tails)} tails never started' if still
                      else f'all {len(done)} attempts finished')
    finally:
        status.cancel()
        pacing.cancel()

if SERVER_UP:
    await solve()

172 test outputs of 120 tasks, one harness attempt per task (its accepted program answers the task's other test outputs) | solving stops at 23:08:31 UTC, 1.75 h from now
sampling temperature 1.0
21:28:31 0 attempts done (0 accepted, 0 with a grid), 64 running, 108 waiting, 0 answered by their task's program; tails 0 done / 0 waiting; 0.0 per hour, no attempt finished yet; 1.67 h left | KV 13%, parked 0, parks 0 (0 to host RAM), polls unanswered 0 | server: Decode batch, #running-req: 62, #full token: 420736, full token usage: 0.12, mamba num: 63, mamba usage: 0.93, accept len: 2.68, accept rate: 0.56, cuda graph: True, gen throughput (token/s): 386.49, #queue-req: 2
21:33:31 0 attempts done (0 accepted, 0 with a grid), 64 running, 108 waiting, 0 answered by their task's program; tails 0 done / 0 waiting; 0.0 per hour, no attempt finished yet; 1.58 h left | KV 25%, parked 0, parks 0 (0 to host RAM), polls unanswered 0 | server: Decode batch, #running-req: 62, #full token: 832448, full t

## 11. Finish

- **Shut down:** the server and every process on the GPUs are killed, and the cell waits until the GPUs are free. Every command there is bounded and allowed to fail, so this cell can never be what stops the notebook.
- **Final file:** `submission.json` is written one last time and checked against the format Kaggle scores: every task, one entry per test input, two valid grids each.
- **The harness, in numbers:** attempts finished, cancelled at the stop and never started; outputs answered by their task's program; tokens; accepted programs and second rules; per-attempt turns and tool calls; parks, errors, sandbox resets; spent budgets, budget notes, cut turns, loops, malformed and bare calls; the final answer turns and what each gave; where the answers came from.
- **The later attempts (2-4) and the pace:** tasks queued by class, started, finished and cut; their tokens; the pace lines and whether the frugal switch fired; when the first pass ended.
- **Score, outside the scored rerun:** every test output of the run is checked against its public solution and listed as solved by attempt 1 or 2, wrong, without an answer, or not started; then the first pass alone, what the second attempts added and lost, and the per-task average (the board's).
- **A commit fails** if the model answered nothing, if the harness checks failed, or if the harness never called a tool.
- **Speed:** the server's decode lines at full load.
- **Bench throughput** (`REHEARSAL = 'bench'`): the KV pool, the generated tokens every 30 minutes, mean running and acceptance, and the peak pool use. These are the numbers the serving levers change.

In [36]:
%%bash
# Every command here is bounded and allowed to fail: this cell's exit status must never be what stops the notebook
echo "compiled into /tmp/cache after the boot: $(find /tmp/cache -type f -newer /tmp/jit_seed_boot_marker -not -path '*/tilelang/tmp/*' -not -path '*/.staging*' 2>/dev/null | wc -l) files; Triton compiles after serving started: $(grep -c 'to compile after serving started' sglang.log 2>/dev/null) (cache loads: $(grep -c 'device-loaded after serving started' sglang.log 2>/dev/null)); TileLang compiles: $(grep -c 'TileLang begins to compile' sglang.log 2>/dev/null)"
pkill -9 -f "[s]glang.launch_server" || true
pkill -9 -f "[s]glang::" || true
# only our Python PIDs: in a gVisor sandbox (Modal) nvidia-smi reports PID 1, the container's init
for pid in $(timeout 20 nvidia-smi --query-compute-apps=pid --format=csv,noheader 2>/dev/null); do
    [ "$pid" -gt 1 ] 2>/dev/null && grep -qa python "/proc/$pid/cmdline" 2>/dev/null && kill -9 "$pid" 2>/dev/null || true
done
for i in $(seq 60); do [ -z "$(timeout 20 nvidia-smi --query-compute-apps=pid --format=csv,noheader 2>/dev/null)" ] && break; sleep 2; done
timeout 20 nvidia-smi --query-gpu=index,memory.used --format=csv,noheader || true
kill "$(cat host_mem.pid 2>/dev/null)" 2>/dev/null || true   # the host RAM log
true


compiled into /tmp/cache after the boot: 400 files; Triton compiles after serving started: 191 (cache loads: 208); TileLang compiles: 0
0, 0 MiB
1, 0 MiB
2, 0 MiB
3, 0 MiB


In [37]:
from collections import Counter

write_submission()
sub = json.load(open(SUBMISSION))
check(set(sub) == set(tasks) and all(len(sub[t]) == len(tasks[t]['test']) for t in tasks),
      'submission.json does not have one entry per test input')
check(all(kaggle_grid(e['attempt_1']) and kaggle_grid(e['attempt_2']) for v in sub.values() for e in v),
      'submission.json has a grid Kaggle cannot score')
model_outputs = sum(1 for g in answers.values() if any(g))
print(f'submission.json: {len(sub)} tasks; the harness answered {model_outputs} of {len(outputs)} test outputs; '
      f'{(time.time() - T0) / 3600:.2f} h since the first cell')

records = [json.loads(l) for l in open('attempts.jsonl')] if os.path.exists('attempts.jsonl') else []
firsts = [r for r in records if r.get('kind', 'harness') == 'harness']
tail_recs = [r for r in records if r.get('kind') == 'tail']
fin = [r for r in firsts if not r.get('unfinished')]
toks = sorted(r.get('tokens', 0) for r in fin)
print(f'harness attempts: {len(fin)} finished, {len(firsts) - len(fin)} cancelled at the stop, '
      f'{len(set(outputs) - {(r["task"], r["test"]) for r in firsts} - set(shared))} never started' + (f'; tokens per finished attempt: mean {sum(toks) / len(toks):,.0f}, '
      f'median {toks[len(toks) // 2]:,}, total {sum(toks):,}' if toks else ''))
print(f'{len(shared)} test outputs answered by their task\'s accepted program, with no attempt of their own')
ends = [r['at_min'] for r in fin if r.get('test') == 0 and r.get('at_min') is not None]
if ends:
    print(f'the first pass: its last first attempt ended at {max(ends):.0f} min of {(STOP - t_solve) / 60:.0f}')
if fin:
    n = len(fin)
    count = lambda k: sum(bool(r.get(k)) for r in fin)
    per = lambda k: sum(r.get(k) or 0 for r in fin) / n
    print(f'  accepted programs {count("verified")}, second rules {count("alternative")}, asked for a second rule '
          f'{count("first_accept")}, same-output second programs noted {count("same_noted")}')
    print(f'  per attempt: turns {per("turns"):.1f}, python {per("python_calls"):.1f}, submits {per("submits"):.1f}, '
          f'newly prefilled {per("prefill_new"):,.0f}; parks {sum(r.get("parks") or 0 for r in fin)}, '
          f'errors {count("error")}, sandbox resets {sum(r.get("sandbox_timeouts") or 0 for r in fin)}')
    print(f'  budget spent {count("spent")}, context full {count("context_full")}, budget notes '
          f'{sum(r.get("budget_notes") or 0 for r in fin)}, cut turns {sum(r.get("cut_turns") or 0 for r in fin)}, '
          f'loops removed {sum(r.get("loops") or 0 for r in fin)}, malformed calls {sum(r.get("malformed") or 0 for r in fin)}, '
          f'bare calls read {sum(r.get("bare_calls") or 0 for r in fin)}')
    finals = Counter(r.get('final_from') for r in fin if r.get('committed'))
    print(f'  final answer turns {count("committed")}, what they gave: {dict(finals)}')
    print('  answers from:', dict(Counter(r.get('source') for r in fin)))
looks = [r for r in records if r.get('gestalt_tokens') is not None]
if looks:
    print(f'picture looks: {len(looks)} attempts had one; it wrote something in '
          f'{sum(bool(r.get("gestalt")) for r in looks)}, needed the answer turn in {sum(bool(r.get("gestalt_forced")) for r in looks)}, '
          f'failed in {sum(bool(r.get("gestalt_error")) for r in looks)}; mean {sum(r["gestalt_tokens"] for r in looks) / len(looks):,.0f} tokens')
tail_fin = [r for r in tail_recs if not r.get('unfinished')]
print(f'later attempts (2-{MAX_ATTEMPTS}, in the idle end): {len(tail_cls)} tasks queued a second '
      f'({dict(Counter(tail_cls.values()))}); started by attempt {dict(Counter(r.get("att", 2) for r in tail_recs))}, '
      f'{len(tail_fin)} finished, {len(tail_recs) - len(tail_fin)} cancelled at the stop, {len(tails)} never started; '
      f'accepted {sum(bool(r.get("verified")) for r in tail_fin)}, errors {sum(bool(r.get("error")) for r in tail_fin)}, '
      f'settled (next_tier None) {sum(r.get("next_tier") is None for r in tail_fin)}, '
      f'tokens {sum(r.get("tokens") or 0 for r in tail_recs):,}')
if os.path.exists('pace.jsonl'):
    pace_lines = [json.loads(l) for l in open('pace.jsonl')]
    print(f'pace lines: {len(pace_lines)}; frugal on in {sum(bool(p.get("frugal")) for p in pace_lines)};',
          'last:', pace_lines[-1] if pace_lines else None)
for r in records:
    if r.get('error'):
        print('  ERROR', r.get('kind'), r['task'], r['test'], r['error'])

if not RERUN:
    with open('answers_final.json', 'w') as f:   # every output's final pair, and its pair before any second attempt
        json.dump({f'{t}_{i}': dict(final=answers.get((t, i)), first=cands.get((t, i), {}).get('pair_first'))
                   for t, i in outputs}, f)
    solutions = {}
    for split in ('training', 'evaluation'):
        solutions.update(json.load(open(f'{DATA}/arc-agi_{split}_solutions.json')))
    if REHEARSAL == 'bench' and BENCH_SET != 'eval':   # the bench set's own solutions
        solutions.update(json.load(open(f'{BENCH_DIR}/{BENCH_SET}_solutions.json')))
    started = {(r['task'], r['test']) for r in firsts}
    unfinished = {(r['task'], r['test']) for r in firsts if r.get('unfinished')}
    print(f'\nscore on this {"full" if FULL else REHEARSAL} run, against the public solutions:')
    for tid, i in outputs:
        grids, sol, c = answers.get((tid, i), [None, None]), solutions[tid][i], cands.get((tid, i), {})
        verdict = ('SOLVED by attempt 1' if grids[0] == sol else 'SOLVED by attempt 2' if grids[1] == sol
                   else 'wrong' if any(grids) else 'no answer' if (tid, i) in started else 'not started')
        how = 'cancelled at the stop' if (tid, i) in unfinished else c.get('source')
        tail = (f"; later attempts {[e['att'] for e in c['later']]} ({sum(bool(e['tv']) for e in c['later'])} accepted), "
                f"first pass {'right' if sol in (c.get('pair_first') or []) else 'wrong'}" if c.get('later') else '')
        print(f'  {tid} test {i}: {verdict}  ({how}; {sum(g is not None for g in grids)} grid(s){tail})')
    right = sum(solutions[tid][i] in answers.get((tid, i), [None, None]) for tid, i in outputs)
    first_pass = lambda o: cands.get(o, {}).get('pair_first') or answers.get(o, [None, None])
    right_first = sum(solutions[tid][i] in first_pass((tid, i)) for tid, i in outputs)
    print(f'{right} of {len(outputs)} test outputs solved ({right / len(outputs):.0%}); '
          f'{model_outputs} of {len(outputs)} answered before the stop')
    print(f'the first pass alone: {right_first} of {len(outputs)}; the tails: '
          f'{sum(solutions[t][i] in answers.get((t, i), []) and solutions[t][i] not in first_pass((t, i)) for t, i in outputs)} newly right, '
          f'{sum(solutions[t][i] not in answers.get((t, i), []) and solutions[t][i] in first_pass((t, i)) for t, i in outputs)} lost')
    by_task = Counter()
    for tid, i in outputs:
        by_task[tid] += (solutions[tid][i] in answers.get((tid, i), [None, None])) / len(todo[tid]['test'])
    print(f'per task (the board\'s average): {sum(by_task.values()):.2f} of {len(todo)} = {sum(by_task.values()) / len(todo):.1%}')
    if REHEARSAL == 'bench' and BENCH_SET != 'eval' and os.path.exists(f'{BENCH_DIR}/{BENCH_SET}_parents.json'):
        parents = set(json.load(open(f'{BENCH_DIR}/{BENCH_SET}_parents.json')))
        for name, keep in (('evaluation parents', lambda t: t in parents), ('their children', lambda t: t not in parents)):
            outs = [(t, i) for t, i in outputs if keep(t)]
            r = sum(solutions[t][i] in answers.get((t, i), [None, None]) for t, i in outs)
            f = sum(solutions[t][i] in (cands.get((t, i), {}).get('pair_first') or answers.get((t, i), [None, None])) for t, i in outs)
            print(f'  {name}: {r} of {len(outs)} outputs (first pass {f})')
    check(model_outputs > 0, 'the model answered nothing')
    check(HARNESS_OK, 'the harness checks failed')
    check(any((r.get('python_calls') or 0) + (r.get('submits') or 0) + (r.get('tool_calls') or 0) for r in records),
          'the harness never called a tool')

# Decode speed at full load, from GPU 0's decode lines (GPUs 1-3 log the same steps)
decode = [(int(r), float(a), float(t)) for r, a, t in re.findall(
    r'TP0[^\]]*\] Decode batch, #running-req: (\d+),.*?accept len: ([\d.]+),.*?gen throughput \(token/s\): ([\d.]+)',
    open('sglang.log', errors='replace').read() if os.path.exists('sglang.log') else '')]
full = [(r, a, t) for r, a, t in decode if r >= SLOTS - 2 and a > 0]
med = lambda xs: sorted(xs)[len(xs) // 2]
if full:
    print(f'decode at full load ({len(full)} log lines, {med([r for r, _, _ in full])} running): '
          f'{med([t for _, _, t in full]):.1f} tokens/s, accept length {med([a for _, a, _ in full]):.2f}, '
          f'{med([t / r / a for r, a, t in full]):.3f} steps/s per request')
elif decode:
    print(f'no decode line at full load; {len(decode)} decode lines, median {med([t for _, _, t in decode]):.1f} '
          f'tokens/s at {med([r for r, _, _ in decode])} running')
else:
    print('no decode line')

submission.json: 240 tasks; the harness answered 154 of 172 test outputs; 1.90 h since the first cell
harness attempts: 146 finished, 0 cancelled at the stop, 0 never started; tokens per finished attempt: mean 36,508, median 26,032, total 5,330,097
33 test outputs answered by their task's accepted program, with no attempt of their own
the first pass: its last first attempt ended at 105 min of 105
  accepted programs 89, second rules 6, asked for a second rule 85, same-output second programs noted 4
  per attempt: turns 9.8, python 8.5, submits 1.0, newly prefilled 7,220; parks 5, errors 0, sandbox resets 1
  budget spent 0, context full 0, budget notes 11, cut turns 129, loops removed 0, malformed calls 0, bare calls read 0
  final answer turns 61, what they gave: {None: 19, 'python': 30, 'submit': 12}
  answers from: {'program': 89, None: 19, 'final-python': 30, 'final-submit': 8}
picture looks: 146 attempts had one; it wrote something in 146, needed the answer turn in 0, failed in 0;

In [38]:
from datetime import datetime

# Bench throughput, from the server's decode lines: each covers --decode-log-interval 200 steps, so the tokens it adds
# are its throughput times the time since the previous line.
if REHEARSAL == 'bench' and not RERUN:
    log = open('sglang.log', errors='replace').read()
    pool = re.search(r'TP0[^\]]*\] KV Cache is allocated.*?#tokens: (\d+)', log)
    rows = [(datetime.strptime(m[1], '%Y-%m-%d %H:%M:%S').timestamp(), int(m[2]), float(m[3]), float(m[4]), float(m[5]))
            for m in re.finditer(r'\[(\S+ \S+) TP0[^\]]*\] Decode batch, #running-req: (\d+).*?full token usage: ([\d.]+)'
                                 r'.*?accept len: ([\d.]+).*?gen throughput \(token/s\): ([\d.]+)', log)]
    total, marks = 0.0, {}
    for (ta, *_), (tb, _run, _use, _acc, thr) in zip(rows, rows[1:]):
        total += thr * (tb - ta)
        for mark in range(30, int(BENCH_MINUTES) + 30, 30):
            if tb - rows[0][0] <= mark * 60:
                marks[mark] = total
    print(f"Flash-Next K384: KV pool {int(pool[1]) if pool else 0:,} tokens; "
          f"{len(rows)} decode lines over {(rows[-1][0] - rows[0][0]) / 60:.1f} min" if rows else 'no decode lines')
    print('generated tokens by minute:', {m: f'{v:,.0f}' for m, v in marks.items()})
    if rows:
        print(f'mean running {sum(r[1] for r in rows) / len(rows):.1f}, mean accept {sum(r[3] for r in rows) / len(rows):.2f}, '
              f'peak pool use {max(r[2] for r in rows):.2f}')


Flash-Next K384: KV pool 3,467,136 tokens; 166 decode lines over 102.7 min
generated tokens by minute: {30: '1,712,092', 60: '3,399,636', 90: '4,970,612', 120: '5,638,044'}
mean running 58.7, mean accept 2.90, peak pool use 0.90
